In [ ]:
import os, tempfile


sf_options = {
    "sfUrl": "KOMODOHEALTH-STD_APRCRWRZYNKCIWYRUD.snowflakecomputing.com",
    "sfUser": "PRAKHAR.PATHAK@TAKEDA.COM",
    "sfPassword": "",  
    "sfRole": "TAKEDA_TA_SENTINEL_ROLE",
    "sfWarehouse": "WH_ADHOC",
    "sfDatabase": "DSVC_TAKEDA_TA_PRIVATE",
    "sfSchema": "TAK861",
}
if not sf_options["sfPassword"].strip():
    raise ValueError("Enter your PAT in sf_options['sfPassword'] in this notebook's first code cell.")


SCRATCH_DIRECTORY = ("/local_disk0/tmp/tak861_automl" if os.path.isdir("/local_disk0")
                     else os.environ.get("TMPDIR", tempfile.gettempdir()))
os.makedirs(SCRATCH_DIRECTORY, exist_ok=True)
os.environ["TMPDIR"] = SCRATCH_DIRECTORY
tempfile.tempdir = SCRATCH_DIRECTORY
with tempfile.TemporaryFile(dir=SCRATCH_DIRECTORY) as scratch_probe:
    scratch_probe.write(b"scratch-write-check")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
RUN_ID = "R20261023_OLD_A"  
SOURCE_PREFIX = "TAK861_TX_READY_V63_DL_POC"
CONFIG = {
    "client_universe": {"table": "DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA",
                        "table_name": "TAK861_TX_READY_V63_MODEL_DATA",
                        "expected_population": {"snapshots": 23151, "patients": 12447, "positives": 1345,
                            "per_split": {"train": {"snapshots": 16256, "patients": 8712, "positives": 941},
                                          "validation": {"snapshots": 3481, "patients": 1867, "positives": 202},
                                          "test": {"snapshots": 3414, "patients": 1868, "positives": 202}}},
                        "split_seed": 42, "max_missing_feature_fraction": .10,
                        "sources": {}, "common_scope": None, "family_column": None},
    "date_policy": "month_end_history", "seeds": [42, 142, 242],
    "require_count_reference": True, "include_automl_only_controls": False,
    "include_lightgbm": False, "primary_lift_fraction": .10,
    "raw_automl": {"date_policy": "month_end_history", "chunk_size": 5000, "max_features": 512},
    "selection": {"n_splits": 3, "seed": 42,
                  "candidate_sizes": [16, 32, 64, 128, 256, 512, 1028],
                  "proxy_C": .1, "proxy_max_iter": 1000, "weighting": "patient"},
    "model": {"d_model": 64, "heads": 4, "depth": 1, "ff": 128,
              "dropout": .2, "weight_decay": .001, "learning_rate": .0003,
              "epochs": 30, "patience": 5, "batch_size": 256,
              "pooling": "mean", "gradient_clip": 1., "scheduler": "plateau",
              "weighting": "patient", "balance": "sqrt", "threads": 8, "device": "cpu"}}
if "spark" not in globals():
    raise RuntimeError("An active Spark session and the existing private source connection are required.")

sf_options_dl_poc = dict(sf_options)

sf_options_dl_poc.update(sfDatabase="DSVC_TAKEDA_TA_PRIVATE", sfSchema="DS_ML")


In [ ]:
import types, sys, json, hashlib, io, time
import numpy as np, pandas as pd
from IPython.display import display
EMBEDDED_SOURCES = {'inputs': '"""Historical AutoML input construction. Never derives history from snapshot values.\n\nOnly extraction/assembly live here. Every learned transform belongs to TRAIN-only\npreparation. Cached panels are reused by exact patient/date and checked definition\nand content fingerprints. This module does not assert unavailable source lineage.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport time\nimport os\nimport tempfile\nfrom typing import Callable\n\nimport numpy as np\nimport pandas as pd\n\n\nKEYS = ["PATIENT_ID", "AS_OF_DATE"]\nPANEL_MEMORY_LIMIT = 16 * 1024 * 1024\nARRAY_ROW_CHUNK = 2048\n\n\ndef disk_array(shape, dtype=np.float32, prefix="tak861-array-"):\n    """Allocate an owned scratch memmap; callers release it after persistence."""\n    handle, path = tempfile.mkstemp(prefix=prefix, suffix=".npy")\n    os.close(handle)\n    try:\n        value=np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n    except BaseException:\n        os.unlink(path)\n        raise\n    value._scratch_owner_id=id(value)\n    return value\n\n\ndef _require_array_owner(value):\n    if not isinstance(value,np.memmap) or getattr(value,\'_scratch_owner_id\',None)!=id(value):\n        raise ValueError("Cannot release an array view or unowned mapping; release the owning array")\n\n\ndef readonly_array(value):\n    """Transfer one owned scratch file into a read-only owning mapping."""\n    _require_array_owner(value)\n    path=value.filename\n    value.flush()\n    value._mmap.close()\n    result=np.load(path,mmap_mode=\'r\',allow_pickle=False)\n    result._scratch_owner_id=id(result)\n    value._scratch_owner_id=None\n    return result\n\n\ndef release_array(value):\n    """Release only the explicit scratch owner, never an ndarray/memmap view."""\n    if isinstance(value, np.memmap):\n        _require_array_owner(value)\n        path = value.filename\n        if not value._mmap.closed:\n            value.flush()\n            value._mmap.close()\n        try:\n            os.unlink(path)\n        except FileNotFoundError:\n            pass\n    elif isinstance(value,np.ndarray):\n        current=value.base\n        while isinstance(current,np.ndarray):\n            if isinstance(current,np.memmap):\n                raise ValueError("Cannot release an array view; release the owning array")\n            current=current.base\n\n\nclass DiskPanel:\n    """Ordered metadata plus a disk-backed float64 matrix, never a wide merge.\n\n    Float64 storage preserves the original validated-panel fingerprint; model\n    tensors are still float32, exactly as in the original assembly path.\n    """\n    def __init__(self, metadata, values, names):\n        self.metadata = metadata.reset_index(drop=True)\n        self.values = values\n        self.names = list(names)\n        self.columns = pd.Index(KEYS + ["IS_OBSERVED"] + self.names +\n            [c for c in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE") if c in metadata])\n        self._validated_specs_hash = None\n        self._panel_hash = None\n        self._panel_cache_key = None\n        self._key_hash = None\n\n    def __len__(self):\n        return len(self.metadata)\n\n    @property\n    def IS_OBSERVED(self):\n        return self.metadata.IS_OBSERVED\n\n    def __contains__(self, key):\n        return key in self.columns\n\n    def __getitem__(self, key):\n        if isinstance(key, str) and key in self.metadata:\n            return self.metadata[key]\n        if isinstance(key, list) and all(k in self.metadata for k in key):\n            return self.metadata[key]\n        raise TypeError("DiskPanel features require bounded iter_frames(), not a full frame copy")\n\n    def iter_frames(self, rows=ARRAY_ROW_CHUNK):\n        for start in range(0, len(self), rows):\n            stop = min(start + rows, len(self))\n            part = self.metadata.iloc[start:stop].reset_index(drop=True).copy()\n            features = pd.DataFrame(self.values[start:stop], columns=self.names, copy=False)\n            yield pd.concat([part, features], axis=1)[list(self.columns)]\n\n    def close(self):\n        release_array(self.values)\nRESERVED_COLUMNS = frozenset({\n    "PATIENT_ID", "AS_OF_DATE", "END_DT", "RESP", "SPLIT", "TARGET", "LABEL",\n    "OUTCOME", "SCORE", "PREDICTION", "IS_OBSERVED", "MAX_EVENT_DATE",\n    "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE", "TIME_STEP", "SNAPSHOT_ROW",\n})\nDATE_POLICIES = ("same_day_months", "month_end_history")\nRECOVERY_QUERY = """SELECT PROCEDURE_CATALOG, PROCEDURE_SCHEMA, PROCEDURE_NAME,\nARGUMENT_SIGNATURE, CREATED, LAST_ALTERED, PROCEDURE_DEFINITION\nFROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.PROCEDURES\nWHERE PROCEDURE_SCHEMA IN (\'DS_ML_DEV\', \'DS_ML\', \'DS_ML_PRO\', \'DS_ML_PRD\')\nAND PROCEDURE_NAME IN (\'FEATURE_CREATION_PROD\', \'FEATURE_CREATION_PROC\', \'FEATURE_ENGINEERING_PROD\')\nORDER BY PROCEDURE_NAME, ARGUMENT_SIGNATURE"""\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef stable_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                    allow_nan=False, default=str).encode()).hexdigest()\n\n\ndef _dates(values, label):\n    result = pd.to_datetime(values, errors="raise")\n    _require(result.dt.tz is None, label + ": timezone conversion must be explicit")\n    _require(result.notna().all(), label + ": missing dates")\n    _require(result.eq(result.dt.normalize()).all(), label + ": expected exact dates, not intraday timestamps")\n    return result\n\n\ndef _patients(values):\n    _require(values.notna().all(), "Missing patient keys")\n    \n    out = values.map(str)\n    _require(out.str.len().gt(0).all(), "Empty patient keys")\n    return out\n\n\ndef normalize_snapshots(metadata):\n    """Keep input order exactly; reject duplicate snapshots and cross-split patients."""\n    needed = ["PATIENT_ID", "END_DT", "RESP", "SPLIT"]\n    _require(isinstance(metadata, pd.DataFrame) and not metadata.columns.duplicated().any(),\n             "Snapshot metadata must be a DataFrame with unique columns")\n    _require(set(needed).issubset(metadata.columns), "Metadata requires PATIENT_ID, END_DT, RESP, SPLIT")\n    frame = metadata[needed].copy().reset_index(drop=True)\n    _require(len(frame) > 0, "No snapshots")\n    frame["PATIENT_ID"] = _patients(frame.PATIENT_ID)\n    frame["END_DT"] = _dates(frame.END_DT, "END_DT")\n    _require(not frame.duplicated(["PATIENT_ID", "END_DT"]).any(), "Duplicate snapshot keys")\n    _require(frame.RESP.notna().all() and frame.RESP.isin([0, 1]).all(), "RESP must be binary without missing labels")\n    frame["RESP"] = frame.RESP.astype(np.int64)\n    _require(frame.SPLIT.notna().all(), "Missing split labels")\n    frame["SPLIT"] = frame.SPLIT.astype(str).str.lower().replace({"val": "validation", "valid": "validation"})\n    _require(frame.SPLIT.isin(["train", "validation", "test"]).all(), "Unknown split label")\n    _require(frame.groupby("PATIENT_ID").SPLIT.nunique().eq(1).all(), "Patient appears in multiple splits")\n    return frame\n\n\ndef train_patient_hash(metadata):\n    """The canonical scope fingerprint for any upstream TRAIN-fitted encoding."""\n    meta = normalize_snapshots(metadata)\n    return stable_hash(sorted(meta.loc[meta.SPLIT.eq("train"), "PATIENT_ID"].unique().tolist()))\n\n\ndef verify_fitting_scope(contract, metadata):\n    contract = validate_contract(contract)\n    if contract["fit_scope"] == "train_only_fitted":\n        _require(contract["fitting_patients_sha256"] == train_patient_hash(metadata),\n                 "Upstream learned feature fitting patients differ from current TRAIN patients")\n    return contract\n\n\ndef make_requests(metadata, steps=12, date_policy="same_day_months"):\n    """Return deduplicated extraction keys and a row-aligned snapshot/date grid.\n\n    TIME_STEP zero is newest, preserving the source tensor convention. Older\n    dates are independently derived from the original END_DT, not iterated from\n    previously clamped dates. No label or split is sent to the extractor.\n    """\n    _require(isinstance(steps, int) and 1 <= steps <= 120, "Invalid number of steps")\n    _require(date_policy in DATE_POLICIES, "Unknown historical date policy")\n    meta = normalize_snapshots(metadata)\n    parts = []\n    for step in range(steps):\n        part = meta[["PATIENT_ID", "END_DT"]].copy()\n        part["SNAPSHOT_ROW"] = np.arange(len(meta), dtype=np.int64)\n        part["TIME_STEP"] = step\n        if step == 0:\n            part["AS_OF_DATE"] = part.END_DT\n        elif date_policy == "same_day_months":\n            part["AS_OF_DATE"] = part.END_DT.map(lambda date: date - pd.DateOffset(months=step))\n        else:\n            part["AS_OF_DATE"] = (part.END_DT.dt.to_period("M") - step).dt.to_timestamp(how="end").dt.normalize()\n        parts.append(part)\n    grid = pd.concat(parts, ignore_index=True).sort_values(["SNAPSHOT_ROW", "TIME_STEP"]).reset_index(drop=True)\n    _require(grid.AS_OF_DATE.le(grid.END_DT).all(), "Historical request exceeds snapshot cutoff")\n    requests = grid[KEYS].drop_duplicates().sort_values(KEYS).reset_index(drop=True)\n    return requests, grid\n\n\ndef validate_feature_specs(specs):\n    """Explicit, ordered numeric feature contract; no clinical-name substring filtering."""\n    _require(isinstance(specs, (list, tuple)) and len(specs) > 0, "Feature specifications are required")\n    clean = []\n    for original in specs:\n        _require(isinstance(original, dict), "Feature specification must be a dictionary")\n        item = dict(original)\n        name = item.get("name")\n        _require(isinstance(name, str) and bool(name.strip()), "Missing feature name")\n        _require(name.upper() not in RESERVED_COLUMNS, "Identifier/target/provenance column cannot be a feature: " + name)\n        _require(item.get("kind") in ("count", "binary", "continuous"),\n                 name + ": declare count, binary or continuous; categorical fields require a TRAIN-only encoder")\n        _require(isinstance(item.get("definition"), str) and bool(item["definition"].strip()),\n                 name + ": source feature definition/evidence required")\n        clean.append(item)\n    names = [f["name"] for f in clean]\n    _require(len(names) == len(set(names)), "Duplicate feature names")\n    return clean\n\n\ndef infer_numeric_feature_specs(frame, definitions, exact_exclusions=()):\n    """Create continuous declarations only for explicitly documented numeric fields.\n\n    This does not infer encoding, count or binary semantics from feature names.\n    Definitions must map every selected name to a meaningful source description.\n    Use explicit specs for validated count/binary domains.\n    """\n    excluded = RESERVED_COLUMNS | {str(x).upper() for x in exact_exclusions}\n    selected = [c for c in frame.columns if str(c).upper() not in excluded]\n    specs = []\n    for name in selected:\n        _require(name in definitions, "Missing feature definition: " + name)\n        pd.to_numeric(frame[name], errors="raise")\n        specs.append({"name": name, "kind": "continuous", "definition": definitions[name]})\n    return validate_feature_specs(specs)\n\n\ndef validate_contract(contract):\n    _require(isinstance(contract, dict), "Historical extraction contract is required")\n    result = dict(contract)\n    required = ["source_id", "source_version", "extraction_sha256", "cutoff_rule", "fit_scope", "feature_semantics_evidence"]\n    for key in required:\n        _require(isinstance(result.get(key), str) and bool(result[key].strip()), "Missing extraction contract field: " + key)\n    _require(bool(re.fullmatch(r"[0-9a-f]{64}", result["extraction_sha256"])), "extraction_sha256 must hash actual source extraction code")\n    _require(result["fit_scope"] in ("deterministic_raw", "train_only_fitted"), "Unknown or unsafe AutoML fitting scope")\n    if result["fit_scope"] == "train_only_fitted":\n        _require(bool(re.fullmatch(r"[0-9a-f]{64}", result.get("fitting_patients_sha256", ""))),\n                 "Learned upstream features require fitting_patients_sha256")\n    return result\n\n\ndef _normalize_keys(frame):\n    _require(isinstance(frame, pd.DataFrame) and not frame.columns.duplicated().any(), "Invalid panel/request columns")\n    _require(set(KEYS).issubset(frame.columns), "Historical panel requires PATIENT_ID and AS_OF_DATE")\n    result = frame.copy().reset_index(drop=True)\n    result["PATIENT_ID"] = _patients(result.PATIENT_ID)\n    result["AS_OF_DATE"] = _dates(result.AS_OF_DATE, "AS_OF_DATE")\n    _require(not result.duplicated(KEYS).any(), "Duplicate patient/date panel keys")\n    return result\n\n\ndef validate_panel(panel, specs):\n    specs = validate_feature_specs(specs)\n    if isinstance(panel, DiskPanel):\n        names = [s["name"] for s in specs]\n        _require(names == panel.names, "Disk panel feature order differs")\n        key_frame = _normalize_keys(panel.metadata)\n        _require(key_frame[KEYS].equals(key_frame[KEYS].sort_values(KEYS).reset_index(drop=True)),\n                 "Disk panel keys must be sorted")\n        signature = (stable_hash(specs), frame_hash(panel.metadata))\n        if panel._validated_specs_hash != signature:\n            for part in panel.iter_frames():\n                validate_panel(part, specs)\n            panel._validated_specs_hash = signature\n            panel._panel_hash = None\n            panel._key_hash = frame_hash(panel.metadata[KEYS])\n        return panel\n    frame = _normalize_keys(panel)\n    names = [s["name"] for s in specs]\n    _require(set(names + ["IS_OBSERVED"]).issubset(frame.columns), "Panel lacks feature fields or explicit IS_OBSERVED coverage")\n    _require(frame.IS_OBSERVED.notna().all() and frame.IS_OBSERVED.isin([0, 1, False, True]).all(),\n             "Every panel row requires explicit observation coverage")\n    frame["IS_OBSERVED"] = frame.IS_OBSERVED.astype(bool)\n    observed = frame.IS_OBSERVED.to_numpy()\n    for spec in specs:\n        name = spec["name"]\n        values = pd.to_numeric(frame[name], errors="raise").to_numpy(dtype=np.float64, na_value=np.nan)\n        _require(not np.isinf(values).any(), name + ": infinite feature value")\n        _require(np.isnan(values[~observed]).all(), name + ": unobserved history must remain missing, not copied or zero-filled")\n        finite = values[np.isfinite(values)]\n        if spec["kind"] == "count":\n            _require(np.all((finite >= 0) & (finite == np.floor(finite))), name + ": invalid count values")\n        elif spec["kind"] == "binary":\n            _require(np.isin(finite, [0., 1.]).all(), name + ": invalid binary values")\n        frame[name] = values\n    for col in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE"):\n        if col in frame:\n            dates = pd.to_datetime(frame[col], errors="raise")\n            _require(dates.dt.tz is None, col + ": timezone conversion must be explicit")\n            _require(dates.dropna().eq(dates.dropna().dt.normalize()).all(), col + ": expected exact dates")\n            _require((dates.isna() | dates.le(frame.AS_OF_DATE)).all(), col + ": future information at historical cutoff")\n            _require(dates[~observed].isna().all(), col + ": unavailable rows cannot contain event provenance")\n            frame[col] = dates\n    keep = KEYS + ["IS_OBSERVED"] + names + [c for c in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE") if c in frame]\n    return frame[keep].sort_values(KEYS).reset_index(drop=True)\n\n\ndef canonical_row_bytes(row):\n    canonical = []\n    for value in row:\n        if pd.isna(value):\n            canonical.append(None)\n        elif isinstance(value, (pd.Timestamp, np.datetime64)):\n            canonical.append(pd.Timestamp(value).isoformat())\n        elif isinstance(value, (np.bool_, bool)):\n            canonical.append(bool(value))\n        elif isinstance(value, (float, np.floating)):\n            canonical.append(float(value))\n        elif isinstance(value, (int, np.integer)):\n            canonical.append(int(value))\n        else:\n            canonical.append(str(value))\n    return json.dumps(canonical, separators=(",", ":"), allow_nan=False).encode() + b"\\n"\n\n\ndef frame_hash(frame):\n    """Fingerprint content and ordering without returning any patient-level data."""\n    if isinstance(frame, DiskPanel):\n        \n        \n        \n        cache_key=(tuple(frame.columns),tuple(frame.names),frame_hash(frame.metadata))\n        if frame._panel_hash is not None and frame._panel_cache_key == cache_key:\n            return frame._panel_hash\n    h = hashlib.sha256(json.dumps(list(frame.columns), separators=(",", ":")).encode())\n    if isinstance(frame, DiskPanel):\n        for part in frame.iter_frames():\n            for row in part.itertuples(index=False, name=None):\n                h.update(canonical_row_bytes(row))\n        frame._panel_hash = h.hexdigest()\n        frame._panel_cache_key = cache_key\n        return frame._panel_hash\n    for row in frame.itertuples(index=False, name=None):\n        h.update(canonical_row_bytes(row))\n    return h.hexdigest()\n\n\ndef make_panel_manifest(panel, specs, contract):\n    specs = validate_feature_specs(specs)\n    contract = validate_contract(contract)\n    frame = validate_panel(panel, specs)\n    return {"schema": 1, "contract": contract, "contract_sha256": stable_hash(contract),\n            "feature_specs": specs, "feature_specs_sha256": stable_hash(specs),\n            "panel_sha256": frame_hash(frame), "key_sha256": frame._key_hash if isinstance(frame, DiskPanel) else frame_hash(frame[KEYS]),\n            "rows": len(frame), "observed_rows": int(frame.IS_OBSERVED.sum()),\n            "availability_provenance": "MAX_AVAILABLE_DATE present; nulls remain unresolved" if "MAX_AVAILABLE_DATE" in frame else "Still unresolved: source availability clock not supplied",\n            "event_provenance": "MAX_EVENT_DATE checked where supplied" if "MAX_EVENT_DATE" in frame else "Contract-declared only; event maxima not supplied"}\n\n\ndef obtain_panel(requests, specs, contract, cached_panel=None, cached_manifest=None,\n                 extractor: Callable | None = None):\n    """Reuse checked patient/date rows and extract missing keys once.\n\n    An extractor receives *only* keys and must explicitly return every requested\n    row, including unavailable history. It may not silently omit incomplete rows.\n    The supplied manifest is provenance evidence, not automatic verification that\n    the warehouse extraction really implemented the declared semantics.\n    """\n    started = time.perf_counter()\n    req = _normalize_keys(requests[KEYS]).sort_values(KEYS).reset_index(drop=True)\n    _require(len(req) > 0, "No historical extraction keys")\n    specs = validate_feature_specs(specs)\n    contract = validate_contract(contract)\n    if cached_panel is None:\n        _require(cached_manifest is None, "Manifest supplied without cached values")\n        cached = None\n        missing = req\n    else:\n        _require(isinstance(cached_manifest, dict), "Cached data requires its original definition/content manifest")\n        cached = validate_panel(cached_panel, specs)\n        actual = make_panel_manifest(cached, specs, contract)\n        for key in ("schema", "contract_sha256", "feature_specs_sha256", "panel_sha256", "key_sha256", "rows"):\n            _require(actual[key] == cached_manifest.get(key), "Cached panel mismatch: " + key)\n        missing = req.merge(cached[KEYS], on=KEYS, how="left", indicator=True, validate="one_to_one")\n        missing = missing.loc[missing._merge.eq("left_only"), KEYS].reset_index(drop=True)\n    if len(missing):\n        _require(callable(extractor),\n                 f"Historical AutoML extraction required for {len(missing)} unique patient/date keys. "\n                 "Provide the recovered source query/procedure with exact historical cutoff semantics; a current snapshot table cannot fill history.")\n        fresh = validate_panel(extractor(missing.copy()), specs)\n        coverage = missing.merge(fresh[KEYS], on=KEYS, how="outer", indicator=True, validate="one_to_one")\n        _require(coverage._merge.eq("both").all(), "Extractor must return exactly the requested keys, including explicit unavailable rows")\n        if cached is not None:\n            _require(list(cached.columns) == list(fresh.columns), "Fresh/cached provenance schemas differ; resolve before combining")\n            combined = combine_panels(cached, fresh) if isinstance(cached, DiskPanel) or isinstance(fresh, DiskPanel) else pd.concat([cached, fresh], ignore_index=True)\n        else:\n            combined = fresh\n    else:\n        combined = cached\n    combined = validate_panel(combined, specs)\n    manifest = make_panel_manifest(combined, specs, contract)\n    manifest["extraction_report"] = {\n        "requested_unique_keys": len(req), "cache_keys_reused": len(req) - len(missing),\n        "keys_extracted": len(missing), "panel_cache_rows": len(combined),\n        "elapsed_seconds": time.perf_counter() - started,\n    }\n    return combined, manifest\n\n\ndef combine_panels(left, right):\n    disk = left if isinstance(left, DiskPanel) else right\n    names = disk.names\n    def metadata(part):\n        return part.metadata if isinstance(part, DiskPanel) else part[[c for c in part if c not in names]]\n    meta = pd.concat([metadata(left), metadata(right)], ignore_index=True)\n    order = meta.sort_values(KEYS).index.to_numpy()\n    inverse = np.empty(len(order), dtype=np.int64); inverse[order] = np.arange(len(order))\n    values = disk_array((len(meta),len(names)), np.float64, \'tak861-panel-\')\n    offset = 0\n    for part in (left, right):\n        for start in range(0,len(part),ARRAY_ROW_CHUNK):\n            stop = min(start+ARRAY_ROW_CHUNK,len(part))\n            block = part.values[start:stop] if isinstance(part,DiskPanel) else part.iloc[start:stop][names].to_numpy(dtype=np.float64)\n            values[inverse[offset+start:offset+stop]] = block\n        offset += len(part)\n    values=readonly_array(values)\n    return DiskPanel(meta.iloc[order].reset_index(drop=True),values,names)\n\n\ndef assemble_inputs(metadata, panel, specs, steps=12, date_policy="same_day_months"):\n    """Assemble S=Nx1xF and T=Nx12xF with common ordering and coverage masks."""\n    meta = normalize_snapshots(metadata)\n    specs = validate_feature_specs(specs)\n    frame = validate_panel(panel, specs)\n    requests, grid = make_requests(meta, steps, date_policy)\n    names = [s["name"] for s in specs]\n    panel_keys = frame.metadata[KEYS] if isinstance(frame,DiskPanel) else frame[KEYS]\n    order = pd.MultiIndex.from_frame(panel_keys).get_indexer(pd.MultiIndex.from_frame(grid[KEYS]))\n    _require((order >= 0).all(), "Historical panel does not cover every requested patient/date")\n    x = disk_array((len(meta), steps, len(names)), np.float32, \'tak861-sequence-\')\n    summary = disk_array((len(meta),1,len(names)),np.float32,\'tak861-summary-\')\n    observed = frame.IS_OBSERVED.to_numpy(dtype=bool)\n    mask = observed[order].reshape(len(meta),steps)\n    flat = x.reshape(-1,len(names))\n    try:\n        for start in range(0,len(order),ARRAY_ROW_CHUNK):\n            stop = min(start+ARRAY_ROW_CHUNK,len(order))\n            rows = order[start:stop]\n            block = frame.values[rows].astype(np.float32) if isinstance(frame,DiskPanel) else frame.iloc[rows][names].to_numpy(dtype=np.float32)\n            _require(not np.isinf(block).any(), "Numeric features overflow float32; inspect source scale before modeling")\n            _require(np.isnan(block[~observed[rows]]).all(), "Unobserved steps unexpectedly have values")\n            flat[start:stop] = block\n        for start in range(0,len(meta),ARRAY_ROW_CHUNK):\n            summary[start:start+ARRAY_ROW_CHUNK] = x[start:start+ARRAY_ROW_CHUNK,:1,:]\n        x.flush(); summary.flush()\n    except BaseException:\n        release_array(summary); release_array(x)\n        raise\n    \n    \n    report = {"snapshots": len(meta), "feature_count": len(names), "steps": steps,\n              "unique_patient_dates": len(requests), "sequence_positions": len(grid),\n              "reused_positions": len(grid) - len(requests),\n              "unobserved_steps": int((~mask).sum()),\n              "snapshots_without_current_automl_evidence": int((~mask[:, 0]).sum()),\n              "snapshots_with_incomplete_history": int((~mask).any(axis=1).sum()),\n              "date_policy": date_policy, "orientation": "0=newest",\n              "sequence_kind": "Historical as-of feature summaries; lookbacks may overlap, not disjoint monthly counts",\n              "feature_order_sha256": stable_hash(names),\n              "feature_specs_sha256": stable_hash(specs)}\n    return {"metadata": meta, "y": meta.RESP.to_numpy(dtype=np.int64),\n            "indices": {s: np.flatnonzero(meta.SPLIT.to_numpy() == s) for s in ("train", "validation", "test")},\n            "X_summary": summary, "X_sequence": x,\n            "summary_mask": mask[:, :1].copy(), "sequence_mask": mask,\n            "feature_names": names, "feature_specs": specs,\n            "hashes": {"metadata_sha256": frame_hash(meta), "panel_sha256": frame_hash(frame),\n                       "requests_sha256": frame_hash(requests), "feature_specs_sha256": stable_hash(specs),\n                       "grid_sha256": frame_hash(grid)}, "report": report}\n\n\ndef make_query_extractor(read_query, sql_template, chunk_size=1000):\n    """Bind recovered read-only SQL using a keys-only request CTE.\n\n    sql_template must contain the literal ``{request_sql}`` exactly once, e.g.\n    ``WITH request AS ({request_sql}) SELECT ...``. It must return the standard\n    panel columns. The actual feature calculation is supplied/recovered source\n    code, never an invented fallback. Reader can return pandas or Spark frames.\n    """\n    _require(callable(read_query), "A source query reader is required")\n    _require(isinstance(sql_template, str) and sql_template.count("{request_sql}") == 1,\n             "Recovered SQL must contain exactly one {request_sql} placeholder")\n    _require(isinstance(chunk_size, int) and 1 <= chunk_size <= 10000, "Invalid query chunk size")\n\n    def extractor(requests):\n        req = _normalize_keys(requests[KEYS])\n        outputs = []\n        for start in range(0, len(req), chunk_size):\n            values = []\n            for patient, date in req.iloc[start:start + chunk_size].itertuples(index=False, name=None):\n                \n                patient_sql = patient.replace("\'", "\'\'")\n                values.append(f"SELECT \'{patient_sql}\' AS PATIENT_ID, TO_DATE(\'{date:%Y-%m-%d}\') AS AS_OF_DATE")\n            query = sql_template.replace("{request_sql}", " UNION ALL ".join(values))\n            result = read_query(query)\n            outputs.append(result if isinstance(result, pd.DataFrame) else result.toPandas())\n        _require(outputs, "No historical keys requested")\n        return pd.concat(outputs, ignore_index=True)\n\n    extractor.extraction_sha256 = hashlib.sha256(sql_template.encode()).hexdigest()\n    return extractor\n', 'count_loader': '"""Reuse saved raw monthly counts, never a fitted model or prior predictions.\n\nAdapted from rigorous_transformer.workflow.load_count_inputs. Four read-only\ntables: FEATURE_MAP, SNAPSHOTS, PATIENT_SPLIT, TENSOR_MONTHLY. The mask means an\nexplicit saved calendar position exists, NOT confirmed clinical observation.\n"""\nimport hashlib\nimport json\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\ninputs = None  \n\n\ndef array_sha256(value):\n    hasher = hashlib.sha256()\n    class Sink:\n        def write(self, block):\n            hasher.update(block)\n            return len(block)\n    np.save(Sink(), np.asarray(value), allow_pickle=False)\n    return hasher.hexdigest()\n\n\ndef load_count_inputs(read_sf, source_prefix, source_version):\n    """read_sf(suffix) returns Spark frames from the configured source prefix."""\n    from pyspark.sql import functions as F\n    if not source_prefix or not source_version:\n        raise ValueError("Declare exact reused count source and version")\n    mapping = [r.asDict() for r in read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()]\n    if len(mapping) != 1028:\n        raise ValueError("The count reference requires the full 1,028 feature universe")\n    features = [r["FEATURE_NAME"] for r in mapping]\n    aliases = [r["FEATURE_COLUMN"] for r in mapping]\n    if ([r["FEATURE_INDEX"] for r in mapping] != list(range(1028))\n            or aliases != [f"F{i:04d}" for i in range(1028)] or len(set(features)) != 1028):\n        raise ValueError("Count feature order/index/alias mismatch")\n    frozen_raw = read_sf("PATIENT_SPLIT").select("PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()\n    if frozen_raw.SPLIT_CONFIG.isna().any() or frozen_raw.SPLIT_CONFIG.nunique() != 1:\n        raise ValueError("Count split creation configurations are inconsistent")\n    split_creation = json.loads(frozen_raw.SPLIT_CONFIG.iloc[0])\n    if split_creation.get("n_timesteps") != 12:\n        raise ValueError("Count source split does not declare 12 monthly timesteps")\n    frozen = inputs.normalize_snapshots(frozen_raw).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n    source_raw = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()\n    source_raw["SPLIT"] = "train"\n    source = inputs.normalize_snapshots(source_raw).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n    if not source[["PATIENT_ID", "END_DT", "RESP"]].equals(frozen[["PATIENT_ID", "END_DT", "RESP"]]):\n        raise ValueError("Count source and frozen split cohort/labels differ")\n    monthly = read_sf("TENSOR_MONTHLY")\n    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):\n        raise ValueError("Count tensor schema differs from feature map")\n    grouped = monthly.groupBy("PATIENT_ID", "END_DT", "RESP").agg(F.collect_list(F.struct(\n        F.col("TIME_STEP").alias("T"), F.array(*[F.col(name).cast("double") for name in aliases]).alias("V"))).alias("SEQUENCE"))\n    temporary = tempfile.TemporaryDirectory(prefix="tak861_automl_count_reference_")\n    values = np.memmap(Path(temporary.name) / "counts.f32", dtype="<f4", mode="w+", shape=(len(frozen), 12, 1028))\n    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(frozen.itertuples())}\n    seen = np.zeros(len(frozen), dtype=bool)\n    try:\n        for row in grouped.toLocalIterator(prefetchPartitions=False):\n            date = pd.Timestamp(row["END_DT"])\n            if date.tzinfo is not None or date != date.normalize():\n                raise ValueError("Count tensor dates must be exact dates")\n            key = (str(row["PATIENT_ID"]), date)\n            if key not in positions:\n                raise ValueError("Count tensor contains unexpected snapshot")\n            i = positions[key]\n            seq = sorted(row["SEQUENCE"], key=lambda r: r["T"])\n            if seen[i] or row["RESP"] != frozen.RESP.iloc[i] or [r["T"] for r in seq] != list(range(12)):\n                raise ValueError("Count tensor duplicate, changed label or incomplete calendar positions")\n            block = np.asarray([r["V"] for r in seq], dtype=np.float64)\n            finite = np.isfinite(block)\n            if block.shape != (12, 1028) or not finite.all() or (block[finite] < 0).any() or (block[finite] != np.floor(block[finite])).any():\n                raise ValueError("Invalid raw count values")\n            converted = block.astype(np.float32)\n            if (converted[finite].astype(np.float64) != block[finite]).any():\n                raise ValueError("Count exceeds exact float32 representation")\n            values[i] = converted\n            seen[i] = True\n        if not seen.all():\n            raise ValueError("Count source omits snapshots or zero-activity calendar rows")\n        values.flush()\n        values.flags.writeable = False\n        \n        \n        valid = np.ones(values.shape[:2], dtype=bool)\n        _, grid = inputs.make_requests(frozen, date_policy="month_end_history")\n        current_order_hash = hashlib.sha256(json.dumps(features, ensure_ascii=False).encode()).hexdigest()\n        contract = dict(source_id=source_prefix, source_version=source_version,\n            content_sha256=array_sha256(values), feature_names=features,\n            provenance_status="Recovered source construction: monthly calendar buckets; historical execution/availability unresolved",\n            orientation="0=newest", date_policy="month_end_history", n_steps=12,\n            asof_grid_sha256=inputs.frame_hash(grid),\n            mask_kind="saved_calendar_position_not_clinical_observation",\n            mask_sha256=array_sha256(valid),\n            metadata_sha256=inputs.frame_hash(frozen),\n            historical_feature_order_match=(split_creation.get("feature_order_sha256") == current_order_hash),\n            split_creation=split_creation,\n            calendar_evidence="Original count initialization: TIME_STEP calendar-month difference; current bucket truncated END_DT, older buckets complete months")\n        return dict(X=values, metadata=frozen, features=features, valid=valid, contract=contract,\n                    temporary_directory=temporary)\n    except BaseException:\n        values._mmap.close()\n        temporary.cleanup()\n        raise\n\n\ndef release_count_inputs(data):\n    if data and isinstance(data.get("X"), np.memmap):\n        data["X"]._mmap.close()\n        data["temporary_directory"].cleanup()\n', 'modeling': '"""TRAIN-only preparation and bounded fresh fits for the two AutoML experiments.\n\nSummary uses one token; history uses twelve genuinely historical tokens. Zero is\na measurement, never a padding rule. The caller must supply date-validated masks,\npatient-disjoint splits and a frozen source feature order. This module never\naccepts TEST data while fitting. No historical fitted state is reused.\n"""\nimport copy\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport importlib.metadata\nimport random\nimport time\nimport warnings\nimport tempfile\nimport weakref\n\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score, log_loss\nfrom sklearn.model_selection import StratifiedGroupKFold\n\n\n\nMEMORY_ROW_BLOCK = 512\nMEMORY_FEATURE_BLOCK = 64\nMEMORY_MAP_THRESHOLD = 8 * 1024 * 1024\n\n\ndef _close_backing(path, mapping):\n    try:\n        mapping.close()\n    finally:\n        try:\n            os.unlink(path)\n        except FileNotFoundError:\n            pass\n\n\ndef allocate_array(shape, dtype=np.float32):\n    """Allocate large intermediates on local disk, with automatic cleanup."""\n    shape, dtype = tuple(shape), np.dtype(dtype)\n    if int(np.prod(shape)) * dtype.itemsize <= MEMORY_MAP_THRESHOLD:\n        return np.empty(shape, dtype=dtype)\n    handle, path = tempfile.mkstemp(prefix="tak861_model_", suffix=".npy")\n    os.close(handle)\n    try:\n        out = np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n    except BaseException:\n        os.unlink(path)\n        raise\n    out._temporary_finalizer = weakref.finalize(out, _close_backing, path, out._mmap)\n    return out\n\n\ndef release_array(value):\n    """Release only the allocating owner, after all consumers/views finish.\n\n    Closing a mapping through a view invalidates its still-live owner and can\n    crash a later NumPy read. Views must never transfer cleanup ownership.\n    """\n    original = value\n    visited = set()\n    while value is not None and id(value) not in visited:\n        visited.add(id(value))\n        finalizer = getattr(value, "_temporary_finalizer", None)\n        if finalizer is not None:\n            registration = finalizer.peek()\n            if value is not original or (registration is not None and registration[0] is not original):\n                raise ValueError("Cannot release a view; release the owning array after all views are finished")\n            finalizer()\n            return\n        value = getattr(value, "base", None)\n\n\ndef take_rows(value, indices, dtype=None):\n    """Gather a split/fold without allocating its whole tensor on the driver."""\n    indices = np.asarray(indices, dtype=np.int64)\n    out = allocate_array((len(indices),) + value.shape[1:], value.dtype if dtype is None else dtype)\n    for start in range(0, len(indices), MEMORY_ROW_BLOCK):\n        out[start:start+MEMORY_ROW_BLOCK] = value[indices[start:start+MEMORY_ROW_BLOCK]]\n    return out\n\n\ndef stable_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef lift_at_fraction(y, scores, fraction=.10):\n    """Snapshot lift at exact ceil(fraction*N), with label-blind row-order ties.\n\n    Scores may be arbitrary finite ranking values. This is a snapshot-capacity\n    estimand, not unique-patient interventions or a patient-weighted loss.\n    """\n    y, scores = np.asarray(y), np.asarray(scores)\n    if (y.ndim != 1 or len(y) == 0 or scores.shape != y.shape\n            or not set(np.unique(y)) <= {0, 1} or y.sum() == 0\n            or not np.isfinite(scores).all() or not np.isfinite(fraction)\n            or not 0 < fraction <= 1):\n        raise ValueError("Lift requires aligned binary labels with positives, finite scores and fraction in (0,1]")\n    k = max(1, int(np.ceil(float(fraction) * len(y))))\n    order = np.lexsort((np.arange(len(y)), -scores.astype(float)))\n    return float((y[order[:k]].sum() / k) / y.mean())\n\n\ndef _array(x, valid=None, *, finite=False, allow_empty=False, row_indices=None):\n    x = np.asarray(x)\n    if x.ndim == 2:\n        x = x[:, None, :]\n    if x.ndim != 3 or min(x.shape) < 1 or not np.issubdtype(x.dtype, np.number):\n        raise ValueError("Expected numeric N x T x F inputs")\n    if valid is None:\n        valid = np.ones(x.shape[:2], dtype=bool)\n    valid = np.asarray(valid)\n    if valid.dtype != bool or valid.shape != x.shape[:2]:\n        raise ValueError("Availability mask must be boolean N x T; True means available")\n    rows = None if row_indices is None else np.asarray(row_indices, dtype=np.int64)\n    if not allow_empty and not (valid if rows is None else valid[rows]).any(axis=1).all():\n        raise ValueError("A snapshot has no available timestep; resolve or explicitly exclude it upstream")\n    if np.iscomplexobj(x):\n        raise ValueError("Observed inputs must be real and finite (NaN allowed only before imputation)")\n    for start in range(0, len(x) if rows is None else len(rows), MEMORY_ROW_BLOCK):\n        selected = slice(start, start+MEMORY_ROW_BLOCK) if rows is None else rows[start:start+MEMORY_ROW_BLOCK]\n        observed = x[selected][valid[selected]]\n        if np.isinf(observed).any() or (finite and not np.isfinite(observed).all()):\n            raise ValueError("Observed inputs must be real and finite (NaN allowed only before imputation)")\n    return x, valid\n\n\ndef sample_weights(groups, mode="patient"):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) == 0 or any(x is None for x in groups):\n        raise ValueError("Expected nonempty patient IDs")\n    if mode == "snapshot":\n        return np.ones(len(groups), dtype=np.float32)\n    if mode != "patient":\n        raise ValueError("weighting must be patient or snapshot")\n    _, inverse, counts = np.unique(groups.astype(str), return_inverse=True, return_counts=True)\n    w = 1. / counts[inverse]\n    return (w / w.mean()).astype(np.float32)\n\n\ndef class_weight(y, weights, balance):\n    y, weights = np.asarray(y), np.asarray(weights)\n    if (set(np.unique(y)) != {0, 1} or y.shape != weights.shape\n            or not np.isfinite(weights).all() or (weights <= 0).any()):\n        raise ValueError("Class weighting requires both binary classes and positive aligned weights")\n    ratio = float(weights[y == 0].sum() / weights[y == 1].sum())\n    if balance not in {"none", "sqrt", "full"}:\n        raise ValueError("balance must be none, sqrt or full")\n    return {"none": 1., "sqrt": ratio ** .5, "full": ratio}[balance]\n\n\ndef _kinds(feature_names, feature_kinds):\n    if feature_kinds is None:\n        return ["numeric"] * len(feature_names)\n    kinds = ([feature_kinds.get(name, "numeric") for name in feature_names]\n             if isinstance(feature_kinds, dict) else list(feature_kinds))\n    if len(kinds) != len(feature_names) or not set(kinds) <= {"numeric", "continuous", "count", "binary"}:\n        raise ValueError("Kinds must be numeric/continuous/count/binary; categorical values need an explicit encoding contract")\n    return kinds\n\n\ndef _fit_transform(summary, sequence, valid, feature_names, feature_kinds):\n    """Fit exact feature statistics, with sequential disk reads for large inputs."""\n    if sequence.nbytes <= MEMORY_MAP_THRESHOLD:\n        return _fit_transform_featurewise(summary, sequence, valid, feature_names, feature_kinds)\n    \n    \n    major = allocate_array((sequence.shape[2], sequence.shape[0], sequence.shape[1]), sequence.dtype)\n    try:\n        for start in range(0, len(sequence), MEMORY_ROW_BLOCK):\n            stop = min(start + MEMORY_ROW_BLOCK, len(sequence))\n            for feature in range(0, sequence.shape[2], MEMORY_FEATURE_BLOCK):\n                end = min(feature + MEMORY_FEATURE_BLOCK, sequence.shape[2])\n                major[feature:end, start:stop, :] = sequence[start:stop, :, feature:end].transpose(2, 0, 1)\n        return _fit_transform_featurewise(summary, sequence, valid, feature_names, feature_kinds,\n                                         feature_major=major)\n    finally:\n        release_array(major)\n\n\ndef _fit_transform_featurewise(summary, sequence, valid, feature_names, feature_kinds, feature_major=None):\n    """Fit one common transformation from TRAIN observations of both arms."""\n    \n    names, kinds = list(feature_names), _kinds(feature_names, feature_kinds)\n    quality, medians, centers, scales, actual_kinds, eligible, binary_contracts = [], [], [], [], [], [], []\n    seen = {}\n    for j, name in enumerate(names):\n        column = sequence[:, :, j] if feature_major is None else feature_major[j]\n        a = np.asarray(column[valid], dtype=np.float64)\n        observed = a[np.isfinite(a)]\n        unique, freq = np.unique(observed, return_counts=True)\n        kind = kinds[j]\n        binary_contract = kind == "binary"\n        if kind == "numeric":\n            kind = "binary" if len(unique) and set(unique) <= {0., 1.} else "continuous"\n        if kind == "binary" and not set(unique) <= {0., 1.}:\n            raise ValueError("Non-binary measurements in declared binary feature: " + name)\n        if kind == "count" and (np.any(observed < 0) or np.any(observed != np.floor(observed))):\n            raise ValueError("Noninteger or negative value in declared count feature: " + name)\n        transformed = np.log1p(a) if kind == "count" else a\n        median = float(np.nanmedian(transformed)) if len(observed) else 0.\n        imputed = np.where(np.isnan(transformed), median, transformed)\n        center = 0. if kind == "binary" else float(imputed.mean())\n        sd = float(imputed.std())\n        \n        scale = 1. if kind == "binary" else (max(sd, 1.) if kind == "count" else max(sd, 1e-6))\n        reason = "all_missing" if not len(observed) else ("constant" if len(unique) == 1 else "retained")\n        duplicate_of = None\n        canonical = np.nan_to_num(a, nan=0.)\n        digest = hashlib.sha256(canonical.tobytes() + np.isnan(a).tobytes()).hexdigest()\n        if reason == "retained" and digest in seen:\n            reason, duplicate_of = "exact_duplicate", names[seen[digest]]\n        elif reason == "retained":\n            seen[digest] = j\n            eligible.append(j)\n        quality.append(dict(feature=name, source_index=j, kind=kind, reason=reason,\n                            duplicate_of=duplicate_of, missing_fraction=float(np.isnan(a).mean()),\n                            observed_values=int(len(observed)), unique_values=int(len(unique)),\n                            dominant_fraction=float(freq.max() / freq.sum()) if len(freq) else None,\n                            near_constant=bool(len(freq) and freq.max() / freq.sum() >= .995),\n                            variance=float(np.var(observed)) if len(observed) else None,\n                            low_variance_flag=bool(len(observed) and np.var(observed) < 1e-6),\n                            min=float(observed.min()) if len(observed) else None,\n                            max=float(observed.max()) if len(observed) else None,\n                            p01=float(np.quantile(observed, .01)) if len(observed) else None,\n                            p99=float(np.quantile(observed, .99)) if len(observed) else None))\n        medians.append(median); centers.append(center); scales.append(scale); actual_kinds.append(kind)\n        binary_contracts.append(binary_contract)\n    if not eligible:\n        raise ValueError("No nonconstant nonduplicate feature survives TRAIN quality checks")\n    return dict(feature_names=names, kinds=actual_kinds, median=medians, center=centers,\n                scale=scales, eligible_indices=eligible, quality=quality, binary_contract=binary_contracts,\n                clipping="none", fitted_on="TRAIN valid sequence positions once (summary is step zero)",\n                near_constant_policy="flag only; preserve rare clinical signals")\n\n\ndef _transform_all(x, state, valid=None, *, row_indices=None, feature_indices=None):\n    if row_indices is not None and valid is not None and len(valid) == len(row_indices) and len(valid) != len(x):\n        full_mask = np.zeros(np.shape(x)[:2], dtype=bool)\n        full_mask[row_indices] = valid\n        valid = full_mask\n    \n    \n    x, valid = _array(x, valid, allow_empty=True, row_indices=row_indices)\n    if x.shape[-1] != len(state["feature_names"]):\n        raise ValueError("Raw input width differs from fitted feature order")\n    rows = np.arange(len(x)) if row_indices is None else np.asarray(row_indices, dtype=np.int64)\n    features = list(range(x.shape[-1])) if feature_indices is None else list(feature_indices)\n    output_positions = {j: out_j for out_j, j in enumerate(features)}\n    a = allocate_array((len(rows), x.shape[1], len(features)))\n    try:\n        for start in range(0, len(rows), MEMORY_ROW_BLOCK):\n            ids = rows[start:start+MEMORY_ROW_BLOCK]\n            mask = valid[ids]\n            unavailable = ~mask\n            \n            \n            for j in range(x.shape[-1]):\n                kind = state["kinds"][j]\n                column = np.asarray(x[ids, :, j], dtype=np.float64).copy()\n                column[unavailable] = 0.\n                if kind == "count":\n                    vals = column[mask]\n                    finite = vals[np.isfinite(vals)]\n                    if (finite < 0).any() or (finite != np.floor(finite)).any():\n                        raise ValueError("New data violates declared count semantics")\n                    np.log1p(column, out=column)\n                elif kind == "binary" and state.get("binary_contract", [False] * len(state["kinds"]))[j]:\n                    vals = column[mask]\n                    if not set(np.unique(vals[np.isfinite(vals)])) <= {0., 1.}:\n                        raise ValueError("New data violates binary semantics")\n                column[np.isnan(column)] = state["median"][j]\n                column -= state["center"][j]\n                column /= state["scale"][j]\n                column[unavailable] = 0.\n                if not np.isfinite(column).all():\n                    raise ValueError("Nonfinite transformed values")\n                cast_column = column.astype(np.float32)\n                if not np.isfinite(cast_column).all():\n                    raise ValueError("Transformed values overflow float32")\n                if j in output_positions:\n                    a[start:start+len(ids), :, output_positions[j]] = cast_column\n        return a\n    except BaseException:\n        release_array(a)\n        raise\n\n\ndef transform_inputs(x, preparation, valid=None, *, row_indices=None):\n    """Transform just the selected columns/split, in the frozen selected order."""\n    return _transform_all(x, preparation["transform"], valid, row_indices=row_indices,\n                          feature_indices=preparation["selected_indices"])\n\n\ndef _proxy(summary_transformed, sequence_transformed, valid):\n    """Same four float64 descriptors, bounded row/feature workspaces."""\n    output = allocate_array((len(sequence_transformed), sequence_transformed.shape[-1], 4), np.float64)\n    try:\n        for row in range(0, len(sequence_transformed), MEMORY_ROW_BLOCK):\n            stop = min(row+MEMORY_ROW_BLOCK, len(sequence_transformed))\n            mask = valid[row:stop, :, None]\n            count = np.maximum(valid[row:stop].sum(1)[:, None], 1)\n            t = np.arange(sequence_transformed.shape[1], dtype=float)[None, :, None]\n            tmean = np.where(mask, t, 0.).sum(1) / count\n            centered_t = t - tmean[:, None, :]\n            denom = np.where(mask, centered_t**2, 0.).sum(1)\n            for start in range(0, sequence_transformed.shape[-1], MEMORY_FEATURE_BLOCK):\n                end = min(start+MEMORY_FEATURE_BLOCK, sequence_transformed.shape[-1])\n                x = sequence_transformed[row:stop, :, start:end].astype(np.float64)\n                mean = np.where(mask, x, 0.).sum(1) / count\n                variance = np.where(mask, (x-mean[:, None, :])**2, 0.).sum(1) / count\n                slope = np.where(mask, centered_t * (x - mean[:, None, :]), 0.).sum(1) / np.maximum(denom, 1.)\n                output[row:stop, start:end, 0] = summary_transformed[row:stop, 0, start:end]\n                output[row:stop, start:end, 1] = mean\n                output[row:stop, start:end, 2] = np.sqrt(np.maximum(variance, 0))\n                output[row:stop, start:end, 3] = slope\n        return output\n    except BaseException:\n        release_array(output)\n        raise\n\n\ndef _transformed_proxy(summary, sequence, valid, state, rows=None):\n    a = _transform_all(summary, state, valid[:, :1], row_indices=rows)\n    b = None\n    try:\n        b = _transform_all(sequence, state, valid, row_indices=rows)\n        return _proxy(a, b, valid if rows is None else valid[rows])\n    finally:\n        release_array(a)\n        if b is not None:\n            release_array(b)\n\n\ndef _correlation(full_summary, eligible):\n    """Exact full-row Pearson diagnostic; center eligible columns on disk."""\n    active = np.asarray([i for i, j in enumerate(eligible)\n                         if np.ptp(full_summary[:, 0, j]) > 0], dtype=int)\n    if len(active) <= 1:\n        return active, None\n    \n    centered = allocate_array((len(active), len(full_summary)), np.float64)\n    try:\n        for i, source_index in enumerate(active):\n            col = np.asarray(full_summary[:, 0, eligible[source_index]], dtype=np.float64)\n            centered[i] = col - np.average(col)\n        corr = np.dot(centered, centered.T)\n        corr *= np.true_divide(1, len(full_summary)-1)\n        sd = np.sqrt(np.diag(corr))\n        corr /= sd[:, None]\n        corr /= sd[None, :]\n        np.clip(corr, -1, 1, out=corr)\n        return active, corr\n    finally:\n        release_array(centered)\n\n\ndef _proxy_design(proxy, columns):\n    """C-contiguous float64 solver matrix, populated without full fancy-index copy."""\n    out = allocate_array((len(proxy), len(columns) * proxy.shape[-1]), np.float64)\n    for start in range(0, len(proxy), MEMORY_ROW_BLOCK):\n        out[start:start+MEMORY_ROW_BLOCK] = proxy[start:start+MEMORY_ROW_BLOCK, columns, :].reshape(-1, out.shape[1])\n    return out\n\n\ndef _rank(proxy, y, indices, names, weights, fraction=.10):\n    scores = []\n    for j in indices:\n        descriptors, lifts = [], []\n        for d in range(proxy.shape[-1]):\n            a = proxy[:, j, d]\n            if len(np.unique(a)) <= 1:\n                ap_pos = ap_neg = float(np.average(y, weights=weights))\n                lifts.append(1.)\n            else:\n                ap_pos = float(average_precision_score(y, a, sample_weight=weights))\n                ap_neg = float(average_precision_score(y, -a, sample_weight=weights))\n                lifts.append(max(lift_at_fraction(y, a, fraction), lift_at_fraction(y, -a, fraction)))\n            descriptors.append(max(ap_pos, ap_neg))\n        scores.append(dict(index=int(j), feature=names[j], rank_score=max(lifts),\n                           descriptor_lift=dict(zip(["summary", "historical_mean", "historical_sd", "historical_slope"], lifts)),\n                           descriptor_ap=dict(zip(["summary", "historical_mean", "historical_sd", "historical_slope"], descriptors))))\n    return sorted(scores, key=lambda r: (-r["rank_score"], r["index"]))\n\n\ndef prepare_features(summary_train, sequence_train, y_train, patient_ids_train,\n                     feature_names, valid_train=None, feature_kinds=None, config=None, *, row_indices=None):\n    """Common feature subset, selected strictly inside TRAIN by grouped proxy CV.\n\n    The proxy uses summary, historical mean, standard deviation and time slope.\n    Groups of four descriptors rank/select original features, preserving features\n    that vary only historically. It does not claim a Transformer-optimal subset.\n    Fold-local preprocessing and ranking prevent fitting-fold target leakage.\n    """\n    cfg = dict(n_splits=3, seed=42, candidate_sizes=[16, 32, 64, 128, 256, 512, 1028],\n               proxy_C=.1, proxy_max_iter=1000, weighting="patient", primary_lift_fraction=.10)\n    incoming = config or {}\n    if set(incoming) - set(cfg):\n        raise ValueError("Unknown preparation configuration: " + str(set(incoming) - set(cfg)))\n    cfg.update(incoming)\n    if not np.isfinite(cfg["primary_lift_fraction"]) or not 0 < cfg["primary_lift_fraction"] <= 1:\n        raise ValueError("primary_lift_fraction must be in (0,1]")\n    owned_inputs = []\n    if row_indices is not None:\n        sequence_train = take_rows(sequence_train, row_indices)\n        summary_train = take_rows(summary_train, row_indices)\n        owned_inputs = [sequence_train, summary_train]\n        if valid_train is not None and len(valid_train) != len(row_indices):\n            valid_train = valid_train[row_indices]\n    sequence, valid = _array(sequence_train, valid_train, allow_empty=True)\n    summary, smask = _array(summary_train, valid[:, :1], allow_empty=True)\n    y, groups = np.asarray(y_train), np.asarray(patient_ids_train)\n    names = list(feature_names)\n    if (summary.shape[1] != 1 or sequence.shape[1] != 12 or summary.shape[0] != sequence.shape[0]\n            or summary.shape[-1] != sequence.shape[-1] or len(names) != summary.shape[-1]\n            or len(set(names)) != len(names) or y.shape != (len(summary),) or groups.shape != y.shape\n            or set(np.unique(y)) != {0, 1}):\n        raise ValueError("Aligned summary N x 1 x F, monthly N x 12 x F, labels and patient IDs required")\n    if any(not isinstance(n, str) or not n.strip() for n in names):\n        raise ValueError("Feature names must be nonempty unique strings")\n    if {"PATIENT_ID", "END_DT", "RESP", "SPLIT", "LABEL", "TARGET"} & {n.upper() for n in names}:\n        raise ValueError("Identifier or target is present in feature names")\n    if (type(cfg["n_splits"]) is not int or cfg["n_splits"] < 3\n            or any(type(k) is not int or k < 1 for k in cfg["candidate_sizes"])):\n        raise ValueError("At least three grouped folds and positive integer K candidates required")\n    started = time.perf_counter()\n    state = _fit_transform(summary, sequence, valid, names, feature_kinds)\n    eligible = state["eligible_indices"]\n    candidates = sorted({min(k, len(eligible)) for k in cfg["candidate_sizes"]} | {len(eligible)})\n    weights = sample_weights(groups, cfg["weighting"])\n    full_summary = _transform_all(summary, state, smask)\n    full_proxy = _transformed_proxy(summary, sequence, valid, state)\n    ranking = _rank(full_proxy, y, eligible, names, weights, cfg["primary_lift_fraction"])\n    release_array(full_proxy); del full_proxy\n    splitter = StratifiedGroupKFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["seed"])\n    cv_rows, fold_rankings = [], []\n    for fold, (ftrain, fval) in enumerate(splitter.split(summary, y, groups)):\n        if set(groups[ftrain]) & set(groups[fval]) or any(set(np.unique(y[idx])) != {0, 1} for idx in (ftrain, fval)):\n            raise ValueError("Grouped CV requires patient separation and both classes in every fold")\n        fold_sequence = take_rows(sequence, ftrain)\n        try:\n            fold_state = _fit_transform(None, fold_sequence, valid[ftrain], names, feature_kinds)\n        finally:\n            release_array(fold_sequence)\n        a = _transformed_proxy(summary, sequence, valid, fold_state, ftrain)\n        b = _transformed_proxy(summary, sequence, valid, fold_state, fval)\n        w = sample_weights(groups[ftrain], cfg["weighting"])\n        ranks = _rank(a, y[ftrain], fold_state["eligible_indices"], names, w, cfg["primary_lift_fraction"])\n        order = [r["index"] for r in ranks]\n        fold_rankings.append(order)\n        for k in candidates:\n            cols = order[:min(k, len(order))]\n            clf = LogisticRegression(C=cfg["proxy_C"], solver="lbfgs", max_iter=cfg["proxy_max_iter"],\n                                     random_state=cfg["seed"], tol=1e-4)\n            fit_design, validation_design = _proxy_design(a, cols), _proxy_design(b, cols)\n            try:\n                with warnings.catch_warnings(record=True) as caught:\n                    warnings.simplefilter("always")\n                    clf.fit(fit_design, y[ftrain], sample_weight=w)\n                score = clf.predict_proba(validation_design)[:, 1]\n            finally:\n                release_array(fit_design); release_array(validation_design)\n            cv_rows.append(dict(fold=fold, candidate_k=k, fitted_k=len(cols),\n                                lift=lift_at_fraction(y[fval], score, cfg["primary_lift_fraction"]),\n                                average_precision=float(average_precision_score(y[fval], score)),\n                                warnings=[str(w.message) for w in caught]))\n        release_array(a); release_array(b)\n        del a, b, clf\n    cv_summary = []\n    for k in candidates:\n        ap = np.asarray([r["average_precision"] for r in cv_rows if r["candidate_k"] == k])\n        lifts = np.asarray([r["lift"] for r in cv_rows if r["candidate_k"] == k])\n        cv_summary.append(dict(k=k, mean_ap=float(ap.mean()), se_ap=float(ap.std(ddof=1) / np.sqrt(len(ap))),\n                               mean_lift=float(lifts.mean()), se_lift=float(lifts.std(ddof=1) / np.sqrt(len(lifts)))))\n    best = max(cv_summary, key=lambda r: (r["mean_lift"], -r["k"]))\n    boundary = best["mean_lift"] - best["se_lift"]\n    selected_k = min(r["k"] for r in cv_summary if r["mean_lift"] >= boundary)\n    selected_indices = [r["index"] for r in ranking[:selected_k]]\n    for rank, row in enumerate(ranking, 1):\n        ranks = [order.index(row["index"]) + 1 for order in fold_rankings if row["index"] in order]\n        row.update(rank=rank, selected=row["index"] in selected_indices,\n                   fold_rank_mean=float(np.mean(ranks)) if ranks else None,\n                   fold_rank_sd=float(np.std(ranks)) if ranks else None,\n                   fold_eligible_count=len(ranks))\n    selected = set(selected_indices)\n    for row in state["quality"]:\n        row["selected"] = row["source_index"] in selected\n        row["final_reason"] = ("proxy_rank_below_selected_prefix" if row["reason"] == "retained"\n                               and not row["selected"] else row["reason"])\n    \n    active, corr = _correlation(full_summary, eligible)\n    release_array(full_summary)\n    correlations = []\n    condition = None\n    if len(active) > 1:\n        ii, jj = np.where(np.triu(np.abs(corr) >= .95, k=1))\n        correlations = [dict(feature_a=names[eligible[active[i]]], feature_b=names[eligible[active[j]]],\n                             pearson=float(corr[i, j])) for i, j in zip(ii, jj)]\n        eigenvalues = np.linalg.eigvalsh(corr)\n        condition = (None if eigenvalues[0] <= 1e-12 else float(eigenvalues[-1] / eigenvalues[0]))\n    result = dict(schema_version=1, transform=state, selected_indices=selected_indices,\n                  selected_names=[names[j] for j in selected_indices], quality=state["quality"],\n                  ranking=ranking, cv_rows=cv_rows, cv_summary=cv_summary, config=cfg,\n                  selection_rule="smallest K within one SE of best TRAIN-grouped-CV proxy lift at primary capacity",\n                  primary_lift_fraction=cfg["primary_lift_fraction"],\n                  rank_metric="max bidirectional snapshot lift across TRAIN summary/mean/SD/slope; AP secondary diagnostic",\n                  lift_tie_policy="frozen input row order; no label-based tie breaking",\n                  proxy_limitation="Grouped summary/mean/SD/slope proxy; not a claim of Transformer-optimal K",\n                  selection_used_test=False, selection_used_external_validation=False,\n                  train_snapshots=len(y), train_patients=len(np.unique(groups)),\n                  correlation_pairs_ge_095=correlations,\n                  correlation_condition_number=condition,\n                  singular_correlation=bool(condition is None and len(active) > 1),\n                  preprocessing_note="No clipping; binary preserved; log1p only declared counts; other numeric standardized",\n                  wall_seconds=time.perf_counter() - started)\n    result["preparation_sha256"] = stable_hash(result)\n    for owned in owned_inputs:\n        release_array(owned)\n    return result\n\n\nDEFAULT_ENCODER = dict(d_model=64, heads=4, ff=128, depth=1, dropout=.2, pooling="mean",\n                       weight_decay=1e-3, learning_rate=3e-4, batch_size=256,\n                       epochs=30, patience=5, gradient_clip=1., scheduler="plateau",\n                       weighting="patient", balance="sqrt", threads=8, device="cpu",\n                       optimizer_betas=[.9, .999], optimizer_eps=1e-8, primary_lift_fraction=.10)\n\n\ndef _config(config):\n    incoming = config or {}\n    if set(incoming) - set(DEFAULT_ENCODER):\n        raise ValueError("Unknown encoder settings: " + str(set(incoming) - set(DEFAULT_ENCODER)))\n    cfg = {**copy.deepcopy(DEFAULT_ENCODER), **copy.deepcopy(incoming)}\n    if not np.isfinite(cfg["primary_lift_fraction"]) or not 0 < cfg["primary_lift_fraction"] <= 1:\n        raise ValueError("primary_lift_fraction must be in (0,1]")\n    for name in ("d_model", "heads", "ff", "depth", "batch_size", "epochs", "patience", "threads"):\n        if type(cfg[name]) is not int or cfg[name] < 1:\n            raise ValueError(name + " must be a positive integer")\n    if cfg["d_model"] < 2 or cfg["d_model"] % cfg["heads"] or not 0 <= cfg["dropout"] < 1:\n        raise ValueError("Invalid encoder dimension, heads or dropout")\n    if cfg["pooling"] not in {"mean", "attention"} or cfg["scheduler"] not in {"none", "plateau"}:\n        raise ValueError("Unknown pooling/scheduler")\n    for name in ("weight_decay", "learning_rate", "gradient_clip", "optimizer_eps"):\n        if not np.isfinite(cfg[name]) or cfg[name] < 0 or (name != "weight_decay" and cfg[name] == 0):\n            raise ValueError("Invalid " + name)\n    if len(cfg["optimizer_betas"]) != 2 or not all(0 <= b < 1 for b in cfg["optimizer_betas"]):\n        raise ValueError("Invalid optimizer betas")\n    if cfg["weighting"] not in {"patient", "snapshot"} or cfg["balance"] not in {"none", "sqrt", "full"}:\n        raise ValueError("Unknown weighting or balance")\n    return cfg\n\n\ndef seed_run(seed):\n    import torch\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n\n\ndef make_encoder(n_features, n_steps, config, static_dim=0):\n    import torch\n    from torch import nn\n    cfg = _config(config)\n    d = cfg["d_model"]\n    class Encoder(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.projection = nn.Linear(n_features, d)\n            self.input_dropout = nn.Dropout(cfg["dropout"])\n            layer = nn.TransformerEncoderLayer(d, cfg["heads"], cfg["ff"], cfg["dropout"],\n                                               activation="gelu", batch_first=True, norm_first=True)\n            self.encoder = nn.TransformerEncoder(layer, cfg["depth"], enable_nested_tensor=False)\n            for block in self.encoder.layers:\n                for p in block.parameters():\n                    if p.ndim > 1:\n                        nn.init.xavier_uniform_(p)\n            self.norm = nn.LayerNorm(d)\n            self.pool_score = nn.Linear(d, 1) if cfg["pooling"] == "attention" else None\n            self.head = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(d, 1))\n            positions = torch.arange(n_steps, dtype=torch.float32)[:, None]\n            freq = torch.exp(torch.arange(0, d, 2) * (-np.log(10000.) / d))\n            pe = torch.zeros(n_steps, d)\n            pe[:, 0::2] = torch.sin(positions * freq)\n            pe[:, 1::2] = torch.cos(positions * freq[:pe[:, 1::2].shape[1]])\n            self.register_buffer("position", pe)\n            self.static_dim = int(static_dim)\n            self.static_branch = None\n            if static_dim:\n                \n                \n                self.static_branch = nn.Sequential(nn.Linear(static_dim, d), nn.GELU(),\n                                                   nn.Dropout(cfg["dropout"]))\n                nn.init.zeros_(self.static_branch[0].weight)\n                nn.init.zeros_(self.static_branch[0].bias)\n\n        def forward(self, x, valid, static=None):\n            if valid.dtype != torch.bool or not bool(valid.any(1).all()):\n                raise ValueError("Every input needs at least one explicitly available timestep")\n            x = x.masked_fill(~valid[:, :, None], 0.)\n            z = self.input_dropout(self.projection(x)) + self.position[:x.shape[1]]\n            z = self.norm(self.encoder(z, src_key_padding_mask=~valid))\n            if self.pool_score is None:\n                pooled = z.masked_fill(~valid[:, :, None], 0.).sum(1) / valid.sum(1, keepdim=True)\n            else:\n                weights = self.pool_score(z).squeeze(-1).masked_fill(~valid, -torch.inf).softmax(1)\n                pooled = (z * weights[:, :, None]).sum(1)\n            if self.static_branch is not None:\n                if static is None or static.shape != (len(x), self.static_dim):\n                    raise ValueError("Aligned static input required by the fitted fusion architecture")\n                pooled = pooled + self.static_branch(static)\n            elif static is not None:\n                raise ValueError("Static inputs supplied to sequence-only architecture")\n            return self.head(pooled).squeeze(-1)\n    return Encoder()\n\n\ndef quick_metrics(y, scores, primary_lift_fraction=.10):\n    y, scores = np.asarray(y), np.asarray(scores)\n    if (y.shape != scores.shape or set(np.unique(y)) != {0, 1}\n            or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any()):\n        raise ValueError("Aligned binary labels and finite probability scores required")\n    return dict(primary_lift=lift_at_fraction(y, scores, primary_lift_fraction),\n                primary_lift_fraction=primary_lift_fraction,\n                average_precision=float(average_precision_score(y, scores)),\n                roc_auc=float(roc_auc_score(y, scores)),\n                log_loss=float(log_loss(y, np.clip(scores, 1e-7, 1-1e-7))),\n                brier_score=float(np.mean((y - scores) ** 2)))\n\n\ndef _static_array(value, n):\n    if value is None:\n        return None\n    a = np.asarray(value)\n    if a.ndim == 3 and a.shape[1] == 1:\n        a = a[:, 0, :]\n    if a.ndim != 2 or len(a) != n or a.shape[1] < 1 or not np.isfinite(a).all():\n        raise ValueError("Static features must be finite aligned N x F or N x 1 x F")\n    return np.asarray(a, dtype=np.float32)\n\n\ndef _predict(model, x, valid, device="cpu", batch_size=512, static=None):\n    import torch\n    model.eval()\n    output = np.empty(len(x), dtype=np.float32)\n    with torch.inference_mode():\n        for start in range(0, len(x), batch_size):\n            xx = torch.from_numpy(np.ascontiguousarray(x[start:start + batch_size], dtype=np.float32)).to(device)\n            mask = torch.from_numpy(valid[start:start + batch_size]).to(device)\n            static_batch = (None if static is None else\n                            torch.from_numpy(np.ascontiguousarray(static[start:start + batch_size])).to(device))\n            output[start:start+len(xx)] = model(xx, mask, static_batch).sigmoid().cpu().numpy()\n    return output\n\n\ndef train_encoder(X_train, y_train, groups_train, X_val, y_val, valid_train=None,\n                  valid_val=None, config=None, seed=42, groups_val=None,\n                  static_train=None, static_val=None):\n    """Fresh TRAIN-only fit; validation selects stopping, never gradients."""\n    import torch\n    from torch import nn\n    from torch.utils.data import TensorDataset, DataLoader\n    a, ma = _array(X_train, valid_train, finite=True)\n    b, mb = _array(X_val, valid_val, finite=True)\n    y, vy, groups = np.asarray(y_train), np.asarray(y_val), np.asarray(groups_train)\n    if (a.shape[1:] != b.shape[1:] or y.shape != (len(a),) or vy.shape != (len(b),)\n            or groups.shape != y.shape or any(set(np.unique(t)) != {0, 1} for t in (y, vy))):\n        raise ValueError("Training/validation shapes or labels invalid")\n    if groups_val is not None and (np.shape(groups_val) != vy.shape or set(groups) & set(groups_val)):\n        raise ValueError("Patient overlap or malformed validation IDs")\n    if type(seed) is not int or seed < 0:\n        raise ValueError("Seed must be a nonnegative integer")\n    cfg = _config(config)\n    sa, sb = _static_array(static_train, len(a)), _static_array(static_val, len(b))\n    if (sa is None) != (sb is None) or (sa is not None and sa.shape[1] != sb.shape[1]):\n        raise ValueError("Both training and validation need the same static feature dimension")\n    static_dim = 0 if sa is None else sa.shape[1]\n    started = time.perf_counter()\n    seed_run(seed); torch.set_num_threads(cfg["threads"])\n    a, b = np.asarray(a, dtype=np.float32), np.asarray(b, dtype=np.float32)\n    model = make_encoder(a.shape[-1], a.shape[1], cfg, static_dim).to(cfg["device"])\n    weights = sample_weights(groups, cfg["weighting"])\n    positive_weight = class_weight(y, weights, cfg["balance"])\n    opt = torch.optim.AdamW([\n        {"params": [p for p in model.parameters() if p.ndim > 1], "weight_decay": cfg["weight_decay"]},\n        {"params": [p for p in model.parameters() if p.ndim <= 1], "weight_decay": 0.}],\n        lr=cfg["learning_rate"], betas=tuple(cfg["optimizer_betas"]), eps=cfg["optimizer_eps"])\n    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=.5, patience=2)\n                 if cfg["scheduler"] == "plateau" else None)\n    loss_fn = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(positive_weight, device=cfg["device"]))\n    static_dataset = np.zeros((len(a), 0), dtype=np.float32) if sa is None else sa\n    loader = DataLoader(TensorDataset(torch.from_numpy(np.ascontiguousarray(a)), torch.from_numpy(ma),\n                                     torch.tensor(y, dtype=torch.float32), torch.from_numpy(weights),\n                                     torch.from_numpy(np.ascontiguousarray(static_dataset))),\n                        batch_size=cfg["batch_size"], shuffle=True, num_workers=0,\n                        generator=torch.Generator().manual_seed(seed))\n    best_lift, best_epoch, stale, best_state, history = -np.inf, None, 0, None, []\n    for epoch in range(1, cfg["epochs"] + 1):\n        model.train(); total = 0.\n        for xx, mask, yy, ww, ss in loader:\n            xx, mask, yy, ww, ss = [t.to(cfg["device"]) for t in (xx, mask, yy, ww, ss)]\n            opt.zero_grad(set_to_none=True)\n            loss = (loss_fn(model(xx, mask, ss if static_dim else None), yy) * ww).mean()\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite training loss")\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"], error_if_nonfinite=True)\n            opt.step(); total += float(loss.detach()) * len(xx)\n        train_scores = _predict(model, a, ma, cfg["device"], static=sa)\n        val_scores = _predict(model, b, mb, cfg["device"], static=sb)\n        tm, vm = quick_metrics(y, train_scores, cfg["primary_lift_fraction"]), quick_metrics(vy, val_scores, cfg["primary_lift_fraction"])\n        history.append(dict(epoch=epoch, optimization_loss=total/len(a), train_ap=tm["average_precision"],\n                            train_lift=tm["primary_lift"], validation_lift=vm["primary_lift"],\n                            validation_ap=vm["average_precision"], train_log_loss=tm["log_loss"],\n                            validation_log_loss=vm["log_loss"], learning_rate=opt.param_groups[0]["lr"]))\n        if vm["primary_lift"] > best_lift:\n            best_lift, best_epoch, stale = vm["primary_lift"], epoch, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        if scheduler is not None:\n            scheduler.step(vm["primary_lift"])\n        if stale >= cfg["patience"]:\n            break\n    model.load_state_dict(best_state, strict=True)\n    train_scores = _predict(model, a, ma, cfg["device"], static=sa)\n    val_scores = _predict(model, b, mb, cfg["device"], static=sb)\n    stream = io.BytesIO(); torch.save(best_state, stream); blob = stream.getvalue()\n    summary = dict(seed=seed, config=cfg, config_sha256=stable_hash(cfg), input_shape=list(a.shape[1:]),\n                   runtime_versions={"python": platform.python_version(), **{package: importlib.metadata.version(package)\n                       for package in ("numpy", "pandas", "scikit-learn", "torch")}},\n                   model_sha256=hashlib.sha256(blob).hexdigest(), artifact_format="torch_weights_only_state_dict",\n                   train_metrics=quick_metrics(y, train_scores, cfg["primary_lift_fraction"]),\n                   validation_metrics=quick_metrics(vy, val_scores, cfg["primary_lift_fraction"]),\n                   generalization_gap_lift=lift_at_fraction(y, train_scores, cfg["primary_lift_fraction"])-lift_at_fraction(vy, val_scores, cfg["primary_lift_fraction"]),\n                   generalization_gap_ap=float(average_precision_score(y, train_scores)-average_precision_score(vy, val_scores)),\n                   parameter_count=sum(p.numel() for p in model.parameters()), best_epoch=best_epoch,\n                   epochs_observed=len(history), positive_weight=positive_weight,\n                   patient_disjointness_verified=groups_val is not None, wall_seconds=time.perf_counter()-started,\n                   selection_metric="validation_lift_at_primary_capacity", primary_lift_fraction=cfg["primary_lift_fraction"],\n                   checkpoint_tie_rule="earliest epoch with maximum validation lift; AP is not a tiebreaker",\n                   lift_population="patient_snapshots; exact ceil capacity; frozen row-order score ties", test_used=False,\n                   static_dim=static_dim,\n                   architecture="pooled_sequence_plus_GELU_static_branch" if static_dim else "sequence_encoder",\n                   projection_column_norm=model.projection.weight.detach().cpu().norm(dim=0).tolist(),\n                   projection_norm_note="parameter magnitude, not predictive feature importance")\n    return dict(summary=summary, model_blob=blob, config=cfg, history=history,\n                train_scores=train_scores, val_scores=val_scores, validation_scores=val_scores)\n\n\ndef predict_encoder(run, X, valid=None, device="cpu", static=None):\n    import torch\n    s = run["summary"]\n    if hashlib.sha256(run["model_blob"]).hexdigest() != s["model_sha256"] or stable_hash(s["config"]) != s["config_sha256"]:\n        raise ValueError("Saved model or configuration hash mismatch")\n    x, mask = _array(X, valid, finite=True)\n    if list(x.shape[1:]) != s["input_shape"]:\n        raise ValueError("Prediction shape differs from trained input")\n    static_values = _static_array(static, len(x))\n    expected_static = s.get("static_dim", 0)\n    if (0 if static_values is None else static_values.shape[1]) != expected_static:\n        raise ValueError("Static feature dimension differs from fitted model")\n    model = make_encoder(x.shape[-1], x.shape[1], s["config"], expected_static).to(device)\n    model.load_state_dict(torch.load(io.BytesIO(run["model_blob"]), map_location=device, weights_only=True), strict=True)\n    return _predict(model, x, mask, device, static=static_values)\n\n\ndef distribution_diagnostics(train_values, validation_values, feature_names,\n                             valid_train=None, valid_validation=None):\n    """TRAIN/VALIDATION descriptive checks only; never used to fit transforms."""\n    a, ma = _array(train_values, valid_train, finite=True, allow_empty=True)\n    b, mb = _array(validation_values, valid_validation, finite=True, allow_empty=True)\n    if a.shape[-1] != b.shape[-1] or len(feature_names) != a.shape[-1]:\n        raise ValueError("Diagnostic feature names/width mismatch")\n    n_a, n_b = int(ma.sum()), int(mb.sum())\n    result = []\n    for j, name in enumerate(feature_names):\n        aa = np.asarray(a[:, :, j][ma], dtype=np.float64)\n        bb = np.asarray(b[:, :, j][mb], dtype=np.float64)\n        am, bm = (float(aa.mean()) if n_a else None), (float(bb.mean()) if n_b else None)\n        ad, bd = (float(aa.std()) if n_a else None), (float(bb.std()) if n_b else None)\n        result.append(dict(feature=name, train_mean=am, validation_mean=bm,\n                           train_observed_rows=n_a, validation_observed_rows=n_b,\n                           train_sd=ad, validation_sd=bd,\n                           standardized_mean_difference=(float((bm-am)/max(ad, 1e-6)) if n_a and n_b else None),\n                           status="available_observations_only" if n_a and n_b else "one_split_has_no_observed_auxiliary_values"))\n    return result\n', 'workflow': '"""Run-scoped orchestration for summary and historical AutoML experiments.\n\nPrivate inputs and row-level predictions stay in the configured artifact store.\nSelection sees TRAIN/VALIDATION only. TEST prediction requires an explicit lock.\nThe notebook injects input/model/evaluation modules and immutable store functions.\n"""\nimport hashlib\nimport io\nimport json\nimport re\nimport time\nimport tempfile\nimport shutil\nimport zipfile\nimport weakref\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\ninputs = modeling = comparison = None\nsave_artifacts = read_artifacts = table_exists = None\n\n\ndef plain(value):\n    if isinstance(value, dict):\n        return {str(k): plain(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [plain(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return plain(value.tolist())\n    if isinstance(value, np.generic):\n        return plain(value.item())\n    if isinstance(value, float) and not np.isfinite(value):\n        return None\n    return value\n\n\ndef json_bytes(value):\n    return json.dumps(plain(value), sort_keys=True, separators=(",", ":"),\n                      allow_nan=False).encode()\n\n\ndef digest(value):\n    return hashlib.sha256(json_bytes(value)).hexdigest()\n\n\ndef array_bytes(value):\n    out = io.BytesIO()\n    np.save(out, np.asarray(value), allow_pickle=False)\n    return out.getvalue()\n\n\ndef array_sha256(value):\n    """Hash the .npy representation without allocating a second full count tensor."""\n    hasher = hashlib.sha256()\n    class Sink:\n        def write(self, block):\n            hasher.update(block)\n            return len(block)\n    np.save(Sink(), np.asarray(value), allow_pickle=False)\n    return hasher.hexdigest()\n\n\ndef bytes_array(value):\n    return np.load(io.BytesIO(value), allow_pickle=False)\n\n\ndef npz_bytes(arrays):\n    out = io.BytesIO()\n    np.savez_compressed(out, **arrays)\n    return out.getvalue()\n\n\ndef prefix(run_id):\n    if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,23}", run_id):\n        raise ValueError("RUN_ID must use uppercase letters, digits and underscores")\n    return "TAK861_AUTOML_SEQUENCE_V1_" + run_id\n\n\ndef assign_patient_split(metadata, seed=42):\n    """Match the documented sorted-patient 70/15/15 stratification algorithm.\n\n    Reproducibility is not an assertion that historical assignments match. If a\n    SPLIT column is supplied, retain and verify it rather than resplitting.\n    """\n    out = metadata.copy()\n    needs_split = "SPLIT" not in out\n    if needs_split:\n        out["SPLIT"] = "train"  \n    out = inputs.normalize_snapshots(out)\n    if needs_split:\n        from sklearn.model_selection import train_test_split\n        patients = out.groupby("PATIENT_ID", sort=True)["RESP"].max()\n        a, rest = train_test_split(patients.index.to_numpy(), test_size=.30,\n                                   random_state=seed, stratify=patients.to_numpy())\n        b, c = train_test_split(rest, test_size=.5, random_state=seed,\n                               stratify=patients.loc[rest].to_numpy())\n        mapping = {p: s for s, values in (("train", a), ("validation", b), ("test", c)) for p in values}\n        out["SPLIT"] = out.PATIENT_ID.map(mapping)\n    if set(out.SPLIT) != {"train", "validation", "test"} or out.SPLIT.isna().any():\n        raise ValueError("Require complete train/validation/test membership")\n    if out.groupby("PATIENT_ID").SPLIT.nunique().max() != 1:\n        raise ValueError("A patient appears in multiple splits")\n    for split, frame in out.groupby("SPLIT"):\n        if set(frame.RESP.unique()) != {0, 1}:\n            raise ValueError("Both classes required in " + split)\n    return out\n\n\nINPUT_NAMES = {"arrays.npz", "metadata.csv", "input_manifest.json"}\n\n\ndef save_inputs(run_prefix, data, panel_manifest, config, implementation_hash):\n    metadata = assign_patient_split(data["metadata"])\n    inputs.verify_fitting_scope(panel_manifest["contract"], metadata)\n    _validate_input_arrays(data, metadata, data["feature_names"])\n    metadata["END_DT"] = pd.to_datetime(metadata.END_DT).dt.strftime("%Y-%m-%d")\n    arrays = {key: np.asarray(data[key]) for key in\n              ("X_summary", "X_sequence", "summary_mask", "sequence_mask", "y")}\n    if "X_count" in data:\n        arrays["X_count"] = data["X_count"]\n        arrays["count_mask"] = data["count_mask"]\n    with tempfile.TemporaryDirectory(prefix="tak861_inputs_write_") as directory:\n        arrays_blob = Path(directory) / "arrays.npz"\n        np.savez_compressed(arrays_blob, **arrays)\n        metadata_blob = metadata.to_csv(index=False).encode()\n        manifest = dict(version=1, implementation_hash=implementation_hash,\n                        configuration=plain(config), configuration_hash=digest(config),\n                        panel_manifest=panel_manifest, feature_names=data["feature_names"],\n                        feature_specs=data.get("feature_specs", panel_manifest["feature_specs"]),\n                        input_report=data.get("report", {}),\n                        array_sha256=file_sha256(arrays_blob),\n                        metadata_sha256=hashlib.sha256(metadata_blob).hexdigest(),\n                        count_contract=data.get("count_contract"),\n                        split_counts={s: int((metadata.SPLIT == s).sum()) for s in ("train", "validation", "test")},\n                        prediction_unit="patient_snapshot", private_row_level_artifacts=True)\n        if table_exists(run_prefix + "_INPUTS"):\n            previous_data = load_inputs(run_prefix, implementation_hash)\n            previous = previous_data["manifest"]\n            release_inputs(previous_data)\n            \n            def semantic(value):\n                value = json.loads(json_bytes(value))\n                value["panel_manifest"].pop("extraction_report", None)\n                return value\n            if semantic(previous) != semantic(manifest):\n                raise ValueError("Frozen input source/content/configuration changed; use a new RUN_ID")\n            return previous\n        save_artifacts(run_prefix + "_INPUTS", {"arrays.npz": arrays_blob,\n                       "metadata.csv": metadata_blob, "input_manifest.json": json_bytes(manifest)})\n        return manifest\n\n\ndef file_sha256(path):\n    hasher = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            hasher.update(block)\n    return hasher.hexdigest()\n\n\nclass _InputDirectory:\n    """Close mappings before scratch cleanup, including notebook restarts/GC."""\n    def __init__(self):\n        self.name = tempfile.mkdtemp(prefix=\'tak861_inputs_read_\')\n        self.mappings = []\n        self.finalizer = weakref.finalize(self, self._remove, self.name, self.mappings)\n\n    @staticmethod\n    def _remove(path, mappings):\n        for mapping in mappings:\n            mapping.close()\n        mappings.clear()\n        shutil.rmtree(path)\n\n    def cleanup(self):\n        self.finalizer()\n\n\ndef release_inputs(data):\n    """Release file-backed raw arrays after a stage; never delete persisted inputs."""\n    temporary = data.pop(\'_input_directory\', None)\n    if temporary is not None:\n        temporary.cleanup()\n    else:\n        for key in (\'X_summary\', \'X_sequence\'):\n            inputs.release_array(data.get(key))\n        modeling.release_array(data.get(\'X_count\'))\n\n\ndef load_inputs(run_prefix, implementation_hash):\n    temporary = _InputDirectory()\n    data = {}\n    try:\n        artifacts = read_artifacts(run_prefix + "_INPUTS", INPUT_NAMES,\n                                  disk_names={\'arrays.npz\'}, output_dir=temporary.name)\n        manifest = json.loads(artifacts["input_manifest.json"])\n        if manifest["implementation_hash"] != implementation_hash:\n            raise ValueError("Inputs belong to a different notebook implementation; use a new RUN_ID")\n        if (file_sha256(artifacts[\'arrays.npz\']) != manifest[\'array_sha256\'] or\n                hashlib.sha256(artifacts[\'metadata.csv\']).hexdigest() != manifest[\'metadata_sha256\']):\n            raise ValueError(\'Input content changed\')\n        allowed = {\'X_summary\',\'X_sequence\',\'X_count\',\'summary_mask\',\'sequence_mask\',\'count_mask\',\'y\'}\n        with zipfile.ZipFile(artifacts[\'arrays.npz\']) as archive:\n            for name in archive.namelist():\n                key = name[:-4] if name.endswith(\'.npy\') else \'\'\n                if key not in allowed or key in data:\n                    raise ValueError(\'Unexpected or duplicate input array\')\n                target = Path(temporary.name) / name\n                with archive.open(name) as source, target.open(\'wb\') as dest:\n                    shutil.copyfileobj(source, dest, length=1024*1024)\n                data[key] = np.load(target, mmap_mode=\'r\', allow_pickle=False)\n                temporary.mappings.append(data[key]._mmap)\n        metadata = pd.read_csv(io.BytesIO(artifacts["metadata.csv"]), dtype={"PATIENT_ID": str}, keep_default_na=False)\n        metadata = assign_patient_split(metadata)\n        inputs.verify_fitting_scope(manifest["panel_manifest"]["contract"], metadata)\n        data.update(metadata=metadata, manifest=manifest, feature_names=manifest["feature_names"],\n                    feature_specs=manifest["feature_specs"], _input_directory=temporary,\n                    indices={s: np.flatnonzero(metadata.SPLIT.to_numpy() == s) for s in ("train", "validation", "test")})\n        _validate_input_arrays(data, metadata, manifest["feature_names"])\n        return data\n    except BaseException:\n        data[\'_input_directory\'] = temporary\n        release_inputs(data)\n        raise\n\n\ndef _validate_input_arrays(data, metadata, feature_names):\n    n, f = len(metadata), len(feature_names)\n    if f == 0 or len(set(feature_names)) != f:\n        raise ValueError("Invalid AutoML feature order")\n    for value, mask, steps in (("X_summary", "summary_mask", 1), ("X_sequence", "sequence_mask", 12)):\n        x, valid = np.asarray(data[value]), np.asarray(data[mask])\n        if x.shape != (n, steps, f) or valid.shape != (n, steps) or valid.dtype != bool:\n            raise ValueError("Invalid AutoML array/mask shape: " + value)\n        for start in range(0,n,256):\n            block, mask = x[start:start+256], valid[start:start+256]\n            if np.isinf(block).any() or not np.isnan(block[~mask]).all():\n                raise ValueError("Invalid feature values or missing-history padding")\n    for start in range(0,n,256):\n        if not np.array_equal(data["X_summary"][start:start+256], data["X_sequence"][start:start+256, :1], equal_nan=True):\n            raise ValueError("Summary inputs differ from historical current snapshot")\n    if not np.array_equal(data["summary_mask"], data["sequence_mask"][:, :1]):\n        raise ValueError("Invalid current snapshot observation mask")\n    if np.asarray(data["y"]).shape != (n,) or not np.array_equal(data["y"], metadata.RESP.to_numpy()):\n        raise ValueError("Saved labels do not align with metadata")\n    if "X_count" in data:\n        if np.asarray(data["X_count"]).shape != (n, 12, 1028) or np.asarray(data["count_mask"]).shape != (n, 12):\n            raise ValueError("Count reference shape mismatch")\n        if np.asarray(data["count_mask"]).dtype != bool:\n            raise ValueError("Invalid count mask or nonfinite count values")\n        for start in range(0,n,256):\n            if not np.isfinite(data["X_count"][start:start+256]).all():\n                raise ValueError("Invalid count mask or nonfinite count values")\n\n\ndef attach_count_reference(data, values, metadata, contract, valid=None):\n    """Optional verified reuse, aligned by exact keys and independently hashed.\n\n    A provenance declaration limits claims; it is not evidence of an unknown\n    historical extraction rule. Never rename a different universe to 1,028.\n    """\n    if not isinstance(contract, dict) or not all(contract.get(k) for k in\n            ("source_id", "source_version", "content_sha256", "feature_names", "provenance_status",\n             "orientation", "date_policy", "mask_kind", "mask_sha256", "asof_grid_sha256")):\n        raise ValueError("Count reuse needs source/version/content, calendar, explicit mask and provenance fingerprints")\n    values = np.asanyarray(values)\n    names = contract["feature_names"]\n    if values.ndim != 3 or values.shape != (len(metadata), 12, 1028) or len(names) != 1028 or len(set(names)) != 1028:\n        raise ValueError("Count reference must preserve the complete 1,028 candidate universe")\n    if contract.get("n_steps") != 12 or contract["orientation"] != "0=newest" or contract["date_policy"] not in inputs.DATE_POLICIES:\n        raise ValueError("Count cache calendar orientation/length must be explicit")\n    valid = np.asarray(valid)\n    if valid.shape != values.shape[:2] or valid.dtype != bool:\n        raise ValueError("Count reuse requires an explicit boolean position mask")\n    if array_sha256(valid) != contract["mask_sha256"]:\n        raise ValueError("Count position mask hash mismatch")\n    if array_sha256(values) != contract["content_sha256"]:\n        raise ValueError("Count cache content hash mismatch")\n    for block in np.array_split(values, max(1, len(values) // 512)):\n        if not np.isfinite(block).all() or (block < 0).any():\n            raise ValueError("Invalid count values")\n        finite = block[np.isfinite(block)]\n        if (finite != np.floor(finite)).any():\n            raise ValueError("Fractional count values")\n        if np.any(finite.astype(np.float32).astype(np.float64) != finite):\n            raise ValueError("Count loses exact float32 precision")\n    old = inputs.normalize_snapshots(metadata)\n    target = inputs.normalize_snapshots(data["metadata"])\n    if contract.get("metadata_sha256") != inputs.frame_hash(old):\n        raise ValueError("Count cache snapshot metadata/order fingerprint mismatch")\n    _, old_grid = inputs.make_requests(old, date_policy=contract["date_policy"])\n    if inputs.frame_hash(old_grid) != contract["asof_grid_sha256"]:\n        raise ValueError("Count cache date grid mismatch")\n    target_policy = data.get("report", {}).get("date_policy", data.get("manifest", {}).get("input_report", {}).get("date_policy"))\n    if target_policy != contract["date_policy"]:\n        raise ValueError("Count and AutoML historical date policies differ; do not concatenate misaligned dates")\n    old["_row"] = np.arange(len(old))\n    joined = target.merge(old[["PATIENT_ID", "END_DT", "RESP", "_row"]],\n                          on=["PATIENT_ID", "END_DT"], how="left", validate="one_to_one", suffixes=("", "_cached"))\n    if joined._row.isna().any() or not joined.RESP.eq(joined.RESP_cached).all() or len(old) != len(target):\n        raise ValueError("Count cache cohort/label mismatch")\n    order = joined._row.to_numpy(dtype=int)\n    if values.dtype == np.float32 and np.array_equal(order,np.arange(len(values))):\n        data["X_count"] = values  \n    else:\n        data["X_count"] = modeling.take_rows(values,order,dtype=np.float32)\n    data["count_mask"] = valid[order].copy()\n    data["count_contract"] = dict(contract, coverage_note="Position mask semantics are explicit; a saved row does not establish clinical observation coverage")\n    return data\n\n\nPREP_NAMES = {"preparation.json", "protocol.json"}\n\n\ndef prepare_run(data, run_prefix, config, implementation_hash):\n    if data["manifest"]["panel_manifest"]["contract"]["fit_scope"] != "deterministic_raw":\n        raise ValueError("This fold-local selection protocol requires raw deterministic AutoML inputs. "\n                         "A transform or vocabulary selected using all TRAIN labels is not fold-local. "\n                         "Recover raw extraction without RESP-based fitting, or implement upstream reconstruction inside each fold.")\n    table = run_prefix + "_PREPARATION"\n    expected = dict(input_manifest_hash=digest(data["manifest"]), implementation_hash=implementation_hash,\n                    config_hash=digest(config))\n    if table_exists(table):\n        saved = read_artifacts(table, PREP_NAMES)\n        prep, protocol = json.loads(saved["preparation.json"]), json.loads(saved["protocol.json"])\n        if protocol["identity"] != expected:\n            raise ValueError("Preparation identity changed; use a new RUN_ID")\n        return prep, protocol\n    ti = data["indices"]["train"]\n    groups = data["metadata"].PATIENT_ID.to_numpy()[ti]\n    primary_fraction = float(config.get("primary_lift_fraction", .10))\n    if not np.isfinite(primary_fraction) or not 0 < primary_fraction <= 1:\n        raise ValueError("primary_lift_fraction must be in (0,1]")\n    selection_config, model_config = dict(config.get("selection", {})), dict(config.get("model", {}))\n    for settings in (selection_config, model_config):\n        if "primary_lift_fraction" in settings and settings["primary_lift_fraction"] != primary_fraction:\n            raise ValueError("Conflicting primary capacity; set primary_lift_fraction once in the top-level configuration")\n        settings["primary_lift_fraction"] = primary_fraction\n    declared_kinds = [s["kind"] for s in data["manifest"]["feature_specs"]]\n    if config.get("feature_kinds", declared_kinds) != declared_kinds:\n        raise ValueError("Model preprocessing kinds differ from the recovered source feature contract")\n    prep = {"automl": modeling.prepare_features(\n        data["X_summary"], data["X_sequence"], data["y"][ti], groups,\n        data["feature_names"], valid_train=data["sequence_mask"], row_indices=ti,\n        feature_kinds=declared_kinds, config=selection_config)}\n    arms = {"SUMMARY": dict(family="transformer", input="X_summary", mask="summary_mask", preparation="automl"),\n            "SEQUENCE": dict(family="transformer", input="X_sequence", mask="sequence_mask", preparation="automl")}\n    if "X_count" in data:\n        count = data["X_count"]\n        prep["count"] = modeling.prepare_features(count[:, :1], count, data["y"][ti], groups,\n            data["manifest"]["count_contract"]["feature_names"], valid_train=data["count_mask"], row_indices=ti,\n            feature_kinds=["count"] * count.shape[-1], config=selection_config)\n        arms["COUNT"] = dict(family="transformer", input="X_count", mask="count_mask", preparation="count")\n        arms["STATIC_FUSION"] = dict(arms["COUNT"], static_input="X_summary", static_mask="summary_mask",\n                                    static_preparation="automl", experiment="count_sequence_plus_snapshot_automl")\n        arms["TEMPORAL_FUSION"] = dict(arms["COUNT"], temporal_input="X_sequence", temporal_mask="sequence_mask",\n                                      temporal_preparation="automl", experiment="count_plus_historical_automl_per_timestep")\n    if config.get("require_count_reference", True) and "COUNT" not in arms:\n        raise ValueError("Both requested incremental-value experiments require the matched 1,028-count reference")\n    if not config.get("include_automl_only_controls", False):\n        arms = {name: arm for name, arm in arms.items() if name not in ("SUMMARY", "SEQUENCE")}\n    if config.get("include_lightgbm", False):\n        raise ValueError("LightGBM refit is deferred in this bounded two-experiment protocol")\n    seeds = config.get("seeds", [42, 142, 242])\n    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):\n        raise ValueError("Seeds must be a prespecified unique nonempty list")\n    protocol = dict(identity=expected, arms=arms, seeds=seeds, preparation_hash=digest(prep),\n                    model_config=model_config, primary_metric="lift_at_primary_capacity",\n                    primary_lift_fraction=primary_fraction,\n                    primary_capacity_provenance="configured before fit; 10% default is an operating assumption, not a recovered prior agreement",\n                    lift_population="patient_snapshots; not unique-patient interventions",\n                    lift_tie_policy="score descending then frozen snapshot row order; exact ceil(fraction*N)",\n                    selection_split="train_grouped_cv_and_validation", test_used_for_selection=False,\n                    top_fractions=sorted({.1, .2, .3, primary_fraction}),\n                    test_evaluation_ordinal=None if \'client_universe\' in config else 3,\n                    legacy_test_evaluations=2,\n                    test_scope=("Original frozen cohort; previously inspected TEST patients, not an untouched holdout"\n                                if config.get(\'client_universe\',{}).get(\'table_name\')==\'TAK861_TX_READY_V63_MODEL_DATA\' else\n                                "Revised universe includes previously inspected TEST patients and possibly new patients; not an untouched holdout"\n                                if \'client_universe\' in config else "retrospective; this cohort already evaluated twice"),\n                    partial_seed_rule="all completed runs shown; incomplete protocol cannot be locked",\n                    comparison_scope="matched count control versus static and historical AutoML augmentation; descriptive development evidence")\n    save_artifacts(table, {"preparation.json": json_bytes(prep), "protocol.json": json_bytes(protocol)})\n    return prep, protocol\n\n\nRUN_NAMES = {"model.bin", "run.json", "train.npy", "validation.npy"}\n\n\ndef pack_run(run, identity):\n    summary = {k: v for k, v in run.items() if k not in ("model_blob", "train_scores", "val_scores", "validation_scores", "model")}\n    blobs = {"model.bin": run["model_blob"], "train.npy": array_bytes(run["train_scores"]),\n             "validation.npy": array_bytes(run["val_scores"])}\n    summary.update(identity=identity, payload_hashes={k: hashlib.sha256(v).hexdigest() for k, v in blobs.items()})\n    blobs["run.json"] = json_bytes(summary)\n    return blobs\n\n\ndef unpack_run(artifacts, identity):\n    summary = json.loads(artifacts["run.json"])\n    if summary["identity"] != identity:\n        raise ValueError("Cached fitted model identity changed")\n    for name, expected in summary["payload_hashes"].items():\n        if hashlib.sha256(artifacts[name]).hexdigest() != expected:\n            raise ValueError("Fitted model artifact hash mismatch")\n    return dict(summary, model_blob=artifacts["model.bin"], train_scores=bytes_array(artifacts["train.npy"]),\n                val_scores=bytes_array(artifacts["validation.npy"]))\n\n\ndef arm_arrays(data, prep, arm, split, subset=None):\n    idx = data[\'indices\'][split]\n    if subset is not None:\n        idx = idx[subset]\n    valid = data[arm[\'mask\']][idx]\n    x = static = transformed = temporal = merged = None\n    try:\n        x = modeling.transform_inputs(data[arm[\'input\']], prep[arm[\'preparation\']],\n                                      valid=data[arm[\'mask\']], row_indices=idx)\n        if \'static_input\' in arm:\n            mask = data[arm[\'static_mask\']][idx]\n            transformed = modeling.transform_inputs(data[arm[\'static_input\']], prep[arm[\'static_preparation\']],\n                                                    valid=data[arm[\'static_mask\']], row_indices=idx)\n            static = modeling.allocate_array((len(idx), transformed.shape[-1]+1), np.float32)\n            for start in range(0,len(idx),256):\n                static[start:start+256,:-1] = transformed[start:start+256,0]\n                static[start:start+256,-1:] = mask[start:start+256]\n            modeling.release_array(transformed)\n            transformed = None\n        if \'temporal_input\' in arm:\n            mask = data[arm[\'temporal_mask\']][idx]\n            temporal = modeling.transform_inputs(data[arm[\'temporal_input\']], prep[arm[\'temporal_preparation\']],\n                                                 valid=data[arm[\'temporal_mask\']], row_indices=idx)\n            if temporal.shape[:2] != x.shape[:2]:\n                raise ValueError(\'Count and AutoML monthly shapes disagree\')\n            merged = modeling.allocate_array((*x.shape[:2], x.shape[-1]+temporal.shape[-1]+1),np.float32)\n            k=x.shape[-1]\n            for start in range(0,len(idx),256):\n                merged[start:start+256,:,:k] = x[start:start+256]\n                merged[start:start+256,:,k:-1] = temporal[start:start+256]\n                merged[start:start+256,:,-1] = mask[start:start+256]\n            modeling.release_array(x);modeling.release_array(temporal)\n            x,merged,temporal = merged,None,None\n        return x,valid,static\n    except BaseException:\n        for value in (x,static,transformed,temporal,merged):\n            modeling.release_array(value)\n        raise\n\n\ndef train_registered(data, prep, protocol, run_prefix):\n    if digest(prep) != protocol["preparation_hash"] or digest(data["manifest"]) != protocol["identity"]["input_manifest_hash"]:\n        raise ValueError("Training inputs or preparation changed")\n    groups = data["metadata"].PATIENT_ID.to_numpy()\n    ti, vi = data["indices"]["train"], data["indices"]["validation"]\n    results = {}\n    for name, arm in protocol["arms"].items():\n        train = validation = static_train = static_val = None\n        try:\n            train, train_mask, static_train = arm_arrays(data, prep, arm, "train")\n            validation, val_mask, static_val = arm_arrays(data, prep, arm, "validation")\n            results[name] = []\n            for seed in protocol["seeds"]:\n                identity = dict(protocol_hash=digest(protocol), arm=name, seed=seed)\n                table = run_prefix + "_" + name + "_S" + str(seed)\n                if not table_exists(table):\n                    if arm["family"] != "transformer":\n                        raise ValueError("Only Transformer arms are registered; LightGBM remains deferred")\n                    trainer = modeling.train_encoder\n                    run = trainer(train, data["y"][ti], groups[ti], validation, data["y"][vi],\n                                  valid_train=train_mask, valid_val=val_mask,\n                                  static_train=static_train, static_val=static_val,\n                                  groups_val=groups[vi],\n                                  config=protocol["model_config"], seed=seed)\n                    save_artifacts(table, pack_run(run, identity))\n                result = unpack_run(read_artifacts(table, RUN_NAMES), identity)\n                if len(result["train_scores"]) != len(ti) or len(result["val_scores"]) != len(vi):\n                    raise ValueError("Prediction length does not match frozen split")\n                results[name].append(result)\n                print(name, "seed", seed, "completed; validation lift",\n                      modeling.lift_at_fraction(data["y"][vi], result["val_scores"], protocol["primary_lift_fraction"]),\n                      "secondary AP", comparison.binary_metrics(data["y"][vi], result["val_scores"])["average_precision"])\n        finally:\n            for array in (train,validation,static_train,static_val):\n                modeling.release_array(array)\n    return results\n\n\ndef read_runs(protocol, run_prefix):\n    return {name: [unpack_run(read_artifacts(run_prefix + "_" + name + "_S" + str(seed), RUN_NAMES),\n                  dict(protocol_hash=digest(protocol), arm=name, seed=seed)) for seed in protocol["seeds"]]\n            for name in protocol["arms"]}\n\n\ndef make_lock(data, prep, protocol, results, run_prefix):\n    if set(results) != set(protocol["arms"]) or digest(prep) != protocol["preparation_hash"]:\n        raise ValueError("Incomplete registered experiments")\n    vi, ti = data["indices"]["validation"], data["indices"]["train"]\n    models, metrics = {}, {}\n    for name, runs in results.items():\n        if len(runs) != len(protocol["seeds"]) or [r["identity"]["seed"] for r in runs] != protocol["seeds"]:\n            raise ValueError("All prespecified seeds must complete before locking")\n        val = np.mean([r["val_scores"] for r in runs], axis=0)\n        train = np.mean([r["train_scores"] for r in runs], axis=0)\n        policy = comparison.select_validation_threshold(data["y"][vi], val)\n        models[name] = dict(predictor_hashes=[r["payload_hashes"] for r in runs], threshold=policy)\n        metrics[name] = dict(train=comparison.binary_metrics(data["y"][ti], train, policy,\n                                                             top_fractions=protocol["top_fractions"]),\n                             validation=comparison.binary_metrics(data["y"][vi], val, policy,\n                                                                  top_fractions=protocol["top_fractions"]))\n        for split_name, yy, ss in (("train", data["y"][ti], train), ("validation", data["y"][vi], val)):\n            metrics[name][split_name]["primary_lift"] = modeling.lift_at_fraction(yy, ss, protocol["primary_lift_fraction"])\n            metrics[name][split_name]["primary_lift_fraction"] = protocol["primary_lift_fraction"]\n    lock = dict(protocol_hash=digest(protocol), preparation_hash=digest(prep),\n                input_manifest_hash=digest(data["manifest"]), models=models, metrics=metrics,\n                test_prediction_performed=False, test_use_status_note="False at immutable lock creation; completed TEST use is recorded in FINAL_TEST report",\n                test_scope=protocol["test_scope"],\n                primary_metric=protocol["primary_metric"], primary_lift_fraction=protocol["primary_lift_fraction"],\n                threshold_note="Validation F1 threshold is secondary; primary lift uses the registered capacity, not this threshold")\n    save_artifacts(run_prefix + "_LOCK", {"lock.json": json_bytes(lock)})\n    return lock\n\n\ndef paired_lift_bootstrap(y, scores_a, scores_b, patient_ids, fractions=(.1, .2, .3),\n                          n_bootstrap=1000, seed=20261013, confidence=.95):\n    """Paired patient resampling for snapshot lift, with exact repeated-row capacity.\n\n    Patient multiplicities repeat every snapshot of a sampled patient. Ranking\n    ties use frozen original row order in every draw. Integer multiplicities\n    avoid materializing duplicate rows; a boundary row may be selected partly.\n    """\n    y, a, b, groups = [np.asarray(v) for v in (y, scores_a, scores_b, patient_ids)]\n    fractions = sorted(set(float(f) for f in fractions))\n    if (y.ndim != 1 or any(v.shape != y.shape for v in (a, b, groups))\n            or set(np.unique(y)) != {0, 1} or not np.isfinite(a).all() or not np.isfinite(b).all()\n            or not fractions or any(not np.isfinite(f) or not 0 < f <= 1 for f in fractions)\n            or type(n_bootstrap) is not int or n_bootstrap < 20 or not 0 < confidence < 1):\n        raise ValueError("Paired lift bootstrap requires aligned two-class data, valid fractions and at least 20 draws")\n    if any(v is None or str(v) in ("", "nan") for v in groups):\n        raise ValueError("Patient IDs must be complete for clustered lift uncertainty")\n    unique, inverse = np.unique(groups.astype(str), return_inverse=True)\n    orders = [np.lexsort((np.arange(len(y)), -scores.astype(float))) for scores in (a, b)]\n    def lifts(order, multiplicities):\n        counts = multiplicities[order]\n        before = np.cumsum(counts) - counts\n        n = int(multiplicities.sum())\n        prevalence = float(np.dot(y, multiplicities) / n)\n        values = []\n        for fraction in fractions:\n            k = max(1, int(np.ceil(fraction*n)))\n            taken = np.minimum(counts, np.maximum(k-before, 0))\n            values.append(float(np.dot(y[order], taken) / k / prevalence))\n        return np.asarray(values)\n    points = lifts(orders[0], np.ones(len(y), dtype=int)) - lifts(orders[1], np.ones(len(y), dtype=int))\n    rng, draws, invalid = np.random.default_rng(seed), [], 0\n    for _ in range(n_bootstrap):\n        sampled = rng.integers(0, len(unique), size=len(unique))\n        multiplicities = np.bincount(sampled, minlength=len(unique))[inverse]\n        positives, total = int(np.dot(y, multiplicities)), int(multiplicities.sum())\n        if positives == 0 or positives == total:\n            invalid += 1\n            continue\n        draws.append(lifts(orders[0], multiplicities)-lifts(orders[1], multiplicities))\n    reliable = len(draws) >= max(20, int(np.ceil(.8*n_bootstrap)))\n    values = np.asarray(draws)\n    tail = (1-confidence)/2\n    by_fraction = {}\n    for j, fraction in enumerate(fractions):\n        lower, upper = (np.quantile(values[:, j], [tail, 1-tail]).tolist() if reliable else (None, None))\n        by_fraction[str(fraction)] = dict(fraction=fraction, delta=float(points[j]), lower=lower, upper=upper,\n                                           excludes_zero=bool(reliable and (lower > 0 or upper < 0)))\n    return dict(lift_by_fraction=by_fraction, unit="patient_cluster", population="patient_snapshots",\n                patients=len(unique), requested_draws=n_bootstrap, valid_draws=len(draws), single_class_draws=invalid,\n                seed=seed, confidence=confidence, direction="model_a_minus_model_b", paired=True,\n                interval_reliable=reliable, tie_rule="frozen original row order; repeated rows counted with patient multiplicity",\n                capacity_rule="exact ceil(fraction * resampled_snapshot_count)")\n\n\ndef compare_run(data, prep, protocol, results, lock, run_prefix, *, evaluate_test=False, n_bootstrap=1000):\n    if lock["protocol_hash"] != digest(protocol) or lock["input_manifest_hash"] != digest(data["manifest"]) or lock["preparation_hash"] != digest(prep):\n        raise ValueError("Evaluation lock mismatch")\n    saved = json.loads(read_artifacts(run_prefix + "_LOCK", {"lock.json"})["lock.json"])\n    if saved != plain(lock):\n        raise ValueError("Persisted lock mismatch")\n    split = "test" if evaluate_test else "validation"\n    table = run_prefix + ("_FINAL_TEST" if evaluate_test else "_VALIDATION_COMPARISON")\n    names = {"report.json"} | {n + ".npy" for n in protocol["arms"]}\n    if table_exists(table):\n        report = json.loads(read_artifacts(table, names)["report.json"])\n        if report["lock_hash"] != digest(lock) or report["bootstrap_resamples"] != n_bootstrap:\n            raise ValueError("Existing comparison differs from lock or uncertainty protocol")\n        return report\n    idx = data["indices"][split]\n    scores, metrics = {}, {}\n    for name, arm in protocol["arms"].items():\n        runs = results[name]\n        if [r["payload_hashes"] for r in runs] != lock["models"][name]["predictor_hashes"]:\n            raise ValueError("Final fitted model changed")\n        if evaluate_test:\n            x, valid, static = arm_arrays(data, prep, arm, "test")\n            try:\n                if arm["family"] != "transformer":\n                    raise ValueError("Only Transformer arms are registered; LightGBM remains deferred")\n                predict = modeling.predict_encoder\n                scores[name] = np.mean([predict(r, x, valid=valid, static=static) for r in runs], axis=0)\n            finally:\n                modeling.release_array(x)\n                modeling.release_array(static)\n        else:\n            scores[name] = np.mean([r["val_scores"] for r in runs], axis=0)\n        metrics[name] = comparison.binary_metrics(data["y"][idx], scores[name], lock["models"][name]["threshold"],\n                                                   top_fractions=protocol["top_fractions"])\n        metrics[name]["primary_lift"] = modeling.lift_at_fraction(data["y"][idx], scores[name], protocol["primary_lift_fraction"])\n        metrics[name]["primary_lift_fraction"] = protocol["primary_lift_fraction"]\n    pairs = [("SEQUENCE", "SUMMARY")] if "SEQUENCE" in scores and "SUMMARY" in scores else []\n    if "COUNT" in scores:\n        pairs += [(name, "COUNT") for name in ("SUMMARY", "SEQUENCE", "STATIC_FUSION", "TEMPORAL_FUSION") if name in scores]\n    if "STATIC_FUSION" in scores and "TEMPORAL_FUSION" in scores:\n        pairs += [("TEMPORAL_FUSION", "STATIC_FUSION")]\n    for name in ("SUMMARY", "SEQUENCE"):\n        if "GB_" + name in scores:\n            pairs.append((name, "GB_" + name))\n    comparisons = {}\n    for a, b in pairs:\n        uncertainty = comparison.paired_patient_bootstrap(data["y"][idx], scores[a], scores[b],\n            data["metadata"].PATIENT_ID.to_numpy()[idx], n_bootstrap=n_bootstrap, seed=20261013)\n        lift_uncertainty = paired_lift_bootstrap(data["y"][idx], scores[a], scores[b],\n            data["metadata"].PATIENT_ID.to_numpy()[idx], fractions=protocol["top_fractions"],\n            n_bootstrap=n_bootstrap, seed=20261013)\n        comparisons[a + "_MINUS_" + b] = dict(model_a=a, model_b=b, **uncertainty,\n            primary_lift=lift_uncertainty["lift_by_fraction"][str(protocol["primary_lift_fraction"])],\n            lift_by_fraction=lift_uncertainty["lift_by_fraction"], lift_bootstrap=lift_uncertainty)\n    report = dict(lock_hash=digest(lock), split=split, metrics=metrics, comparisons=comparisons,\n                  test_prediction_performed=bool(evaluate_test),\n                  bootstrap_resamples=n_bootstrap, execution_status="completed_on_configured_inputs",\n                  primary_metric=protocol["primary_metric"], primary_lift_fraction=protocol["primary_lift_fraction"],\n                  lift_population=protocol["lift_population"], lift_tie_policy=protocol["lift_tie_policy"],\n                  source_id=data["manifest"]["panel_manifest"]["contract"]["source_id"],\n                  test_evaluation_ordinal=protocol["test_evaluation_ordinal"] if evaluate_test else None,\n                  limitations=["Validation is development evidence, not independent confirmation" if not evaluate_test else protocol["test_scope"],\n                    "Bootstrap conditions on fitted predictions; excludes training/search uncertainty",\n                    "Validation intervals condition on validation-selected checkpoints and are optimistic development evidence",\n                    "Cohort and source-vintage comparability must be verified before comparing absolute lift with prior runs",\n                    "Unadjusted intervals across fixed contrasts; no multiplicity-controlled superiority claim",\n                    "Augmentation changes input width/capacity as reported; no matched zero-input capacity ablation",\n                    "Primary capacity was fixed before fitting; other capacity points are secondary, not selected from TEST",\n                    "Lift counts patient snapshots; repeated monthly opportunities are not unique-patient intervention counts",\n                    "Input cutoff and source availability claims limited to recorded construction evidence",\n                    "Augmentation comparisons are conditional on the registered training/selection procedure; no causal claim"],\n                  count_reference_status="matched refit provided" if "COUNT" in scores else "not available; added-value-versus-1028 conclusion pending",\n                  lightgbm_status="included" if "GB_SUMMARY" in scores else "deferred")\n    save_artifacts(table, dict({"report.json": json_bytes(report)}, **{n + ".npy": array_bytes(s) for n, s in scores.items()}))\n    return report\n\n\ndef report_table(report):\n    return pd.DataFrame([dict(arm=name, split=report["split"],\n        **{"lift_" + str(int(round(100*row["fraction"]))) + "pct": row["lift"] for row in values["topk"]},\n        **{key: values.get(key) for key in\n        ("primary_lift", "primary_lift_fraction", "snapshots", "positives", "average_precision", "roc_auc", "log_loss", "brier_score", "precision", "recall", "f1", "specificity")})\n        for name, values in report["metrics"].items()])\n', 'reliance': '"""Post-lock VALIDATION permutation reliance, never feature/model selection.\n\nGrouped perturbations describe fitted predictive dependence, not clinical\nimportance or causality. Correlation can hide or redistribute reliance. This is\nnot projection-weight magnitude and does not impose agreement with proxy rank.\n"""\nimport numpy as np\nimport pandas as pd\n\n\nPERMUTATION_SEEDS = (271, 811, 1297)\n\n\ndef _groups(arm, prep):\n    counts = len(prep["count"]["selected_names"]) if "count" in prep else 0\n    automl = len(prep["automl"]["selected_names"])\n    families = []\n    if arm in {"COUNT", "STATIC_FUSION", "TEMPORAL_FUSION"}:\n        families.append(("count", "sequence", np.arange(counts)))\n    if arm == "STATIC_FUSION":\n        families.append(("automl", "static", np.arange(automl)))\n    elif arm == "TEMPORAL_FUSION":\n        families.append(("automl", "sequence", np.arange(counts, counts + automl)))\n    elif arm in {"SUMMARY", "SEQUENCE"}:\n        families.append(("automl", "sequence", np.arange(automl)))\n    return [(family, destination, name, columns) for family, destination, columns in families\n            for name, columns in zip(("top", "middle", "bottom"), np.array_split(columns, 3))]\n\n\ndef run(data, prep, protocol, results, workflow, modeling, comparison):\n    """Evaluate all registered seeds on one common validation subset.\n\n    Run only after the upstream predictor lock. Caller persists this aggregate\n    result with that lock. No TEST row or label is inspected here.\n    """\n    vi = np.asarray(data["indices"]["validation"])\n    if vi.ndim != 1 or not len(vi):\n        raise ValueError("Nonempty validation indices required")\n    meta = data["metadata"].iloc[vi].reset_index(drop=True)\n    if not meta.SPLIT.str.lower().eq("validation").all():\n        raise ValueError("Reliance diagnostics are validation-only")\n    latest = comparison.latest_patient_indices(meta.PATIENT_ID.to_numpy(), meta.END_DT.to_numpy())\n    \n    complete = np.ones(len(vi), bool)\n    for key in ("sequence_mask", "count_mask", "summary_mask"):\n        if key in data:\n            mask = np.asarray(data[key])[vi]\n            if mask.dtype != bool or mask.ndim != 2:\n                raise ValueError("Availability masks must be boolean N x T")\n            complete &= mask.all(axis=1)\n    selected = latest[complete[latest]]\n    report = dict(schema_version=1, split="validation", prediction_unit="latest_snapshot_per_patient",\n                  patient_subset="latest validation snapshot with all count/AutoML position-availability flags true",\n                  validation_snapshots=len(vi), latest_patients=len(latest), patients=len(selected),\n                  excluded_incomplete_latest=int(len(latest)-len(selected)),\n                  permutation_seeds=list(PERMUTATION_SEEDS), rows=[], status="pending",\n                  model_selection_performed=False, test_accessed=False,\n                  limitations=["Post-lock descriptive diagnostic; do not use for feature or model selection in this run",\n                               "Complete recorded-position subset may differ from the operational population",\n                               "Position availability is not proof of enrollment or complete clinical observation",\n                               "Within-date grouped shuffling can disrupt inter-feature dependencies",\n                               "A small or negative lift/AP drop does not establish feature irrelevance",\n                               "TRAIN proxy ranking is not expected to equal fitted Transformer reliance"])\n    if not len(selected):\n        return dict(report, status="skipped", reason="no_complete_latest_validation_snapshots")\n    labels = np.asarray(data["y"])[vi][selected]\n    if set(np.unique(labels)) != {0, 1}:\n        return dict(report, status="skipped", reason="diagnostic_subset_requires_both_classes")\n    dates = pd.to_datetime(meta.END_DT.iloc[selected]).to_numpy(dtype="datetime64[D]")\n    strata = [np.flatnonzero(dates == date) for date in np.unique(dates)]\n    shuffleable = [rows for rows in strata if len(rows) >= 2]\n    report.update(date_strata=len(strata), shuffleable_strata=len(shuffleable),\n                  shuffleable_patients=sum(len(rows) for rows in shuffleable))\n    if not shuffleable:\n        return dict(report, status="skipped", reason="no_date_stratum_has_two_patients")\n    for arm, config in protocol["arms"].items():\n        if config["family"] != "transformer":\n            raise ValueError("This registered reliance diagnostic expects Transformer arms")\n        runs = results[arm]\n        if len(runs) != len(protocol["seeds"]) or [r["summary"]["seed"] for r in runs] != protocol["seeds"]:\n            raise ValueError("All registered seeds required for reliance ensemble")\n        x, valid, static = workflow.arm_arrays(data, prep, config, "validation", subset=selected)\n        try:\n            def ensemble(values, static_values):\n                return np.mean([modeling.predict_encoder(fit, values, valid=valid, static=static_values)\n                                for fit in runs], axis=0)\n            primary_fraction = float(protocol.get("primary_lift_fraction", .10))\n            base_scores = ensemble(x, static)\n            baseline = comparison.binary_metrics(labels, base_scores)["average_precision"]\n            baseline_lift = modeling.lift_at_fraction(labels, base_scores, primary_fraction)\n            for family, destination, rank_group, columns in _groups(arm, prep):\n                if len(columns) == 0:\n                    report["rows"].append(dict(arm=arm, feature_family=family, rank_group=rank_group,\n                                               status="skipped_empty_rank_group", n_features=0, patients=len(selected)))\n                    continue\n                for repeat, seed in enumerate(PERMUTATION_SEEDS, 1):\n                    rng = np.random.default_rng(seed)\n                    order = np.arange(len(selected))\n                    for rows in shuffleable:\n                        order[rows] = rng.permutation(rows)\n                    changed = int(np.sum(order != np.arange(len(selected))))\n                    perturbed_x, perturbed_static = x, static\n                    try:\n                        if destination == "static":\n                            if static is None:\n                                raise ValueError("Static reliance group requires a static input branch")\n                            perturbed_static = modeling.take_rows(static, np.arange(len(static)))\n                            for start in range(0, len(static), 256):\n                                perturbed_static[start:start+256, columns] = static[order[start:start+256]][:, columns]\n                        else:\n                            perturbed_x = modeling.take_rows(x, np.arange(len(x)))\n                            for start in range(0, len(x), 256):\n                                perturbed_x[start:start+256, :, columns] = x[order[start:start+256]][:, :, columns]\n                        permuted_scores = ensemble(perturbed_x, perturbed_static)\n                    finally:\n                        if perturbed_x is not x:\n                            modeling.release_array(perturbed_x)\n                        if perturbed_static is not static:\n                            modeling.release_array(perturbed_static)\n                    ap = comparison.binary_metrics(labels, permuted_scores)["average_precision"]\n                    lift = modeling.lift_at_fraction(labels, permuted_scores, primary_fraction)\n                    report["rows"].append(dict(arm=arm, feature_family=family, rank_group=rank_group,\n                                               repeat=repeat, permutation_seed=seed, status="completed",\n                                               n_features=len(columns), patients=len(selected),\n                                               shuffleable_strata=len(shuffleable), shuffled_patients=changed,\n                                               primary_lift_fraction=primary_fraction,\n                                               baseline_lift=float(baseline_lift), permuted_lift=float(lift),\n                                               lift_drop=float(baseline_lift-lift),\n                                               baseline_ap=float(baseline), permuted_ap=float(ap),\n                                               ap_drop=float(baseline-ap), direction="baseline_minus_permuted",\n                                               interpretation="positive drop indicates predictive reliance under this perturbation"))\n        finally:\n            modeling.release_array(x)\n            modeling.release_array(static)\n    report["status"] = "completed"\n    return report\n', 'source_recovery': '"""Read-only retrieval of explicitly supplied procedure signatures.\n\nReturned DDL is evidence for private review, never executed or automatically\napproved as a feature extractor. No source text is printed by this module.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom collections.abc import Mapping\n\nimport pandas as pd\n\n\nDEFAULT_PROCEDURES = {\n    "FEATURE_CREATION_PROC": (\n        "DSVC_TAKEDA_TA_PRIVATE.DS_ML_DEV.FEATURE_CREATION_PROC("\n        "VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR)"),\n    "FEATURE_ENGINEERING_PROD": (\n        "DSVC_TAKEDA_TA_PRIVATE.DS_ML_DEV.FEATURE_ENGINEERING_PROD("\n        "VARCHAR,VARCHAR,VARCHAR)"),\n    "FEATURE_CREATION_PROD": (\n        "DSVC_TAKEDA_TA_PRIVATE.DS_ML_DEV.FEATURE_CREATION_PROD("\n        "VARCHAR,VARCHAR,VARCHAR,VARCHAR)"),\n}\n\n_SIGNATURE = re.compile(\n    r"([A-Z_][A-Z0-9_$]*)\\.([A-Z_][A-Z0-9_$]*)\\."\n    r"([A-Z_][A-Z0-9_$]*)\\((VARCHAR(?:,VARCHAR)*)\\)", re.IGNORECASE)\n_DDL_HEADER = re.compile(\n    r\'\\bCREATE\\s+(?:OR\\s+REPLACE\\s+)?(?:SECURE\\s+)?PROCEDURE\\s+\'\n    r\'((?:"?[A-Z_][A-Z0-9_$]*"?\\.){0,2}"?[A-Z_][A-Z0-9_$]*"?)\\s*\\(\',\n    re.IGNORECASE)\n_OMISSION = re.compile(\n    r"\\[\\s*(?:\\.{3}|\\u2026)[^\\]]{0,2000}\\]"\n    r"|\\[(?:[^\\]\\r\\n]{0,100}\\b(?:omitted|truncated|redacted)\\b[^\\]\\r\\n]{0,100})\\]",\n    re.IGNORECASE)\n\n\ndef _sha256(value):\n    return hashlib.sha256(value).hexdigest()\n\n\ndef _requests(procedures, expected_sha256):\n    if not isinstance(procedures, Mapping) or not procedures:\n        raise ValueError("Procedure signatures must be a nonempty explicit mapping")\n    result = []\n    signatures = set()\n    for name, signature in procedures.items():\n        if not isinstance(name, str) or not re.fullmatch(r"[A-Z_][A-Z0-9_$]*", name):\n            raise ValueError("Procedure evidence names must be uppercase SQL identifiers")\n        if not isinstance(signature, str) or not (match := _SIGNATURE.fullmatch(signature)):\n            raise ValueError("Use an exact fully qualified supplied VARCHAR procedure signature")\n        canonical = signature.upper()\n        if name != match.group(3).upper() or canonical in signatures:\n            raise ValueError("Procedure name/signature mismatch or duplicate request")\n        signatures.add(canonical)\n        result.append((name, canonical))\n    expected = {} if expected_sha256 is None else expected_sha256\n    if not isinstance(expected, Mapping) or set(expected) - set(procedures):\n        raise ValueError("Expected source hashes must name requested procedures only")\n    for value in expected.values():\n        if not isinstance(value, str) or not re.fullmatch(r"[0-9a-fA-F]{64}", value):\n            raise ValueError("Expected source hashes must be SHA-256 hex strings")\n    return result, {name: value.lower() for name, value in expected.items()}\n\n\ndef _ddl_value(response, name, signature):\n    \n    \n    if hasattr(response, "toPandas"):\n        response = response.toPandas()\n    if not isinstance(response, pd.DataFrame) or response.shape != (1, 1):\n        raise ValueError("GET_DDL must return exactly one row and one column: " + name)\n    text = response.iloc[0, 0]\n    if not isinstance(text, str) or not text.strip():\n        raise ValueError("GET_DDL returned empty or non-text source: " + name)\n    header = _DDL_HEADER.search(text)\n    if header is None:\n        raise ValueError("GET_DDL response has no recognizable procedure declaration: " + name)\n    returned_parts = header.group(1).replace(\'"\', \'\').upper().split(\'.\')\n    requested_parts = signature.split(\'(\', 1)[0].split(\'.\')\n    if returned_parts != requested_parts[-len(returned_parts):]:\n        raise ValueError("GET_DDL declaration does not match requested procedure: " + name)\n    return text\n\n\ndef recover_procedure_sources(read_query, procedures=None, expected_sha256=None):\n    """Return ``{\'ddl_texts\': {name: text}, \'receipt\': {...}}`` without printing.\n\n    ``read_query(sql)`` must use an existing authorized connection and return a\n    pandas DataFrame or a Spark-like DataFrame exposing ``toPandas``. Only SELECT\n    GET_DDL statements are issued; no procedure CALL, writes or source execution\n    occur. Custom signatures must come from supplied evidence or read-only\n    metadata, never guesses. Optional expected hashes detect changed retrieved\n    source. Partial retrieval or a mismatch raises instead of returning a bundle\n    that could be mistaken for complete recovery.\n\n    Absence of an omission marker does not prove completeness, correct historical\n    deployment, fitting provenance, cutoff semantics or safe execution. Keep DDL\n    private; the aggregate receipt is suitable for display.\n    """\n    if not callable(read_query):\n        raise TypeError("read_query must be a callable authorized query adapter")\n    requests, expected = _requests(\n        DEFAULT_PROCEDURES if procedures is None else procedures, expected_sha256)\n    ddl_texts, rows = {}, []\n    for name, signature in requests:\n        \n        query = "SELECT GET_DDL(\'PROCEDURE\', \'" + signature + "\') AS PROCEDURE_DDL"\n        try:\n            response = read_query(query)\n        except Exception:\n            \n            \n            raise RuntimeError("Read-only GET_DDL retrieval failed for " + name) from None\n        text = _ddl_value(response, name, signature)\n        encoded = text.encode("utf-8")\n        digest = _sha256(encoded)\n        if name in expected and expected[name] != digest:\n            raise ValueError("Recovered source SHA-256 changed for " + name)\n        markers = list(_OMISSION.finditer(text))\n        ddl_texts[name] = text\n        rows.append({\n            "name": name, "requested_signature": signature,\n            "sha256": digest, "utf8_bytes": len(encoded),\n            "lines": len(text.splitlines()),\n            "query_sha256": _sha256(query.encode("utf-8")),\n            "expected_sha256_verified": name in expected,\n            "explicit_omission_marker_detected": bool(markers),\n            "omission_marker_count": len(markers),\n            "omission_marker_lines": [text.count("\\n", 0, m.start()) + 1 for m in markers],\n            "completeness": "not_certified",\n        })\n    \n    \n    identity = [{"requested_signature": r["requested_signature"], "sha256": r["sha256"]}\n                for r in sorted(rows, key=lambda r: r["requested_signature"])]\n    receipt = {\n        "schema_version": 1,\n        "retrieval_method": "supplied_read_query_SELECT_GET_DDL",\n        "classification": "Recovered from existing artifact",\n        "scope": "Returned deployed DDL evidence; not historical execution verification",\n        "read_only": True, "procedure_count": len(rows), "procedures": rows,\n        "aggregate_sha256": _sha256(json.dumps(identity, sort_keys=True,\n                                                 separators=(",", ":")).encode("utf-8")),\n        "explicit_omission_marker_detected": any(r["explicit_omission_marker_detected"] for r in rows),\n        "completeness": "not_certified", "approved_for_feature_binding": False,\n        "historical_execution_verified": False, "source_executed": False,\n        "clinical_results_produced": False,\n        "next_step": "Privately review recovered source and bind an explicit raw historical extractor; do not execute DDL",\n    }\n    return {"ddl_texts": ddl_texts, "receipt": receipt}\n', 'raw_automl': '"""Target-independent raw AutoML histories adapted from supplied procedure SQL.\n\nOnly SELECT queries run. Legacy RESP-conditioned maps, category filtering,\nbinary inference, caps, learned encodings, RND and model importance are omitted.\nSource values stay private; returned receipts contain aggregate evidence only.\nThe notebook injects ``inputs`` so the embedded module has no package dependency.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport time\nimport sqlite3\nimport tempfile\nimport os\nfrom contextlib import closing\n\nimport numpy as np\nimport pandas as pd\n\ninputs = None\n\nDEFAULT_CONFIG = {\n    "source_schema": "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719",\n    "mapping_table": "DSVC_TAKEDA_TA_PRIVATE.DS_ML_PRD.DX_PX_RX_PLAID",\n    "date_policy": "month_end_history", "chunk_size": 5000,\n    "max_features": None,\n}\n\n\ndef _config(config):\n    cfg = dict(DEFAULT_CONFIG)\n    incoming = config or {}\n    if set(incoming) - set(cfg):\n        raise ValueError("Unknown raw AutoML source configuration: " + str(set(incoming)-set(cfg)))\n    cfg.update(incoming)\n    for key, parts in (("source_schema", 2), ("mapping_table", 3)):\n        if not isinstance(cfg[key], str) or not re.fullmatch(\n                r"[A-Za-z_][A-Za-z0-9_$]*(?:\\.[A-Za-z_][A-Za-z0-9_$]*){"+str(parts-1)+"}", cfg[key]):\n            raise ValueError("Invalid explicit source identifier: " + key)\n    if type(cfg["chunk_size"]) is not int or not 1 <= cfg["chunk_size"] <= 10000:\n        raise ValueError("chunk_size must be between 1 and 10000")\n    if cfg["max_features"] is not None and (type(cfg["max_features"]) is not int or cfg["max_features"] < 1):\n        raise ValueError("max_features must be an explicit positive integer or None")\n    if cfg["date_policy"] not in ("same_day_months", "month_end_history"):\n        raise ValueError("Unknown date policy")\n    return cfg\n\n\ndef raw_sql_template(source_config=None):\n    """Return exact read-only long-form SQL with one keys-only request slot.\n\n    Medical encounter grain preserves the source procedure\'s patient/date/DX\n    grouping, with JSON encoding to avoid null/concatenation collisions. Claims\n    counts and provider counts are distinct counts; recent counts use -4 to -1\n    calendar months inclusive. No label appears in SQL.\n    """\n    cfg = _config(source_config)\n    schema, mapping = cfg["source_schema"], cfg["mapping_table"]\n    tokens = []\n\n    def token(relation, category, entity="ENCOUNTER_KEY", condition="TRUE", rx_paid=True):\n        \n        \n        if relation in ("RX_BASE", "RX_MAPPED") and rx_paid:\n            condition = "UPPER(TRANSACTION_RESULT)=\'PAID\' AND (" + condition + ")"\n        tokens.append("SELECT PATIENT_ID,AS_OF_DATE," + category + " AS CATEGORY,"\n                      + entity + " AS ENTITY_KEY,EVENT_DATE,IS_RECENT FROM " + relation\n                      + " WHERE " + condition)\n\n    \n    \n    for prefix, relation, levels in (("DX", "DX_MAPPED", 2), ("PX", "PX_MAPPED", 3)):\n        token(relation, "\'"+prefix+"_DESC_\'||MAP_DESC")\n        for level in range(1, levels+1):\n            token(relation, "\'"+prefix+"_LEVEL_"+str(level)+"_\'||LEVEL_"+str(level)+"_DESC")\n    for family, relation in (("DX", "DX_BASE"), ("PX", "PX_BASE")):\n        stem = "MX" if family == "DX" else "PX"\n        token(relation, "\'"+stem+"_PAYER_MIX_\'||INSURANCE_GROUP")\n        token(relation, "\'"+stem+"_INSURANCE_SEGMENT_\'||INSURANCE_SEGMENT")\n        token(relation, "\'HCPS_"+family+"_SP_PRIMARY_\'||TRIM(SPLIT_PART(PRIMARY_SPECIALTY,\',\',1))")\n        token(relation, "\'CLAIMS_"+family+"\'")\n        token(relation, "\'HCPS_"+family+"\'", "PROVIDER_ID", "PROVIDER_ID IS NOT NULL")\n    token("DX_BASE", "\'CLAIM_TYP_\'||EVENT_SOURCE")\n    token("DX_BASE", "\'HCP_TYP_\'||PROVIDER_TYPE")\n    token("DX_BASE", "\'REFERRING_ENCOUNTER\'", condition="REFERRING_NPI IS NOT NULL")\n    token("DX_BASE", "\'VISIT_TYP_VST_\'||VISIT_TYPE")\n    token("DX_BASE", "\'BILL_TYP_BILL_TYPE_CODE_\'||BILL_TYPE_CODE")\n    token("DX_BASE", "\'PL_\'||PLACE_OF_SERVICE")\n    \n    token("DX_MAPPED", "\'DX_WITH_CHRONIC_INDICATOR_RECORDED\'", condition="CHRONIC_INDICATOR IS NOT NULL")\n    token("PX_BASE", "\'PROC_TYPE_\'||PROCEDURE_CODE_TYPE")\n    for level in range(1, 5):\n        token("RX_MAPPED", "\'RX_LEVEL_"+str(level)+"_\'||LEVEL_"+str(level)+"_DESC", "CLAIM_KEY")\n    token("RX_BASE", "\'RX_PAYER_MIX_\'||INSURANCE_GROUP", "CLAIM_KEY")\n    token("RX_BASE", "\'RX_INSURANCE_SEGMENT_\'||INSURANCE_SEGMENT", "CLAIM_KEY")\n    token("RX_BASE", "\'RX_CHANNEL_\'||PHARMACY_CHANNEL", "CLAIM_KEY",\n          "UPPER(PHARMACY_CHANNEL)<>\'UNKNOWN\'")\n    token("RX_BASE", "\'HCPS_RX_HCP_S1_\'||TRIM(SPLIT_PART(PRIMARY_SPECIALTY,\',\',1))", "CLAIM_KEY")\n    token("RX_BASE", "\'CLAIMS_RX\'", "CLAIM_KEY")\n    token("RX_BASE", "\'CLAIMS_NRX\'", "CLAIM_KEY", "IS_NRX=1")\n    token("RX_BASE", "\'HCPS_RX\'", "PROVIDER_ID", "PROVIDER_ID IS NOT NULL")\n    token("RX_BASE", "\'CLAIMS_RX_\'||TRANSACTION_RESULT", "CLAIM_KEY", rx_paid=False)\n    token("RX_BASE", "\'CLAIMS_RX_ROUTE_\'||ROUTE", "CLAIM_KEY", "UPPER(TRANSACTION_RESULT)=\'PAID\'")\n    token("RX_BASE", "\'CLAIMS_RX_SOURCE_\'||RX_SOURCE", "CLAIM_KEY", "UPPER(TRANSACTION_RESULT)=\'PAID\'")\n    token_sql = "\\nUNION ALL\\n".join(tokens)\n    return f"""WITH REQUESTS AS ({{request_sql}}),\nMAP AS (\n SELECT DISTINCT UPPER(C_TYPE) AS C_TYPE,TRIM(TO_VARCHAR(CODE)) AS CODE,\n TO_VARCHAR(\\"DESC\\") AS MAP_DESC,TO_VARCHAR(LEVEL_1_DESC) AS LEVEL_1_DESC,\n TO_VARCHAR(LEVEL_2_DESC) AS LEVEL_2_DESC,TO_VARCHAR(LEVEL_3_DESC) AS LEVEL_3_DESC,\n TO_VARCHAR(LEVEL_4_DESC) AS LEVEL_4_DESC,CHRONIC_INDICATOR\n FROM {mapping} WHERE UPPER(CURR_IND)=\'Y\'\n),\nMED AS (\n SELECT R.PATIENT_ID,R.AS_OF_DATE,M.SERVICE_DATE::DATE AS EVENT_DATE,\n TO_JSON(ARRAY_CONSTRUCT(TO_VARCHAR(M.PATIENT_ID),TO_VARCHAR(M.SERVICE_DATE::DATE),M.DIAGNOSIS_CODES)) AS ENCOUNTER_KEY,\n IFF(M.SERVICE_DATE::DATE BETWEEN DATEADD(MONTH,-4,R.AS_OF_DATE) AND DATEADD(MONTH,-1,R.AS_OF_DATE),1,0) AS IS_RECENT,\n TO_VARCHAR(M.DIAGNOSIS_CODES) AS DIAGNOSIS_CODES,TRIM(TO_VARCHAR(M.PROCEDURE_CODE)) AS PROCEDURE_CODE,\n TO_VARCHAR(M.PROCEDURE_CODE_TYPE) AS PROCEDURE_CODE_TYPE,TRIM(TO_VARCHAR(M.NDC11)) AS NDC11,\n TO_VARCHAR(M.EVENT_SOURCE) AS EVENT_SOURCE,TO_VARCHAR(M.PLACE_OF_SERVICE) AS PLACE_OF_SERVICE,\n TO_VARCHAR(M.BILL_TYPE_CODE) AS BILL_TYPE_CODE,TO_VARCHAR(M.VISIT_TYPE) AS VISIT_TYPE,\n TO_VARCHAR(M.REFERRING_NPI) AS REFERRING_NPI,TO_VARCHAR(M.RENDERING_NPI) AS PROVIDER_ID,\n TO_VARCHAR(P.INSURANCE_GROUP) AS INSURANCE_GROUP,TO_VARCHAR(P.INSURANCE_SEGMENT) AS INSURANCE_SEGMENT,\n TO_VARCHAR(H.PRIMARY_SPECIALTY) AS PRIMARY_SPECIALTY,TO_VARCHAR(H.PROVIDER_TYPE) AS PROVIDER_TYPE\n FROM REQUESTS R JOIN {schema}.MEDICAL_EVENTS_LATEST M ON TO_VARCHAR(M.PATIENT_ID)=R.PATIENT_ID\n AND M.SERVICE_DATE::DATE BETWEEN DATEADD(MONTH,-12,R.AS_OF_DATE) AND R.AS_OF_DATE\n LEFT JOIN {schema}.PLANS_LATEST P ON M.KH_PLAN_ID=P.KH_PLAN_ID\n LEFT JOIN {schema}.PROVIDERS_LATEST H ON M.RENDERING_NPI=H.NPI\n),\nDX_BASE AS (SELECT * FROM MED),\nDX_CODES AS (SELECT M.*,TRIM(D.VALUE::VARCHAR) AS DX_CODE FROM DX_BASE M,\n LATERAL SPLIT_TO_TABLE(TRIM(M.DIAGNOSIS_CODES,\'|\'),\'|\') D),\nDX_MAPPED AS (SELECT M.*,P.MAP_DESC,P.LEVEL_1_DESC,P.LEVEL_2_DESC,P.CHRONIC_INDICATOR\n FROM DX_CODES M JOIN MAP P ON P.C_TYPE=\'DX\' AND P.CODE=M.DX_CODE),\nPX_BASE AS (SELECT * FROM MED WHERE NULLIF(PROCEDURE_CODE,\'\') IS NOT NULL),\nPX_MAPPED AS (SELECT M.*,P.MAP_DESC,P.LEVEL_1_DESC,P.LEVEL_2_DESC,P.LEVEL_3_DESC\n FROM PX_BASE M JOIN MAP P ON P.C_TYPE=\'PX\' AND P.CODE=M.PROCEDURE_CODE),\nRX_PHARM AS (\n SELECT R.PATIENT_ID,R.AS_OF_DATE,F.FILL_DATE::DATE AS EVENT_DATE,\n \'RX:\'||TO_VARCHAR(F.PHARMACY_EVENT_ID) AS CLAIM_KEY,\n IFF(F.FILL_DATE::DATE BETWEEN DATEADD(MONTH,-4,R.AS_OF_DATE) AND DATEADD(MONTH,-1,R.AS_OF_DATE),1,0) AS IS_RECENT,\n TRIM(TO_VARCHAR(F.NDC11)) AS NDC11,TO_VARCHAR(F.PRESCRIBER_NPI) AS PROVIDER_ID,\n TO_VARCHAR(F.PHARMACY_CHANNEL) AS PHARMACY_CHANNEL,TO_VARCHAR(F.ROUTE) AS ROUTE,\n TO_VARCHAR(F.TRANSACTION_RESULT) AS TRANSACTION_RESULT,IFF(F.FILL_NUMBER=0,1,0) AS IS_NRX,\'RX\' AS RX_SOURCE,\n TO_VARCHAR(P.INSURANCE_GROUP) AS INSURANCE_GROUP,TO_VARCHAR(P.INSURANCE_SEGMENT) AS INSURANCE_SEGMENT,\n TO_VARCHAR(H.PRIMARY_SPECIALTY) AS PRIMARY_SPECIALTY\n FROM REQUESTS R JOIN {schema}.PHARMACY_EVENTS_LATEST F ON TO_VARCHAR(F.PATIENT_ID)=R.PATIENT_ID\n AND F.FILL_DATE::DATE BETWEEN DATEADD(MONTH,-12,R.AS_OF_DATE) AND R.AS_OF_DATE\n AND UPPER(F.TRANSACTION_STATUS) IN (\'FINAL\',\'STANDALONE\')\n LEFT JOIN {schema}.PLANS_LATEST P ON F.PRIMARY_KH_PLAN_ID=P.KH_PLAN_ID\n LEFT JOIN {schema}.PROVIDERS_LATEST H ON F.PRESCRIBER_NPI=H.NPI\n),\nRX_BASE AS (\n SELECT * FROM RX_PHARM WHERE NULLIF(NDC11,\'\') IS NOT NULL\n UNION ALL\n SELECT PATIENT_ID,AS_OF_DATE,EVENT_DATE,\'MX:\'||ENCOUNTER_KEY AS CLAIM_KEY,IS_RECENT,NDC11,PROVIDER_ID,\n NULL AS PHARMACY_CHANNEL,NULL AS ROUTE,\'PAID\' AS TRANSACTION_RESULT,0 AS IS_NRX,\'MX\' AS RX_SOURCE,\n INSURANCE_GROUP,INSURANCE_SEGMENT,PRIMARY_SPECIALTY FROM MED WHERE NULLIF(NDC11,\'\') IS NOT NULL\n),\nRX_MAPPED AS (SELECT M.*,P.LEVEL_1_DESC,P.LEVEL_2_DESC,P.LEVEL_3_DESC,P.LEVEL_4_DESC\n FROM RX_BASE M JOIN MAP P ON P.C_TYPE=\'RX\' AND P.CODE=M.NDC11),\nTOKENS AS (\n{token_sql}\n),\nCOUNTS AS (\n SELECT PATIENT_ID,AS_OF_DATE,CATEGORY,COUNT(DISTINCT ENTITY_KEY) AS VALUE,\n COUNT(DISTINCT IFF(IS_RECENT=1,ENTITY_KEY,NULL)) AS RECENT_VALUE,MAX(EVENT_DATE) AS MAX_EVENT_DATE\n FROM TOKENS WHERE NULLIF(TRIM(CATEGORY),\'\') IS NOT NULL AND ENTITY_KEY IS NOT NULL\n GROUP BY PATIENT_ID,AS_OF_DATE,CATEGORY\n),\nEVENT_COVERAGE AS (\n SELECT PATIENT_ID,AS_OF_DATE,MAX(EVENT_DATE) AS MAX_EVENT_DATE\n FROM (SELECT PATIENT_ID,AS_OF_DATE,EVENT_DATE FROM MED UNION ALL\n SELECT PATIENT_ID,AS_OF_DATE,EVENT_DATE FROM RX_PHARM)\n GROUP BY PATIENT_ID,AS_OF_DATE\n),\nYOB AS (\n SELECT TO_VARCHAR(PATIENT_ID) AS PATIENT_ID,\n COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PATIENT_YOB)),YEAR(TRY_TO_DATE(TO_VARCHAR(PATIENT_YOB)))) AS BIRTH_YEAR\n FROM {schema}.PATIENT_DEMOGRAPHICS_LATEST\n),\nAGES AS (\n SELECT R.PATIENT_ID,R.AS_OF_DATE,\n IFF(COUNT(DISTINCT Y.BIRTH_YEAR)=1 AND MIN(Y.BIRTH_YEAR) BETWEEN 1850 AND YEAR(R.AS_OF_DATE),\n YEAR(R.AS_OF_DATE)-MIN(Y.BIRTH_YEAR),NULL) AS AGE,\n IFF(COUNT(DISTINCT Y.BIRTH_YEAR)>1,1,0) AS AGE_CONFLICT\n FROM REQUESTS R LEFT JOIN YOB Y ON Y.PATIENT_ID=R.PATIENT_ID GROUP BY R.PATIENT_ID,R.AS_OF_DATE\n)\nSELECT R.PATIENT_ID,R.AS_OF_DATE,\'COVERAGE\' AS ROW_KIND,NULL::VARCHAR AS CATEGORY,\n NULL::NUMBER AS VALUE,NULL::NUMBER AS RECENT_VALUE,C.MAX_EVENT_DATE,A.AGE,A.AGE_CONFLICT\n FROM REQUESTS R LEFT JOIN EVENT_COVERAGE C USING(PATIENT_ID,AS_OF_DATE)\n LEFT JOIN AGES A USING(PATIENT_ID,AS_OF_DATE)\nUNION ALL\nSELECT PATIENT_ID,AS_OF_DATE,\'FEATURE\' AS ROW_KIND,CATEGORY,VALUE,RECENT_VALUE,MAX_EVENT_DATE,\n NULL AS AGE,NULL AS AGE_CONFLICT FROM COUNTS WHERE VALUE>0"""\n\n\ndef _feature_name(category, measure):\n    clean = re.sub(r"[^A-Z0-9]+", "_", category.upper()).strip(\'_\')[:160] or "CATEGORY"\n    suffix = hashlib.sha256(category.encode(\'utf-8\')).hexdigest()[:12]\n    return "AML_" + clean + "__" + suffix + ("_COUNT" if measure == "count" else "_RECENT_PCT")\n\n\ndef _frame(value):\n    result = value if isinstance(value, pd.DataFrame) else value.toPandas()\n    result = result.copy()\n    result.columns = [str(c).upper() for c in result.columns]\n    if result.columns.duplicated().any():\n        raise ValueError("Duplicate raw extraction columns")\n    return result\n\n\ndef _read_long(read_query, requests, template, chunk_size):\n    """Yield bounded aggregate frames; never collect all query results."""\n    for start in range(0, len(requests), chunk_size):\n        batch = requests.iloc[start:start+chunk_size]\n        values = ["(\'"+patient.replace("\'", "\'\'")+"\',\'"+date.strftime(\'%Y-%m-%d\')+"\')"\n                  for patient, date in batch.itertuples(index=False, name=None)]\n        request_sql = "SELECT column1::VARCHAR AS PATIENT_ID,TO_DATE(column2::VARCHAR) AS AS_OF_DATE FROM VALUES " + \',\'.join(values)\n        result = read_query(template.replace(\'{request_sql}\', request_sql))\n        if hasattr(result, \'toLocalIterator\') and not isinstance(result,pd.DataFrame):\n            columns = list(result.columns)\n            \n            \n            \n            integral = {\'ByteType\',\'ShortType\',\'IntegerType\',\'LongType\'}\n            float_columns = [field.name for field in getattr(getattr(result,\'schema\',None),\'fields\',[])\n                             if type(field.dataType).__name__ in integral and field.nullable\n                             and field.name.upper() in {\'VALUE\',\'RECENT_VALUE\',\'AGE\',\'AGE_CONFLICT\'}]\n            def make_frame(rows):\n                frame=pd.DataFrame.from_records(rows, columns=columns)\n                for name in float_columns:\n                    frame[name]=frame[name].astype(np.float64)\n                return _frame(frame)\n            rows = []\n            for row in result.toLocalIterator(prefetchPartitions=False):\n                rows.append(tuple(row))\n                if len(rows) == 4096:\n                    yield make_frame(rows)\n                    rows.clear()\n            if rows:\n                yield make_frame(rows)\n        else:\n            result = _frame(result)\n            for row_start in range(0,len(result),4096):\n                yield result.iloc[row_start:row_start+4096].copy()\n        print("AutoML extraction requests completed:",min(start+chunk_size,len(requests)),"/",len(requests))\n\n\ndef _spool_long(read_query, requests, template, chunk_size, train_patients, db):\n    """Validate once while writing an indexed private disk spool.\n\n    raw_json stores exactly the bytes used by inputs.frame_hash before sorting;\n    SQLite supplies deterministic global key order without a full pandas sort.\n    """\n    db.execute(\'PRAGMA temp_store=FILE\')\n    db.execute(\'PRAGMA cache_size=-16384\')\n    db.execute(\'CREATE TABLE raw(p TEXT,d TEXT,k TEXT,c TEXT,raw_json BLOB,v REAL,r REAL,e TEXT,a REAL,ac REAL, PRIMARY KEY(p,d,k,c)) WITHOUT ROWID\')\n    db.execute(\'CREATE TABLE train(p TEXT PRIMARY KEY) WITHOUT ROWID\')\n    db.executemany(\'INSERT INTO train VALUES (?)\',((p,) for p in sorted(train_patients)))\n    columns = None\n    total = 0\n    needed = {\'PATIENT_ID\',\'AS_OF_DATE\',\'ROW_KIND\',\'CATEGORY\',\'VALUE\',\'RECENT_VALUE\',\'MAX_EVENT_DATE\',\'AGE\',\'AGE_CONFLICT\'}\n    for part in _read_long(read_query,requests,template,chunk_size):\n        if not needed.issubset(part):\n            raise ValueError("Raw extraction response lacks required long-form fields")\n        if columns is None:\n            columns = list(part.columns)\n        elif columns != list(part.columns):\n            raise ValueError("Raw extraction response columns changed between batches")\n        if part.PATIENT_ID.isna().any():\n            raise ValueError("Raw extraction contains null patient keys")\n        part[\'PATIENT_ID\'] = part.PATIENT_ID.map(str)\n        for col in (\'AS_OF_DATE\',\'MAX_EVENT_DATE\'):\n            part[col] = pd.to_datetime(part[col],errors=\'raise\')\n            if part[col].dt.tz is not None or not part[col].dropna().eq(part[col].dropna().dt.normalize()).all():\n                raise ValueError("Expected exact date-grain raw source dates")\n        if not part.ROW_KIND.isin([\'COVERAGE\',\'FEATURE\']).all():\n            raise ValueError("Unknown raw extraction record kind")\n        if part.AS_OF_DATE.isna().any() or (part.MAX_EVENT_DATE.notna() & part.MAX_EVENT_DATE.gt(part.AS_OF_DATE)).any():\n            raise ValueError("Invalid or future raw source date")\n        if (part.MAX_EVENT_DATE.notna() & part.MAX_EVENT_DATE.lt(part.AS_OF_DATE-pd.DateOffset(months=12))).any():\n            raise ValueError("Raw source event evidence predates the 12-month window")\n        records = []\n        for row in part.itertuples(index=False,name=None):\n            item = dict(zip(columns,row))\n            feature = item[\'ROW_KIND\'] == \'FEATURE\'\n            category = item[\'CATEGORY\']\n            event = item[\'MAX_EVENT_DATE\']\n            if feature:\n                if pd.isna(category) or not str(category).strip():\n                    raise ValueError("Raw feature category must be explicit")\n                try:\n                    value,recent = float(item[\'VALUE\']),float(item[\'RECENT_VALUE\'])\n                except (TypeError,ValueError,OverflowError) as exc:\n                    raise ValueError("Raw distinct counts must be finite nonnegative integers") from exc\n                if not np.isfinite([value,recent]).all() or value <= 0 or recent < 0 or value != np.floor(value) or recent != np.floor(recent) or recent > value:\n                    raise ValueError("Raw distinct counts must be finite nonnegative integers with valid annual/recent relationship")\n                if pd.isna(event):\n                    raise ValueError("Feature counts require in-window event evidence for a requested key")\n                age=conflict=None\n            else:\n                value=recent=None\n                age = None if pd.isna(item[\'AGE\']) else float(item[\'AGE\'])\n                conflict = None if pd.isna(item[\'AGE_CONFLICT\']) else float(item[\'AGE_CONFLICT\'])\n                if age is not None and (not np.isfinite(age) or age < 0):\n                    raise ValueError("Invalid raw age")\n            records.append((item[\'PATIENT_ID\'],item[\'AS_OF_DATE\'].strftime(\'%Y-%m-%d\'),item[\'ROW_KIND\'],str(category) if feature else \'\',\n                inputs.canonical_row_bytes(row),value,recent,None if pd.isna(event) else event.strftime(\'%Y-%m-%d\'),age,conflict))\n        try:\n            db.executemany(\'INSERT INTO raw VALUES (?,?,?,?,?,?,?,?,?,?)\',records)\n        except sqlite3.IntegrityError as exc:\n            raise ValueError("Duplicate raw coverage keys or aggregated raw feature rows") from exc\n        total += len(part)\n        db.commit()\n    if columns is None:\n        raise ValueError("Raw extraction response lacks required long-form fields")\n    \n    \n    db.execute(\'CREATE INDEX raw_by_category ON raw(k,c,p,d)\')\n    coverage = pd.read_sql_query("SELECT p AS PATIENT_ID,d AS AS_OF_DATE,e AS MAX_EVENT_DATE,a AS AGE,ac AS AGE_CONFLICT FROM raw WHERE k=\'COVERAGE\' ORDER BY p,d",db)\n    coverage[\'AS_OF_DATE\']=pd.to_datetime(coverage.AS_OF_DATE)\n    coverage[\'MAX_EVENT_DATE\']=pd.to_datetime(coverage.MAX_EVENT_DATE)\n    joined=requests.merge(coverage,on=inputs.KEYS,how=\'outer\',indicator=True,validate=\'one_to_one\')\n    if not joined._merge.eq(\'both\').all():\n        raise ValueError("Raw extraction did not return exact requested coverage keys")\n    coverage=joined.drop(columns=\'_merge\').sort_values(inputs.KEYS).reset_index(drop=True)\n    coverage[\'IS_OBSERVED\']=coverage.MAX_EVENT_DATE.notna()\n    if db.execute("SELECT 1 FROM raw f LEFT JOIN raw c ON f.p=c.p AND f.d=c.d AND c.k=\'COVERAGE\' WHERE f.k=\'FEATURE\' AND (c.p IS NULL OR c.e IS NULL) LIMIT 1").fetchone():\n        raise ValueError("Feature counts require in-window event evidence for a requested key")\n    support=dict(db.execute("SELECT c,COUNT(DISTINCT raw.p) FROM raw JOIN train ON raw.p=train.p WHERE k=\'FEATURE\' GROUP BY c"))\n    h=hashlib.sha256(json.dumps(columns,separators=(\',\',\':\')).encode())\n    for (payload,) in db.execute(\'SELECT raw_json FROM raw ORDER BY p,d,k,c\'):\n        h.update(payload)\n    return coverage,support,h.hexdigest(),total\n\n\ndef _build_raw_automl_inputs(read_query, metadata, source_config, db):\n    """Extract raw long rows once per unique patient/date; return AUTOML_SETUP.\n\n    Vocabulary is label-blind and restricted to TRAIN patients. If max_features\n    is explicitly configured, retain the highest distinct-TRAIN-patient support\n    (ties by deterministic name), recording every excluded candidate. This\n    resource bound is not model feature selection or proof of optimality.\n\n    IS_OBSERVED means an eligible source event exists in the requested window,\n    not a complete observed year. No-event rows remain NaN, never fabricated\n    zero history. AGE alone cannot establish source-window observation.\n    """\n    if inputs is None:\n        raise RuntimeError("Inject the shared inputs module before raw extraction")\n    if not callable(read_query):\n        raise TypeError("read_query must use the authorized existing source connection")\n    cfg = _config(source_config)\n    meta = inputs.normalize_snapshots(metadata)\n    requests, _ = inputs.make_requests(meta, date_policy=cfg[\'date_policy\'])\n    template = raw_sql_template(cfg)\n    started = time.perf_counter()\n    train_patients = set(meta.loc[meta.SPLIT.eq(\'train\'),\'PATIENT_ID\'])\n    coverage,support,long_hash,long_rows = _spool_long(read_query, requests, template,\n        cfg[\'chunk_size\'], train_patients, db)\n    candidates = []\n    for category in sorted(support):\n        for measure in (\'count\',\'recent_pct\'):\n            name = _feature_name(category, measure)\n            candidates.append(dict(name=name, kind=\'count\' if measure==\'count\' else \'continuous\',\n                definition=("Distinct raw claim/encounter/provider identifiers in inclusive preceding 12 calendar months" if measure==\'count\'\n                            else "100*distinct recent/distinct 12-month count, rounded to integer; recent=-4 through -1 months inclusive; emitted only annual>6 and recent>2, otherwise 0"),\n                raw_category=category, measure=measure, train_patients=int(support[category])))\n    candidates.sort(key=lambda item: (-item[\'train_patients\'], item[\'name\']))\n    \n    limit = cfg[\'max_features\']\n    categories = sorted(support, key=lambda category: (-support[category], category))\n    keep_categories = set(categories if limit is None else categories[:max(0,(limit-1)//2)])\n    selected = [item for item in candidates if item[\'raw_category\'] in keep_categories]\n    age = dict(name=\'AML_AGE_YEARS\', kind=\'continuous\', definition=\'Calendar as-of year minus birth year; conflicting/missing birth year remains missing\',\n               raw_category=\'AGE\', measure=\'age\', train_patients=int(len(train_patients)))\n    specs = sorted(selected+[age], key=lambda item:item[\'name\'])\n    names = [s[\'name\'] for s in specs]\n    if len(set(names)) != len(names):\n        raise ValueError("Raw AutoML sanitizer collision")\n    values = inputs.disk_array((len(coverage),len(specs)),np.float64,"tak861-panel-")\n    try:\n        for start in range(0,len(values),inputs.ARRAY_ROW_CHUNK):\n            values[start:start+inputs.ARRAY_ROW_CHUNK] = 0.\n        observed = coverage.IS_OBSERVED.to_numpy()\n        values[~observed] = np.nan\n        key_index = pd.MultiIndex.from_frame(coverage[inputs.KEYS])\n        selected_by_category = {}\n        for j,spec in enumerate(specs):\n            if spec[\'measure\']==\'age\':\n                ages = pd.to_numeric(coverage.AGE,errors=\'raise\').to_numpy(dtype=float)\n                if np.isinf(ages).any() or np.any(ages[np.isfinite(ages)]<0):\n                    raise ValueError("Invalid raw age")\n                values[observed,j] = ages[observed]\n            else:\n                selected_by_category.setdefault(spec[\'raw_category\'],[]).append((j,spec[\'measure\']))\n        for category,columns_for_category in selected_by_category.items():\n            cursor = db.execute("SELECT p,d,v,r FROM raw WHERE k=\'FEATURE\' AND c=? ORDER BY p,d",(category,))\n            while True:\n                batch = cursor.fetchmany(inputs.ARRAY_ROW_CHUNK)\n                if not batch: break\n                part = pd.DataFrame.from_records(batch,columns=inputs.KEYS+[\'VALUE\',\'RECENT_VALUE\'])\n                part[\'AS_OF_DATE\']=pd.to_datetime(part.AS_OF_DATE)\n                rows = key_index.get_indexer(pd.MultiIndex.from_frame(part[inputs.KEYS]))\n                annual,recent=part.VALUE.to_numpy(),part.RECENT_VALUE.to_numpy()\n                pct=np.where((annual>6)&(recent>2),np.floor(100.*recent/annual+.5),0.)\n                for j,measure in columns_for_category:\n                    values[rows,j]=annual if measure==\'count\' else pct\n        values.flush()\n        panel_meta=coverage[inputs.KEYS+[\'IS_OBSERVED\',\'MAX_EVENT_DATE\']].copy()\n        panel_meta[\'OBSERVATION_EVIDENCE\']=np.where(observed,\n            \'eligible_source_event_in_12m_window; continuous_coverage_unknown\',\n            \'no_eligible_source_event; window_availability_unknown\')\n        if values.nbytes <= inputs.PANEL_MEMORY_LIMIT:\n            panel=pd.concat([panel_meta[inputs.KEYS+[\'IS_OBSERVED\']],pd.DataFrame(np.asarray(values).copy(),columns=names)],axis=1)\n            panel[\'MAX_EVENT_DATE\']=panel_meta.MAX_EVENT_DATE\n            panel[\'OBSERVATION_EVIDENCE\']=panel_meta.OBSERVATION_EVIDENCE\n            inputs.release_array(values)\n        else:\n            values=inputs.readonly_array(values)\n            panel=inputs.DiskPanel(panel_meta,values,names)\n    except BaseException:\n        inputs.release_array(values)\n        raise\n    specs=inputs.validate_feature_specs(specs)\n    contract = dict(source_id=cfg[\'source_schema\']+\' raw AutoML categories\',\n        source_version=\'LATEST tables read for this extraction; long_content_sha256=\'+long_hash,\n        extraction_sha256=hashlib.sha256(template.encode(\'utf-8\')).hexdigest(),\n        cutoff_rule=\'event_date between DATEADD(month,-12,AS_OF_DATE) and AS_OF_DATE inclusive; recent -4 through -1 months inclusive\',\n        fit_scope=\'deterministic_raw\',feature_semantics_evidence=\'User-supplied full FEATURE_CREATION_PROC raw event/category branches; label-conditioned steps removed; declared adaptations recorded\',\n        vocabulary_scope=\'TRAIN-patient raw category presence/frequency only; RESP never queried or used\',\n        vocabulary_train_patients_sha256=inputs.train_patient_hash(meta),\n        vocabulary_sha256=inputs.stable_hash(names),source_config=cfg,\n        observation_policy=\'eligible event evidence, not continuous observation; no-event window remains missing\',\n        historical_availability=\'unresolved: LATEST source/mapping tables do not establish historical record availability\',\n        raw_adaptations=[\'target-independent current external mapping only; no RESP-conditioned flags\',\n            \'collision-safe JSON medical encounter key at recovered patient/date/diagnosis grain\',\n            \'source-prefixed RX/MX identifiers avoid cross-source collisions\',\n            \'explicit family/level namespace and deterministic hashed sanitizer replace unavailable regexp_comb\',\n            \'ICD10PCS all_procedures_single enrichment omitted; main procedure mapping retained\',\n            \'AGE uses raw calendar-year difference; no target-inferred binary conversion\',\n            \'no legacy caps, VALUE_P scaling, importance filter or model refitting\'])\n    panel = inputs.validate_panel(panel,specs)\n    manifest = inputs.make_panel_manifest(panel,specs,contract)\n    removed = [item for item in candidates if item[\'name\'] not in set(names)]\n    receipt = dict(execution_status=\'raw_extraction_completed_on_configured_sources\',\n        query_count=int(np.ceil(len(requests)/cfg[\'chunk_size\'])),requested_unique_keys=len(requests),\n        returned_long_rows=long_rows,observed_window_keys=int(observed.sum()),unavailable_window_keys=int((~observed).sum()),\n        age_conflict_keys=int(pd.to_numeric(coverage.AGE_CONFLICT,errors=\'raise\').fillna(0).sum()),\n        original_train_candidate_features=len(candidates)+1,retained_input_features=len(specs),\n        vocabulary_policy=\'label-blind TRAIN patient frequency; category-name ties; keep count/recent pairs together; AGE slot reserved\',\n        vocabulary_max_features=limit,removed_feature_candidates=removed,\n        removed_reason=\'predeclared input vocabulary resource cap, not outcome-based selection\',\n        all_candidate_specs=candidates+[age],elapsed_seconds=time.perf_counter()-started,\n        source_content_sha256=long_hash,sql_template_sha256=contract[\'extraction_sha256\'],\n        source_availability_verified=False,clinical_model_results_produced=False,\n        limitations=contract[\'raw_adaptations\']+[contract[\'historical_availability\'],contract[\'observation_policy\']])\n    manifest[\'raw_extraction_receipt\'] = receipt\n    return dict(feature_specs=specs,contract=contract,cached_panel=panel,cached_manifest=manifest,\n                extractor=None,sql_template=None,receipt=receipt)\n\n\ndef build_raw_automl_inputs(read_query, metadata, source_config=None):\n    """One extraction pass into a bounded disk spool; no full long-frame collect."""\n    with tempfile.TemporaryDirectory(prefix=\'tak861-raw-spool-\') as scratch:\n        with closing(sqlite3.connect(os.path.join(scratch,\'raw.sqlite\'))) as db:\n            return _build_raw_automl_inputs(read_query,metadata,source_config,db)\n', 'fresh_counts': '"""Fresh source-to-count construction for the configured cohort.\n\nThe SQL reuses the recovered RX/PX/DX extraction structure with an explicit new\ncontract: twelve calendar positions ending at each snapshot\'s END_DT, source-row\nRX/PX activity and exploded DX occurrences. It does not claim adjudicated use or\nhistorical record availability. Dictionary filters never use RESP summaries.\nThe caller supplies an inherited dictionary and patient split membership;\nhistorical tensor values are never reused. Adapted from the local source-builder\nprototype, with cohort-specific assumptions removed.\n"""\nimport hashlib\nimport json\nimport re\n\nimport numpy as np\nimport pandas as pd\n\n\nDEFAULT_SOURCES = {\n    "cohort": None,  \n    "pharmacy": "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST",\n    "medical": "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST",\n    "code_reference": "DSVC_TAKEDA_TA_PRIVATE.DS_ML_PRD.DX_PX_RX_PLAID",\n    "dx_reference": None,\n}\nPOLICY = {\n    "version": "fresh_source_counts_v1",\n    "positions": 12,\n    "orientation": "TIME_STEP=0 newest; 11 oldest; no export reversal",\n    "window": "start of END_DT calendar month minus 11 months through END_DT inclusive",\n    "event_date": "FILL_DATE for RX; SERVICE_DATE for DX/PX; date-granular",\n    "rx_count": "source-row activity, including all transaction statuses",\n    "px_count": "source-row activity; no distinct-encounter claim",\n    "dx_count": "each nonblank pipe-delimited diagnosis-code occurrence",\n    "claim_dedup": "none; source-row activity is intentional, not adjudicated utilization",\n    "mapping_dedup": "distinct normalized family/code/category pairs",\n    "mapping_multiplicity": "one code may contribute once to each distinct mapped category",\n    "rx_category": "deepest nonblank LEVEL_4_DESC to LEVEL_1_DESC",\n    "px_category": "nonblank LEVEL_1_DESC",\n    "dx_category": "CCSR_CATEGORY_DESCRIPTION",\n    "code_normalization": "RX trim; PX uppercase+trim; DX uppercase+trim+remove periods",\n    "feature_names": "family + \'__\' + trimmed original category; categories are not renamed",\n    "vocabulary": "fixed inherited 1028 count feature names; no new category selection",\n    "vocabulary_scope": "historical dictionary retained; no new-cohort held-out feature discovery",\n    "zero": "no mapped source activity; not proof of observation or padding",\n    "record_availability": "not reconstructed; event cutoff is enforced but ingestion-as-of is unverified",\n    "dictionary_timing": "upstream reference version captured at build; historical-as-of equivalence unverified",\n    "population_labels": "cohort RESP preserved; shared-key changes require producer review",\n}\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _json(value):\n    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef _hash(value):\n    return hashlib.sha256(_json(value).encode()).hexdigest()\n\n\ndef _fqn(value):\n    _require(isinstance(value, str) and re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*\\.[A-Za-z_][A-Za-z0-9_$]*\\.[A-Za-z_][A-Za-z0-9_$]*", value),\n             "Expected an explicit database.schema.table identifier.")\n    name = value.upper()\n    forbidden = ("_DL_POC_", "_TENSOR_MONTHLY", "_PATIENT_SPLIT", "_SNAPSHOTS", "_FEATURE_MAP",\n                 "_MODEL_RUN_", "_INPUTS", "_PREPARED")\n    _require(not any(part in name for part in forbidden), "Historical pipeline artifacts are not source-builder inputs.")\n    return name\n\n\ndef _column(value):\n    _require(isinstance(value, str) and bool(value) and \'"\' not in value and not any(x in value for x in ("\\n", "\\r", ";")),\n             "Invalid column identifier.")\n    return \'"\' + value + \'"\'\n\n\ndef _literal(value):\n    _require(isinstance(value, str), "SQL literal must be text.")\n    return "\'" + value.replace("\'", "\'\'") + "\'"\n\n\ndef _pandas(result):\n    frame = result.copy() if isinstance(result, pd.DataFrame) else result.toPandas()\n    frame.columns = [str(c).upper() for c in frame.columns]\n    _require(frame.columns.is_unique, "Source result has duplicate case-insensitive columns.")\n    return frame\n\n\ndef _schema(read_query, source):\n    result = read_query("SELECT * FROM " + _fqn(source) + " WHERE 1=0")\n    if isinstance(result, pd.DataFrame):\n        types = {str(k): str(v) for k, v in result.dtypes.items()}\n    else:\n        types = {str(k): str(v) for k, v in result.dtypes}\n    _require(len({c.upper() for c in types}) == len(types), "Ambiguous case-insensitive source columns.")\n    return {c.upper(): {"name": c, "type": t} for c, t in types.items()}\n\n\ndef _needed(schema, names, role):\n    _require(set(names) <= set(schema), role + " lacks required source columns: " + repr(sorted(set(names) - set(schema))))\n\n\ndef _discover_dx(read_query, common_scope=None):\n    databases = ["DSVC_TAKEDA_TA_PRIVATE", "DSVC_TAKEDA_FULLMAP_PLAID_PROD"]\n    if common_scope is not None:\n        _require(isinstance(common_scope, (tuple, list)) and len(common_scope) == 2,\n                 "common_scope must contain database and schema names only.")\n        database, schema = common_scope\n        _fqn(str(database) + "." + str(schema) + ".PROBE")\n        databases.insert(0, str(database).upper())\n    candidates, errors = [], []\n    for database in dict.fromkeys(databases):\n        query = f"""SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME\nFROM {database}.INFORMATION_SCHEMA.COLUMNS\nWHERE COLUMN_NAME IN (\'ICD_10_CM_CODE\',\'CCSR_CATEGORY_DESCRIPTION\')\n  AND TABLE_SCHEMA <> \'INFORMATION_SCHEMA\'"""\n        try:\n            frame = _pandas(read_query(query))\n        except Exception as exc:\n            errors.append({"database": database, "error_class": type(exc).__name__})\n            continue\n        _needed({c: None for c in frame}, ["TABLE_CATALOG", "TABLE_SCHEMA", "TABLE_NAME", "COLUMN_NAME"], "DX discovery")\n        for key, group in frame.groupby(["TABLE_CATALOG", "TABLE_SCHEMA", "TABLE_NAME"]):\n            if {"ICD_10_CM_CODE", "CCSR_CATEGORY_DESCRIPTION"} <= set(group.COLUMN_NAME.str.upper()):\n                candidates.append(_fqn(".".join(map(str, key))))\n    candidates = sorted(set(candidates))\n    \n    if common_scope is not None:\n        prefix = ".".join(map(str, common_scope)).upper() + "."\n        scoped = [x for x in candidates if x.startswith(prefix) and x.rsplit(".", 1)[-1] in {"DX_CCSR_LATEST", "DX_CSSR_LATEST"}]\n        if len(scoped) == 1:\n            return scoped[0], {"candidates": candidates, "metadata_errors": errors, "rule": "unique exact recovered name in common scope"}\n    _require(len(candidates) == 1, "Resolve one DX source with ICD_10_CM_CODE and CCSR_CATEGORY_DESCRIPTION; candidates=" + repr(candidates) + "; metadata error classes=" + repr(errors))\n    return candidates[0], {"candidates": candidates, "metadata_errors": errors, "rule": "unique schema-compatible visible reference"}\n\n\ndef resolve_source_contract(read_query, sources=None, common_scope=None, family_column=None,\n                            rx_values=("rx",), px_values=("px",)):\n    """Inspect raw/reference schemas and resolve an explicit fresh-build contract.\n\n    ``read_query(sql)`` returns a Spark DataFrame or small pandas frame. The\n    upstream code reference is used directly, without a target-filtered V63 map.\n    A CODE_TYPE field is usable only when the requested literal family values\n    really occur; its name is never assumed to imply rx/px semantics.\n    """\n    incoming = {} if sources is None else dict(sources)\n    _require(not (set(incoming) - set(DEFAULT_SOURCES)), "Unknown source binding.")\n    resolved = {**DEFAULT_SOURCES, **incoming}\n    discovery = {"rule": "explicit DX reference"}\n    if resolved["dx_reference"] is None:\n        resolved["dx_reference"], discovery = _discover_dx(read_query, common_scope)\n    resolved = {k: _fqn(v) for k, v in resolved.items()}\n    _require(resolved["code_reference"] != "DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MAPPING",\n             "Use the upstream reference, not the empty/target-filtered historical V63 mapping.")\n    schemas = {role: _schema(read_query, name) for role, name in resolved.items()}\n    _needed(schemas["cohort"], ["PATIENT_ID", "END_DT", "RESP"], "Cohort")\n    _needed(schemas["pharmacy"], ["PATIENT_ID", "FILL_DATE"], "Pharmacy")\n    ndc = [c for c in schemas["pharmacy"] if re.sub(r"[^A-Z0-9]", "", c) == "NDC11"]\n    _require(len(ndc) == 1, "Pharmacy must have exactly one NDC11/NDC 11 field.")\n    _needed(schemas["medical"], ["PATIENT_ID", "SERVICE_DATE", "PROCEDURE_CODE", "DIAGNOSIS_CODES"], "Medical")\n    _needed(schemas["code_reference"], ["CODE", "LEVEL_1_DESC", "LEVEL_2_DESC", "LEVEL_3_DESC", "LEVEL_4_DESC"], "Upstream code reference")\n    _needed(schemas["dx_reference"], ["ICD_10_CM_CODE", "CCSR_CATEGORY_DESCRIPTION"], "DX reference")\n    if family_column is None:\n        possible = [x for x in ("C_TYPE", "CODE_TYPE") if x in schemas["code_reference"]]\n        _require(len(possible) == 1, "Specify which verified reference field supplies RX/PX families; found " + repr(possible))\n        family_column = possible[0]\n    _require(family_column.upper() in schemas["code_reference"], "Configured family column is absent.")\n    family_column = schemas["code_reference"][family_column.upper()]["name"]\n    rx_values, px_values = tuple(sorted(set(str(x).strip().lower() for x in rx_values))), tuple(sorted(set(str(x).strip().lower() for x in px_values)))\n    _require(rx_values and px_values and all(rx_values + px_values) and not set(rx_values).intersection(px_values), "RX/PX family literals must be nonempty and disjoint.")\n    type_query = f"SELECT DISTINCT LOWER(TRIM(TO_VARCHAR({_column(family_column)}))) AS FAMILY_VALUE FROM {resolved[\'code_reference\']}"\n    types = _pandas(read_query(type_query))\n    _needed({c: None for c in types}, ["FAMILY_VALUE"], "Family discovery")\n    observed = sorted(set(types.FAMILY_VALUE.dropna().astype(str)))\n    _require(set(rx_values + px_values) <= set(observed), "Upstream reference does not contain configured RX/PX family values; observed=" + repr(observed))\n    contract = {"sources": resolved, "schemas": schemas, "family_column": family_column,\n                "rx_values": list(rx_values), "px_values": list(px_values),\n                "ndc_column": schemas["pharmacy"][ndc[0]]["name"], "dx_discovery": discovery,\n                "observed_family_values": observed, "policy": dict(POLICY)}\n    contract["contract_sha256"] = _hash(contract)\n    return contract\n\n\ndef _check_contract(contract):\n    _require(isinstance(contract, dict), "Source contract must be a dictionary.")\n    payload = {k: v for k, v in contract.items() if k != "contract_sha256"}\n    _require(contract.get("contract_sha256") == _hash(payload), "Source contract changed after resolution.")\n    _require(contract.get("policy") == POLICY, "Unknown or altered source-count policy.")\n    for value in contract["sources"].values():\n        _fqn(value)\n\n\ndef mapping_ctes(contract):\n    _check_contract(contract)\n    source, dx = contract["sources"]["code_reference"], contract["sources"]["dx_reference"]\n    family = _column(contract["family_column"])\n    rx_types = ",".join(map(_literal, contract["rx_values"]))\n    px_types = ",".join(map(_literal, contract["px_values"]))\n    return f"""rx_map AS (\n SELECT DISTINCT TRIM(TO_VARCHAR(CODE)) AS CODE,\n COALESCE(NULLIF(TRIM(TO_VARCHAR(LEVEL_4_DESC)),\'\'), NULLIF(TRIM(TO_VARCHAR(LEVEL_3_DESC)),\'\'),\n          NULLIF(TRIM(TO_VARCHAR(LEVEL_2_DESC)),\'\'), NULLIF(TRIM(TO_VARCHAR(LEVEL_1_DESC)),\'\')) AS CATEGORY\n FROM {source} WHERE LOWER(TRIM(TO_VARCHAR({family}))) IN ({rx_types})\n), px_map AS (\n SELECT DISTINCT UPPER(TRIM(TO_VARCHAR(CODE))) AS CODE,\n NULLIF(TRIM(TO_VARCHAR(LEVEL_1_DESC)),\'\') AS CATEGORY\n FROM {source} WHERE LOWER(TRIM(TO_VARCHAR({family}))) IN ({px_types})\n), dx_map AS (\n SELECT DISTINCT UPPER(REPLACE(TRIM(TO_VARCHAR(ICD_10_CM_CODE)),\'.\',\'\')) AS CODE,\n NULLIF(TRIM(TO_VARCHAR(CCSR_CATEGORY_DESCRIPTION)),\'\') AS CATEGORY FROM {dx}\n), all_mapping AS (\n SELECT \'RX\' AS FAMILY, CODE, CATEGORY FROM rx_map WHERE CODE IS NOT NULL AND CODE<>\'\' AND CATEGORY IS NOT NULL\n UNION ALL SELECT \'PX\', CODE, CATEGORY FROM px_map WHERE CODE IS NOT NULL AND CODE<>\'\' AND CATEGORY IS NOT NULL\n UNION ALL SELECT \'DX\', CODE, CATEGORY FROM dx_map WHERE CODE IS NOT NULL AND CODE<>\'\' AND CATEGORY IS NOT NULL\n)"""\n\n\ndef mapping_audit_query(contract):\n    return "WITH " + mapping_ctes(contract) + """\nSELECT FAMILY, COUNT(*) AS MAPPING_PAIRS, COUNT(DISTINCT CODE) AS MAPPED_CODES,\n COUNT(DISTINCT CATEGORY) AS CATEGORIES, TO_VARCHAR(HASH_AGG(HASH(FAMILY,CODE,CATEGORY))) AS MAPPING_HASH\nFROM all_mapping GROUP BY FAMILY ORDER BY FAMILY"""\n\n\ndef cohort_query(contract):\n    _check_contract(contract)\n    return "SELECT PATIENT_ID, END_DT, RESP FROM " + contract["sources"]["cohort"]\n\n\ndef monthly_count_query(contract):\n    """Push raw scans and category aggregation to Snowflake, never the driver."""\n    _check_contract(contract)\n    source = contract["sources"]\n    ndc = _column(contract["ndc_column"])\n    return f"""WITH cohort AS (\n SELECT PATIENT_ID, END_DT::DATE AS END_DT,\n DATE_TRUNC(\'MONTH\',DATEADD(\'MONTH\',-11,END_DT::DATE))::DATE AS WINDOW_START\n FROM {source[\'cohort\']}\n), bounds AS (\n SELECT PATIENT_ID, MIN(WINDOW_START) AS MIN_DATE, MAX(END_DT) AS MAX_DATE\n FROM cohort GROUP BY PATIENT_ID\n), {mapping_ctes(contract)},\nrx_code_day AS (\n SELECT r.PATIENT_ID, r.FILL_DATE::DATE AS EVENT_DATE, TRIM(TO_VARCHAR(r.{ndc})) AS CODE, COUNT(*) AS VALUE\n FROM {source[\'pharmacy\']} r JOIN bounds b ON r.PATIENT_ID=b.PATIENT_ID\n AND r.FILL_DATE >= b.MIN_DATE AND r.FILL_DATE < DATEADD(\'DAY\',1,b.MAX_DATE)\n WHERE r.{ndc} IS NOT NULL GROUP BY 1,2,3\n), mx_day AS (\n SELECT m.PATIENT_ID, m.SERVICE_DATE::DATE AS EVENT_DATE, TO_VARCHAR(m.DIAGNOSIS_CODES) AS DIAGNOSIS_CODES,\n UPPER(TRIM(TO_VARCHAR(m.PROCEDURE_CODE))) AS PROCEDURE_CODE, COUNT(*) AS VALUE\n FROM {source[\'medical\']} m JOIN bounds b ON m.PATIENT_ID=b.PATIENT_ID\n AND m.SERVICE_DATE >= b.MIN_DATE AND m.SERVICE_DATE < DATEADD(\'DAY\',1,b.MAX_DATE)\n WHERE m.DIAGNOSIS_CODES IS NOT NULL OR m.PROCEDURE_CODE IS NOT NULL GROUP BY 1,2,3,4\n), dx_exploded AS (\n SELECT m.PATIENT_ID, m.EVENT_DATE, m.VALUE,\n UPPER(REPLACE(TRIM(f.VALUE::VARCHAR),\'.\',\'\')) AS CODE\n FROM mx_day m, LATERAL FLATTEN(INPUT=>SPLIT(m.DIAGNOSIS_CODES,\'|\')) f\n WHERE TRIM(f.VALUE::VARCHAR)<>\'\'\n), event_days AS (\n SELECT r.PATIENT_ID,r.EVENT_DATE,\'RX\' AS FAMILY,d.CATEGORY,SUM(r.VALUE) AS VALUE\n FROM rx_code_day r JOIN rx_map d ON r.CODE=d.CODE WHERE d.CATEGORY IS NOT NULL GROUP BY 1,2,3,4\n UNION ALL\n SELECT m.PATIENT_ID,m.EVENT_DATE,\'PX\',d.CATEGORY,SUM(m.VALUE)\n FROM mx_day m JOIN px_map d ON m.PROCEDURE_CODE=d.CODE WHERE d.CATEGORY IS NOT NULL GROUP BY 1,2,3,4\n UNION ALL\n SELECT m.PATIENT_ID,m.EVENT_DATE,\'DX\',d.CATEGORY,SUM(m.VALUE)\n FROM dx_exploded m JOIN dx_map d ON m.CODE=d.CODE WHERE d.CATEGORY IS NOT NULL GROUP BY 1,2,3,4\n)\nSELECT s.PATIENT_ID,s.END_DT,DATEDIFF(\'MONTH\',e.EVENT_DATE,s.END_DT)::INTEGER AS TIME_STEP,\n e.FAMILY,e.FAMILY||\'__\'||e.CATEGORY AS FEATURE_NAME,SUM(e.VALUE) AS VALUE,\n MIN(e.EVENT_DATE) AS FIRST_EVENT_DATE,MAX(e.EVENT_DATE) AS LAST_EVENT_DATE\nFROM cohort s JOIN event_days e ON s.PATIENT_ID=e.PATIENT_ID\n AND e.EVENT_DATE >= s.WINDOW_START AND e.EVENT_DATE <= s.END_DT\nGROUP BY 1,2,3,4,5"""\n\n\ndef normalize_cohort(frame, expected_population=None):\n    frame = _pandas(frame)\n    _needed({c: None for c in frame}, ["PATIENT_ID", "END_DT", "RESP"], "Cohort")\n    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()\n    _require(len(out) > 0 and not out.isna().any().any(), "Cohort keys/labels must be nonempty and complete.")\n    _require(out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all(), "Original patient keys must remain nonblank strings.")\n    dates = pd.to_datetime(out.END_DT, errors="raise")\n    _require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(), "Snapshot END_DT must be an exact date.")\n    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")\n    _require(out.RESP.isin([0, 1]).all(), "RESP must be exactly 0/1.")\n    out["RESP"] = out.RESP.astype("int64")\n    _require(not out.duplicated(["PATIENT_ID", "END_DT"]).any(), "Duplicate cohort snapshot keys.")\n    out = out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n    if expected_population is not None:\n        observed = {"snapshots": len(out), "patients": int(out.PATIENT_ID.nunique()), "positives": int(out.RESP.sum())}\n        _require(observed == expected_population, "Source V63 population differs from the declared population: " + repr(observed))\n    return out\n\n\ndef validate_aggregate_rows(rows, metadata, feature_map):\n    """Small-frame validation/reference used by tests and optional small extracts.\n\n    The production Spark path applies equivalent bounds and uniqueness checks\n    without collecting source events or the full aggregated table to pandas.\n    """\n    frame = _pandas(rows)\n    required = ["PATIENT_ID", "END_DT", "TIME_STEP", "FEATURE_NAME", "VALUE", "FIRST_EVENT_DATE", "LAST_EVENT_DATE"]\n    _needed({c: None for c in frame}, required, "Monthly aggregates")\n    _require(not frame[required].isna().any().any(), "Null monthly aggregate field.")\n    cohort = normalize_cohort(metadata)\n    keys = set(zip(cohort.PATIENT_ID, cohort.END_DT))\n    dates = pd.to_datetime(frame.END_DT, errors="raise")\n    _require(dates.eq(dates.dt.normalize()).all(), "Aggregate END_DT contains intraday time.")\n    frame["END_DT"] = dates.dt.strftime("%Y-%m-%d")\n    _require(set(zip(frame.PATIENT_ID, frame.END_DT)) <= keys, "Unexpected aggregate snapshot.")\n    _require(set(frame.FEATURE_NAME) <= set(feature_map.FEATURE_NAME), "Unknown aggregate feature.")\n    steps, values = pd.to_numeric(frame.TIME_STEP), pd.to_numeric(frame.VALUE)\n    _require(np.isfinite(steps).all() and steps.isin(range(12)).all(), "Invalid monthly position.")\n    _require(np.isfinite(values).all() and values.gt(0).all() and values.eq(np.floor(values)).all(), "Counts must be positive finite integers.")\n    _require(np.array_equal(values.to_numpy(dtype=np.float64), values.to_numpy(dtype=np.float32).astype(np.float64)), "Count loses exact float32 precision.")\n    _require(not frame.duplicated(["PATIENT_ID", "END_DT", "TIME_STEP", "FEATURE_NAME"]).any(), "Duplicate monthly coordinate.")\n    first, last = pd.to_datetime(frame.FIRST_EVENT_DATE), pd.to_datetime(frame.LAST_EVENT_DATE)\n    period = dates.dt.to_period("M")\n    starts = (period - 11).dt.to_timestamp()\n    first_step = (dates.dt.year - first.dt.year) * 12 + dates.dt.month - first.dt.month\n    last_step = (dates.dt.year - last.dt.year) * 12 + dates.dt.month - last.dt.month\n    _require(first.ge(starts).all() and last.le(dates).all() and first.le(last).all()\n             and first_step.eq(steps).all() and last_step.eq(steps).all(), "Event outside its snapshot cutoff/calendar bin.")\n    return frame\n\n\ndef same_cohort(left, right):\n    """Compare normalized values across pandas object/StringDtype versions."""\n    return normalize_cohort(left).to_csv(index=False) == normalize_cohort(right).to_csv(index=False)\n\n\ndef audit_feature_overlap(observed, reference_map, max_missing_feature_fraction=.10):\n    if not 0 <= max_missing_feature_fraction < 1:\n        raise ValueError("max_missing_feature_fraction must be in [0,1)")\n    missing = sorted(set(reference_map.FEATURE_NAME) - set(observed.FEATURE_NAME))\n    unknown = sorted(set(observed.FEATURE_NAME) - set(reference_map.FEATURE_NAME))\n    print("Count dictionary audit:", len(missing), "inherited features without activity;",\n          len(unknown), "observed categories outside dictionary")\n    if len(missing) / len(reference_map) > max_missing_feature_fraction:\n        raise ValueError("Inherited feature inactivity exceeds declared floor; verify mapping names/source drift before explicitly adjusting max_missing_feature_fraction")\n    return dict(missing_feature_fraction=len(missing)/len(reference_map),\n                max_missing_feature_fraction=max_missing_feature_fraction)\n\n\ndef build_source_frames(spark, read_query, contract, reference_map, prepared_metadata,\n                        max_missing_feature_fraction=.10):\n    """Return fresh Spark frames plus a JSON-serializable source manifest.\n\n    This function performs no warehouse writes. The caller materializes the\n    arrays into the new run\'s INPUTS artifact. Keep\n    ``cached_aggregates`` alive until all outputs are materialized, then unpersist.\n    """\n    from pyspark import StorageLevel\n    from pyspark.sql import functions as F\n    _check_contract(contract)\n    cohort = normalize_cohort(read_query(cohort_query(contract)))\n    if not same_cohort(cohort, prepared_metadata):\n        raise ValueError("Cohort changed between audit and count extraction")\n    mapping_before = _pandas(read_query(mapping_audit_query(contract))).sort_values("FAMILY").reset_index(drop=True)\n    _needed({c: None for c in mapping_before}, ["FAMILY", "MAPPING_PAIRS", "MAPPED_CODES", "CATEGORIES", "MAPPING_HASH"], "Mapping audit")\n    _require(set(mapping_before.FAMILY) == {"RX", "DX", "PX"} and mapping_before.MAPPING_PAIRS.gt(0).all(), "A source mapping branch is empty.")\n    sql = monthly_count_query(contract)\n    aggregates = read_query(sql).persist(StorageLevel.MEMORY_AND_DISK)\n    try:\n        observed = aggregates.groupBy(\'FEATURE_NAME\',\'FAMILY\').agg(F.sum(\'VALUE\').alias(\'ACTIVITY\')).toPandas()\n        fmap = reference_map.copy()\n        overlap_audit = audit_feature_overlap(observed, fmap, max_missing_feature_fraction)\n        family_counts = {str(k): int(v) for k, v in fmap.FAMILY.value_counts().items()}\n        matched = observed[observed.FEATURE_NAME.isin(fmap.FEATURE_NAME)]\n        if set(matched.FAMILY) != {\'DX\', \'PX\', \'RX\'}:\n            raise ValueError("Fresh counts do not match all three inherited dictionary families; verify mappings")\n        metadata = prepared_metadata.copy()\n        split_config = dict(n_timesteps=12, method=\'inherited_patient_membership_plus_hash_new_ids\',\n            feature_order_sha256=hashlib.sha256(json.dumps(fmap.FEATURE_NAME.tolist(), ensure_ascii=False).encode()).hexdigest())\n        metadata[\'SPLIT_CONFIG\'] = _json(split_config)\n        snapshots = spark.createDataFrame([(r.PATIENT_ID, pd.Timestamp(r.END_DT).date(), int(r.RESP)) for r in cohort.itertuples()],\n                                         "PATIENT_ID string, END_DT date, RESP long")\n        split = spark.createDataFrame([(r.PATIENT_ID, pd.Timestamp(r.END_DT).date(), int(r.RESP), r.SPLIT, r.SPLIT_CONFIG) for r in metadata.itertuples()],\n                                     "PATIENT_ID string, END_DT date, RESP long, SPLIT string, SPLIT_CONFIG string")\n        feature_map = spark.createDataFrame([(int(r.FEATURE_INDEX), r.FEATURE_COLUMN, r.FEATURE_NAME, r.FAMILY) for r in fmap.itertuples()],\n                                           "FEATURE_INDEX long, FEATURE_COLUMN string, FEATURE_NAME string, FAMILY string")\n        keys = ["PATIENT_ID", "END_DT"]\n        required = keys + ["TIME_STEP", "FEATURE_NAME", "FAMILY", "VALUE", "FIRST_EVENT_DATE", "LAST_EVENT_DATE"]\n        bad = None\n        for name in required:\n            bad = F.col(name).isNull() if bad is None else bad | F.col(name).isNull()\n        start = F.add_months(F.trunc(F.col("END_DT"), "month"), -11)\n        first_step = (F.year("END_DT") - F.year("FIRST_EVENT_DATE")) * 12 + F.month("END_DT") - F.month("FIRST_EVENT_DATE")\n        last_step = (F.year("END_DT") - F.year("LAST_EVENT_DATE")) * 12 + F.month("END_DT") - F.month("LAST_EVENT_DATE")\n        value = F.col("VALUE").cast("double")\n        bad = (bad | ~F.col("TIME_STEP").isin(list(range(12))) | (value <= 0) | F.isnan(value)\n               | (value != F.floor(value)) | (value.cast("float").cast("double") != value)\n               | (F.col("FIRST_EVENT_DATE") < start) | (F.col("LAST_EVENT_DATE") > F.col("END_DT"))\n               | (F.col("FIRST_EVENT_DATE") > F.col("LAST_EVENT_DATE"))\n               | (first_step != F.col("TIME_STEP")) | (last_step != F.col("TIME_STEP")))\n        _require(aggregates.where(bad).limit(1).count() == 0, "Invalid source count, coordinate or event cutoff.")\n        _require(aggregates.groupBy(*(keys + ["TIME_STEP", "FEATURE_NAME"])).count().where(F.col("count") != 1).limit(1).count() == 0,\n                 "Duplicate monthly source coordinate.")\n        _require(aggregates.join(snapshots.select(*keys), keys, "left_anti").limit(1).count() == 0, "Source aggregation contains an unexpected snapshot.")\n        mapped = aggregates.join(feature_map.select("FEATURE_NAME", "FEATURE_COLUMN"), "FEATURE_NAME", "inner")\n        aliases = fmap.FEATURE_COLUMN.tolist()\n        wide = mapped.groupBy(*(keys + ["TIME_STEP"])).pivot("FEATURE_COLUMN", aliases).agg(F.sum("VALUE"))\n        positions = spark.range(12).select(F.col("id").cast("int").alias("TIME_STEP"))\n        spine = snapshots.crossJoin(positions)\n        tensor = spine.join(wide, keys + ["TIME_STEP"], "left").fillna(0, subset=aliases)\n        tensor = tensor.select("PATIENT_ID", "END_DT", "RESP", "TIME_STEP", *[F.col(c).cast("float").alias(c) for c in aliases])\n        \n        _require(same_cohort(cohort, read_query(cohort_query(contract))), "Source cohort changed during construction.")\n        mapping_after = _pandas(read_query(mapping_audit_query(contract))).sort_values("FAMILY").reset_index(drop=True)\n        _require(mapping_before.equals(mapping_after), "Source mappings changed during construction.")\n        \n        audit_records = [{str(k): (v.item() if isinstance(v, np.generic) else str(v) if not isinstance(v, (str, int, float, bool, type(None))) else v)\n                          for k, v in row.items()} for row in mapping_before.to_dict("records")]\n        manifest = {"schema": 1, "contract": contract, "source_sql_sha256": hashlib.sha256(sql.encode()).hexdigest(),\n                    "mapping_audit": audit_records, "feature_overlap_audit": overlap_audit,\n                    "feature_count": len(fmap), "expected_feature_count": 1028,\n                    "unmapped_observed_categories": sorted(set(observed.FEATURE_NAME)-set(fmap.FEATURE_NAME)),\n                    "out_of_vocabulary_snapshot_activity": float(observed.loc[~observed.FEATURE_NAME.isin(fmap.FEATURE_NAME),\'ACTIVITY\'].sum()),\n                    "total_mapped_snapshot_activity": float(observed.ACTIVITY.sum()),\n                    "inherited_features_without_activity": sorted(set(fmap.FEATURE_NAME)-set(observed.FEATURE_NAME)),\n                    "family_counts": family_counts, "feature_map_sha256": hashlib.sha256(fmap.to_csv(index=False, lineterminator="\\n").encode()).hexdigest(),\n                    "population": {"snapshots": len(cohort), "patients": int(cohort.PATIENT_ID.nunique()), "positives": int(cohort.RESP.sum())},\n                    "split_config": split_config, "historical_inputs_read": [\'FEATURE_MAP\',\'PATIENT_SPLIT\'],\n                    "validation": "source schema/family checks, unique cohort, paired mapping hashes, integer counts, exact snapshot cutoffs/calendar bins, complete new spine"}\n        manifest["manifest_sha256"] = _hash(manifest)\n        return {"FEATURE_MAP": feature_map, "SNAPSHOTS": snapshots, "PATIENT_SPLIT": split, "TENSOR_MONTHLY": tensor,\n                "manifest": manifest, "metadata": metadata, "features": fmap.FEATURE_NAME.tolist(),\n                "feature_map": fmap, "cached_aggregates": aggregates}\n    except BaseException:\n        aggregates.unpersist(blocking=False)\n        raise\n', 'client_universe': '"""Bind the configured population without reusing prior tensors.\n\nOnly dictionary and patient assignments are inherited. Claims are reaggregated\nfor every new universe key; the source procedure\'s RESP is never regenerated.\n"""\nimport hashlib\nimport json\nimport re\n\nimport numpy as np\nimport pandas as pd\n\ninputs = fresh_counts = count_loader = None  \nTABLE_NAME = "TAK861_TX_READY_PATIENT_UNIVERSE_DL"\n\n\ndef frame(value):\n    result = value.copy() if isinstance(value, pd.DataFrame) else value.toPandas()\n    result.columns = [str(c).upper() for c in result.columns]\n    if not result.columns.is_unique:\n        raise ValueError("Duplicate source columns")\n    return result\n\n\ndef resolve_universe(read_query, explicit=None, table_name=TABLE_NAME):\n    if not isinstance(table_name, str) or not re.fullmatch(r\'[A-Z_][A-Z0-9_$]*\', table_name):\n        raise ValueError(\'Invalid configured universe table name\')\n    if explicit:\n        name = fresh_counts._fqn(explicit)\n        if name.rsplit(\'.\', 1)[-1] != table_name:\n            raise ValueError("Expected the configured requested patient universe")\n        return name\n    found = frame(read_query(\n        "SELECT TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME FROM "\n        "DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "\n        "WHERE TABLE_NAME=\'" + table_name + "\'"))\n    names = sorted(set(\'.\'.join(map(str, r)) for r in\n                       found[[\'TABLE_CATALOG\', \'TABLE_SCHEMA\', \'TABLE_NAME\']].itertuples(index=False, name=None)))\n    if len(names) != 1:\n        raise ValueError("Set CONFIG[\'client_universe\'][\'table\'] to the verified fully qualified table; "\n                         "metadata discovery requires exactly one match, found " + repr(names))\n    return fresh_counts._fqn(names[0])\n\n\ndef align_universe(new, prior, seed=42):\n    """Preserve every known patient\'s partition; hash only genuinely new IDs."""\n    new, prior = frame(new), frame(prior)\n    new[\'SPLIT\'] = \'train\'\n    new = inputs.normalize_snapshots(new).sort_values([\'PATIENT_ID\', \'END_DT\']).reset_index(drop=True)\n    prior = inputs.normalize_snapshots(prior).sort_values([\'PATIENT_ID\', \'END_DT\']).reset_index(drop=True)\n    membership = prior.groupby(\'PATIENT_ID\').SPLIT.first().to_dict()\n    if prior.groupby(\'PATIENT_ID\').SPLIT.nunique().gt(1).any():\n        raise ValueError("Prior patient membership crosses splits")\n    def assignment(patient):\n        if patient in membership:\n            return membership[patient]\n        value = int(hashlib.sha256((str(seed) + \':\' + patient).encode()).hexdigest()[:16], 16) / 2**64\n        return \'train\' if value < .7 else \'validation\' if value < .85 else \'test\'\n    new[\'SPLIT\'] = new.PATIENT_ID.map(assignment)\n    keys = [\'PATIENT_ID\', \'END_DT\']\n    overlap = prior.merge(new, on=keys, how=\'outer\', suffixes=(\'_old\', \'_new\'), indicator=True)\n    shared = overlap._merge.eq(\'both\')\n    if overlap._merge.eq(\'left_only\').any():\n        raise ValueError("Cohort removes prior snapshot keys; verify patient-key format and producer scope before proceeding")\n    changed = int((shared & overlap.RESP_old.ne(overlap.RESP_new)).sum())\n    if changed:\n        raise ValueError("Cohort changes RESP on existing snapshot keys; verify producer rules before proceeding")\n    for split in (\'train\', \'validation\', \'test\'):\n        if set(new.loc[new.SPLIT.eq(split), \'RESP\']) != {0, 1}:\n            raise ValueError("Revised universe needs both classes in " + split)\n    receipt = dict(old_snapshots=len(prior), new_snapshots=len(new),\n                   old_positives=int(prior.RESP.sum()), new_positives=int(new.RESP.sum()),\n                   positive_snapshot_ratio=float(new.RESP.sum()/prior.RESP.sum()) if prior.RESP.sum() else None,\n                   old_positive_patients=int(prior.loc[prior.RESP.eq(1), \'PATIENT_ID\'].nunique()),\n                   new_positive_patients=int(new.loc[new.RESP.eq(1), \'PATIENT_ID\'].nunique()),\n                   added_snapshots=int(overlap._merge.eq(\'right_only\').sum()),\n                   removed_snapshots=int(overlap._merge.eq(\'left_only\').sum()), changed_shared_labels=changed,\n                   new_patients=int((~new.PATIENT_ID.isin(membership)).groupby(new.PATIENT_ID).any().sum()),\n                   client_statement=\'configured population measured against independent expected counts\',\n                   split_policy=\'inherit all known patients; new IDs SHA256(seed:patient) 70/15/15\',\n                   seed=seed, cohort_sha256=inputs.frame_hash(new),\n                   prior_membership_sha256=inputs.stable_hash(membership))\n    receipt[\'population_by_split\'] = [dict(universe=label, split=split, snapshots=len(part),\n        patients=int(part.PATIENT_ID.nunique()), positives=int(part.RESP.sum()),\n        positive_rate=float(part.RESP.mean()), first_date=str(part.END_DT.min().date()),\n        last_date=str(part.END_DT.max().date()))\n        for label, population in [(\'old\',prior),(\'new\',new)] for split,part in population.groupby(\'SPLIT\')]\n    receipt[\'snapshot_changes_by_split\'] = [dict(split=split,\n        added=int((overlap._merge.eq(\'right_only\') & overlap.SPLIT_new.eq(split)).sum()),\n        removed=int((overlap._merge.eq(\'left_only\') & overlap.SPLIT_old.eq(split)).sum()),\n        added_positive_snapshots=int((overlap._merge.eq(\'right_only\') & overlap.SPLIT_new.eq(split) & overlap.RESP_new.eq(1)).sum()))\n        for split in (\'train\',\'validation\',\'test\')]\n    receipt[\'snapshot_multiplicity_by_patient_class\'] = [dict(universe=label,patient_class=int(cls),\n        patients=len(part),mean_snapshots=float(part[\'size\'].mean()),max_snapshots=int(part[\'size\'].max()))\n        for label,population in [(\'old\',prior),(\'new\',new)]\n        for cls,part in population.groupby(\'PATIENT_ID\').RESP.agg([\'max\',\'size\']).groupby(\'max\')]\n    return new, receipt\n\n\ndef assert_expected_population(audit, expected):\n    """Assert independently declared aggregate counts before any claim extraction."""\n    splits = {r[\'split\']: {k: int(r[k]) for k in (\'snapshots\', \'patients\', \'positives\')}\n              for r in audit[\'population_by_split\'] if r[\'universe\'] == \'new\'}\n    observed = dict(snapshots=int(audit[\'new_snapshots\']),\n                    patients=sum(r[\'patients\'] for r in splits.values()),\n                    positives=int(audit[\'new_positives\']), per_split=splits)\n    required = (\'snapshots\', \'patients\', \'positives\')\n    complete = (isinstance(expected, dict) and\n                all(type(expected.get(k)) is int and expected[k] >= 0 for k in required) and\n                isinstance(expected.get(\'per_split\'), dict) and\n                set(expected[\'per_split\']) == {\'train\', \'validation\', \'test\'} and\n                all(isinstance(row, dict) and\n                    all(type(row.get(k)) is int and row[k] >= 0 for k in required)\n                    for row in expected[\'per_split\'].values()))\n    if not complete:\n        raise ValueError("Declare CONFIG[\'client_universe\'][\'expected_population\'] from the independently reviewed audit receipt. "\n                         "Observed aggregate population (no claim extraction started): " + json.dumps(observed, sort_keys=True))\n    declared = {k: expected[k] for k in required}\n    declared[\'per_split\'] = {s: {k: row[k] for k in required} for s, row in expected[\'per_split\'].items()}\n    if declared != observed:\n        raise ValueError("Configured population differs from the universe audit; preserve the cohort stop. "\n                         "Expected: " + json.dumps(declared, sort_keys=True) + "; observed: " + json.dumps(observed, sort_keys=True))\n    return observed\n\n\ndef load_client_counts(spark, read_query, read_sf, config):\n    """Fresh raw counts with inherited 1028-column dictionary and patient splits."""\n    name = resolve_universe(read_query, config.get(\'table\'), config.get(\'table_name\', TABLE_NAME))\n    prior = frame(read_sf(\'PATIENT_SPLIT\'))\n    new = frame(read_query(\'SELECT PATIENT_ID,END_DT,RESP FROM \' + name))\n    metadata, audit = align_universe(new, prior, config.get(\'split_seed\', 42))\n    assert_expected_population(audit, config.get(\'expected_population\'))\n    mapping = frame(read_sf(\'FEATURE_MAP\')).sort_values(\'FEATURE_INDEX\').reset_index(drop=True)\n    if (len(mapping) != 1028 or mapping.FEATURE_INDEX.tolist() != list(range(1028))\n            or mapping.FEATURE_COLUMN.tolist() != [f\'F{i:04d}\' for i in range(1028)]\n            or not mapping.FEATURE_NAME.is_unique):\n        raise ValueError("Inherited count dictionary must contain ordered 1028 unique features")\n    mapping[\'FAMILY\'] = mapping.FEATURE_NAME.str.split(\'__\').str[0]\n    if set(mapping.FAMILY) != {\'DX\', \'PX\', \'RX\'}:\n        raise ValueError("Count dictionary naming differs from recovered FAMILY__category contract")\n    sources = dict(config.get(\'sources\', {}), cohort=name)\n    contract = fresh_counts.resolve_source_contract(read_query, sources=sources,\n        common_scope=config.get(\'common_scope\'), family_column=config.get(\'family_column\'))\n    built = fresh_counts.build_source_frames(spark, read_query, contract,\n        reference_map=mapping, prepared_metadata=metadata,\n        max_missing_feature_fraction=config.get(\'max_missing_feature_fraction\', .10))\n    try:\n        counts = count_loader.load_count_inputs(lambda suffix: built[suffix], name, \'fresh_client_universe_v1\')\n        counts[\'contract\'][\'client_universe_audit\'] = audit\n        counts[\'contract\'][\'fresh_source_manifest\'] = built[\'manifest\']\n        counts[\'contract\'][\'inherited_dictionary_provenance\'] = \'Names/order retained from previous FEATURE_MAP; historical vocabulary fitting scope not independently established\'\n        counts[\'contract\'][\'historical_count_value_equivalence\'] = \'Not claimed: new raw aggregation and current LATEST sources; prior tensor values not reused\'\n        counts[\'contract\'][\'provenance_status\'] = \'Fresh aggregation for cohort; current references, historical availability unresolved\'\n        return counts\n    finally:\n        built[\'cached_aggregates\'].unpersist(blocking=False)\n\n\ndef check_activity_alignment(data):\n    """Medical count activity requires raw medical AutoML coverage at step zero.\n\n    RX-only disagreements can reflect final-status filters, so are reported only.\n    This is a consistency check, not an enrollment assertion or arbitrary floor.\n    """\n    names = data[\'count_contract\'][\'feature_names\']\n    ix = [i for i, n in enumerate(names) if n.startswith((\'DX__\', \'PX__\'))]\n    active = np.zeros(len(data[\'y\']), bool)\n    for start in range(0, len(active), 256):\n        if ix:\n            active[start:start+256] = np.any(data[\'X_count\'][start:start+256, :, ix] > 0, axis=(1, 2))\n    missing = ~data[\'summary_mask\'][:, 0]\n    if np.any(active & missing):\n        raise ValueError("Medical count activity but no current AutoML event coverage; inspect patient keys/source alignment")\n    return dict(medical_active_snapshots=int(active.sum()), current_automl_observed=int((~missing).sum()),\n                snapshots=len(missing), medical_coverage_mismatches=int((active & missing).sum()))\n', 'comparison': '"""Aggregate evaluation for locked finalists, never model selection on TEST.\n\nAverage precision (AP) and trapezoidal PR area are deliberately separate.\nBootstrap intervals condition on the fitted predictions and resample patients,\nkeeping all snapshots and both models paired. They do not include retraining\nvariation, resolve unknown provenance, or make a reused holdout prospective.\n"""\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\nfrom sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,\n                             precision_recall_curve, roc_auc_score)\n\n\ndef _inputs(y, scores):\n    y, scores = np.asarray(y), np.asarray(scores, dtype=float)\n    if y.ndim != 1 or scores.ndim != 1 or not len(y) or len(y) != len(scores):\n        raise ValueError("Labels and probability scores must be equal nonempty vectors")\n    if not np.isin(y, [0, 1]).all():\n        raise ValueError("Labels must be binary and nonmissing")\n    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Scores must be finite probabilities in [0,1]; logits require an explicit conversion")\n    return y.astype(np.int8), scores\n\n\ndef _ratio(numerator, denominator):\n    return float(numerator / denominator) if denominator else None\n\n\ndef _json_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef _tie_keys(n, tie_breaker):\n    if tie_breaker is None:\n        return np.arange(n), "frozen_input_row_order"\n    keys = np.asarray(tie_breaker).astype(str)\n    if keys.ndim != 1 or len(keys) != n or len(np.unique(keys)) != n:\n        raise ValueError("Tie keys must be unique, stable snapshot keys in prediction order")\n    return keys, "supplied_unique_snapshot_keys"\n\n\ndef topk_metrics(y, scores, fraction=0.1, tie_breaker=None):\n    """Exactly ceil(fraction*N) selected; stable ties are independent of labels."""\n    y, scores = _inputs(y, scores)\n    if not 0 < float(fraction) <= 1:\n        raise ValueError("Top fraction must lie in (0,1]")\n    keys, tie_rule = _tie_keys(len(y), tie_breaker)\n    k = max(1, int(math.ceil(float(fraction) * len(y))))\n    order = np.lexsort((keys, -scores))\n    selected = order[:k]\n    positives = int(y.sum())\n    captured = int(y[selected].sum())\n    boundary = float(scores[selected[-1]])\n    tied = scores == boundary\n    tied_selected = int(tied[selected].sum())\n    precision = captured / k\n    prevalence = positives / len(y)\n    return dict(fraction=float(fraction), selected=k, selected_positives=captured,\n                actual_population_fraction=k / len(y), precision=precision,\n                recall=_ratio(captured, positives), lift=_ratio(precision, prevalence),\n                cutoff_score=boundary, n_tied_at_cutoff=int(tied.sum()),\n                n_tied_selected=tied_selected, tie_crosses_boundary=bool(tied.sum() > tied_selected),\n                tie_rule=tie_rule, rule="ceil(fraction*N), descending probability, ascending stable key")\n\n\ndef binary_metrics(y, scores, threshold=None, top_fractions=(0.1,), calibration_bins=10,\n                   tie_breaker=None):\n    y, scores = _inputs(y, scores)\n    if isinstance(calibration_bins, bool) or not 1 <= int(calibration_bins) <= 100:\n        raise ValueError("Use 1 to 100 fixed-width calibration bins")\n    calibration_bins = int(calibration_bins)\n    positives, n = int(y.sum()), len(y)\n    both = 0 < positives < n\n    precision, recall, _ = precision_recall_curve(y, scores) if positives else (None, None, None)\n    integrate = getattr(np, "trapezoid", np.trapz if hasattr(np, "trapz") else None)\n    result = dict(snapshots=n, positives=positives, negatives=n - positives, prevalence=positives / n,\n                  average_precision=float(average_precision_score(y, scores)) if positives else None,\n                  pr_auc_trapezoidal=float(integrate(precision[::-1], recall[::-1])) if positives else None,\n                  roc_auc=float(roc_auc_score(y, scores)) if both else None,\n                  brier_score=float(brier_score_loss(y, scores)),\n                  log_loss=float(log_loss(y, scores, labels=[0, 1])),\n                  metric_definition="AP=sum(recall increment*precision); PR area uses trapezoids; these are not interchangeable",\n                  single_class=not both)\n    if threshold is not None:\n        if isinstance(threshold, dict):\n            if threshold.get("selection_split") != "validation" or threshold.get("objective") != "f1":\n                raise ValueError("Threshold policy must be locked on VALIDATION")\n            if threshold.get("policy_sha256") != _json_hash({k: v for k, v in threshold.items() if k != "policy_sha256"}):\n                raise ValueError("Validation threshold policy hash mismatch")\n            value = float(threshold["threshold"])\n        else:\n            value = float(threshold)\n        if not np.isfinite(value) or not 0 <= value <= 1:\n            raise ValueError("Probability threshold must lie in [0,1]")\n        predicted = scores >= value\n        tp, fp = int((predicted & (y == 1)).sum()), int((predicted & (y == 0)).sum())\n        fn, tn = positives - tp, n - positives - fp\n        result.update(threshold=value, tp=tp, fp=fp, fn=fn, tn=tn,\n                      precision=_ratio(tp, tp + fp), recall=_ratio(tp, tp + fn),\n                      specificity=_ratio(tn, tn + fp), f1=_ratio(2 * tp, 2 * tp + fp + fn),\n                      accuracy=(tp + tn) / n,\n                      confusion_matrix=[[tn, fp], [fn, tp]],\n                      threshold_scope="validation_locked" if isinstance(threshold, dict) else "caller_supplied")\n    bins = np.minimum((scores * calibration_bins).astype(int), calibration_bins - 1)\n    rows = []\n    for b in range(calibration_bins):\n        mask = bins == b\n        count = int(mask.sum())\n        rows.append(dict(bin=b, lower=b / calibration_bins, upper=(b + 1) / calibration_bins,\n                         count=count, mean_probability=float(scores[mask].mean()) if count else None,\n                         observed_rate=float(y[mask].mean()) if count else None))\n    result["calibration"] = rows\n    result["expected_calibration_error"] = float(sum(\n        row["count"] / n * abs(row["mean_probability"] - row["observed_rate"])\n        for row in rows if row["count"]))\n    result["calibration_note"] = "Fixed-width descriptive calibration; no calibration model fitted to evaluation labels"\n    result["topk"] = [topk_metrics(y, scores, f, tie_breaker) for f in top_fractions]\n    return result\n\n\ndef select_validation_threshold(y_validation, scores_validation, objective="f1", *, selection_split="validation"):\n    """Choose once on VALIDATION; highest threshold breaks an exact F1 tie."""\n    if selection_split != "validation" or objective != "f1":\n        raise ValueError("Only the prespecified VALIDATION F1 policy is supported")\n    y, scores = _inputs(y_validation, scores_validation)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Threshold selection requires both classes in VALIDATION")\n    precision, recall, thresholds = precision_recall_curve(y, scores)\n    p, r = precision[:-1], recall[:-1]\n    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)\n    best = np.flatnonzero(f1 == f1.max())[-1]\n    record = dict(threshold=float(thresholds[best]), objective=objective, selection_split="validation",\n                  validation_f1=float(f1[best]), validation_snapshots=len(y),\n                  tie_rule="highest probability threshold among exactly equal maximum F1 values",\n                  decision_rule="score >= threshold")\n    record["policy_sha256"] = _json_hash(record)\n    return record\n\n\ndef lock_validation_policy(y_validation, scores_validation, top_fractions=(0.1,)):\n    """Top-K locks population fractions, not a score cutoff derived from TEST."""\n    fractions = list(map(float, top_fractions))\n    if any(not 0 < f <= 1 for f in fractions):\n        raise ValueError("Invalid operational population fractions")\n    return dict(f1=select_validation_threshold(y_validation, scores_validation),\n                top_fractions=fractions,\n                topk_rule="ceil(fraction*N); descending score; ascending prespecified unique snapshot key",\n                selection_split="validation")\n\n\ndef _patient_codes(patient_ids, n):\n    patients = np.asarray(patient_ids)\n    if patients.ndim != 1 or len(patients) != n:\n        raise ValueError("Patient groups must match predictions")\n    if any(v is None or str(v).strip().lower() in ("", "nan", "none", "nat") for v in patients):\n        raise ValueError("Patient IDs must be nonmissing")\n    _, codes = np.unique(patients.astype(str), return_inverse=True)\n    return codes\n\n\ndef paired_patient_bootstrap(y, scores_a, scores_b, patient_ids, n_bootstrap=1000,\n                             seed=20261006, confidence=0.95):\n    """Delta = model A minus B. Cluster multiplicities act as row weights."""\n    y, a = _inputs(y, scores_a)\n    _, b = _inputs(y, scores_b)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Paired AP/AUC inference requires both classes")\n    if isinstance(n_bootstrap, bool) or int(n_bootstrap) != n_bootstrap or n_bootstrap < 20:\n        raise ValueError("At least 20 bootstrap draws required; use >=1000 for reports")\n    if not 0 < confidence < 1:\n        raise ValueError("Confidence must lie in (0,1)")\n    codes = _patient_codes(patient_ids, len(y))\n    groups = int(codes.max()) + 1\n    if groups < 2:\n        raise ValueError("At least two patients required")\n    rng = np.random.default_rng(seed)\n    draws, invalid = [], 0\n    for _ in range(int(n_bootstrap)):\n        multiplicity = np.bincount(rng.integers(0, groups, size=groups), minlength=groups)\n        weights = multiplicity[codes]\n        if weights[y == 1].sum() == 0 or weights[y == 0].sum() == 0:\n            invalid += 1\n            continue\n        ap = average_precision_score(y, a, sample_weight=weights) - average_precision_score(y, b, sample_weight=weights)\n        auc = roc_auc_score(y, a, sample_weight=weights) - roc_auc_score(y, b, sample_weight=weights)\n        draws.append((ap, auc))\n    result = dict(unit="patient", patients=groups, requested_draws=int(n_bootstrap), valid_draws=len(draws),\n                  single_class_draws=invalid, seed=int(seed), confidence=float(confidence),\n                  direction="model_a_minus_model_b", paired=True,\n                  uncertainty_scope="conditional on fitted predictions; excludes retraining and model-selection uncertainty")\n    enough = len(draws) >= max(20, int(math.ceil(0.8 * n_bootstrap)))\n    result["interval_reliable"] = enough\n    values = np.asarray(draws) if draws else np.empty((0, 2))\n    alpha = (1 - confidence) / 2\n    for index, (name, func) in enumerate((("average_precision", average_precision_score), ("roc_auc", roc_auc_score))):\n        result[name] = dict(delta=float(func(y, a) - func(y, b)),\n                            lower=float(np.quantile(values[:, index], alpha)) if enough else None,\n                            upper=float(np.quantile(values[:, index], 1 - alpha)) if enough else None)\n    return result\n\n\ndef latest_patient_indices(patient_ids, end_dates):\n    """One latest snapshot per patient; reject ambiguous duplicate latest keys."""\n    dates = np.asarray(end_dates, dtype="datetime64[D]")\n    codes = _patient_codes(patient_ids, len(dates))\n    if dates.ndim != 1 or np.isnat(dates).any():\n        raise ValueError("Latest-patient evaluation requires exact nonmissing dates")\n    result = []\n    for patient in range(int(codes.max()) + 1):\n        rows = np.flatnonzero(codes == patient)\n        latest = rows[dates[rows] == dates[rows].max()]\n        if len(latest) != 1:\n            raise ValueError("Duplicate latest patient-snapshot keys")\n        result.append(int(latest[0]))\n    return np.asarray(result, dtype=int)\n\n\ndef summarize_seed_runs(results):\n    """Accept training.py\'s candidate -> complete seed-run list; retain every seed."""\n    if not isinstance(results, dict) or not results:\n        raise ValueError("Expected candidate-to-seed-runs dictionary")\n    all_seeds, candidates = [], []\n    for candidate, runs in results.items():\n        seeds = []\n        for run in runs:\n            summary = run["summary"]\n            if not summary.get("training_complete"):\n                raise ValueError("Incomplete runs cannot disappear from the final comparison")\n            seed = int(summary["seed"])\n            if seed in seeds:\n                raise ValueError("Duplicate seed in candidate")\n            seeds.append(seed)\n            row = dict(candidate=candidate, seed=seed,\n                       family=summary["config"]["family"],\n                       parameter_count=summary.get("parameter_count"),\n                       complexity_unit=summary.get("complexity_unit"),\n                       best_epoch=summary.get("best_epoch"), wall_seconds=summary.get("wall_seconds"),\n                       validation_evaluations=summary.get("validation_evaluations"))\n            for split in ("train", "validation", "test"):\n                for metric, value in summary.get(split + "_metrics", {}).items():\n                    if isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_)):\n                        row[split + "_" + metric] = float(value)\n            train_ap, val_ap = row.get("train_average_precision"), row.get("validation_average_precision")\n            row["train_validation_gap_ap"] = train_ap - val_ap if train_ap is not None and val_ap is not None else None\n            test_ap = row.get("test_average_precision")\n            row["train_test_gap_ap"] = train_ap - test_ap if train_ap is not None and test_ap is not None else None\n            all_seeds.append(row)\n        if not seeds:\n            raise ValueError("Candidate has no completed seed repetitions")\n        candidate_rows = [r for r in all_seeds if r["candidate"] == candidate]\n        values = [r["validation_average_precision"] for r in candidate_rows]\n        candidates.append(dict(candidate=candidate, seeds=seeds, n_seeds=len(seeds),\n                               mean_validation_ap=float(np.mean(values)),\n                               sd_validation_ap=float(np.std(values, ddof=1)) if len(values) > 1 else None,\n                               min_validation_ap=float(np.min(values)), max_validation_ap=float(np.max(values)),\n                               seed_selection="all prespecified seeds, none selected by outcome"))\n    return dict(all_seeds=all_seeds, candidates=candidates,\n                note="Seed SD is training variability, not a confidence interval; bootstrap intervals are separate")\n\n\ndef comparison_conclusion(bootstrap, provenance):\n    """Wording is gated by comparison provenance, never VALIDATION significance."""\n    provenance = dict(provenance or {})\n    base = dict(status="UNRESOLVED", conclusion=None, restrictions=[])\n    if provenance.get("baseline_kind") == "original_v63" or not provenance.get("fit_membership_verified", False):\n        base.update(status="DESCRIPTIVE_ONLY", conclusion="The existing LightGBM scores have unresolved fitting provenance; they cannot establish comparative generalization or Transformer superiority.")\n        return base\n    required = ("same_cohort", "same_labels", "patient_disjoint_split", "selection_locked")\n    missing = [key for key in required if not provenance.get(key, False)]\n    if missing or provenance.get("test_used_for_selection", True):\n        base["restrictions"] = missing + (["test_used_for_selection_or_unknown"] if provenance.get("test_used_for_selection", True) else [])\n        base["conclusion"] = "A matched final comparison is not established; resolve the listed protocol conditions before making a superiority claim."\n        return base\n    if provenance.get("comparison_split") != "test":\n        base.update(status="DEVELOPMENT_ONLY", conclusion="Validation differences are development evidence after model selection; final-test superiority is not established.")\n        return base\n    for key in ("upstream_cutoff_verified", "holdout_never_previously_inspected"):\n        if not provenance.get(key, False):\n            base["restrictions"].append(key)\n    if not provenance.get("same_feature_information", False):\n        base["restrictions"].append("different_feature_information_architecture_effect_not_isolated")\n    ap = bootstrap["average_precision"]\n    if not bootstrap.get("interval_reliable") or ap["lower"] is None:\n        base.update(status="UNCERTAINTY_UNRESOLVED", conclusion="Too few valid patient-bootstrap draws support a reliable interval; report the point difference without a superiority conclusion.")\n        return base\n    prefix = "In this retrospective matched comparison" if base["restrictions"] else "On the locked patient-disjoint test cohort"\n    if ap["lower"] > 0:\n        verdict = "Transformer AP is higher than matched LightGBM AP"\n        status = "RETROSPECTIVE_AP_ADVANTAGE" if base["restrictions"] else "TEST_AP_ADVANTAGE"\n    elif ap["upper"] < 0:\n        verdict = "matched LightGBM AP is higher than Transformer AP"\n        status = "LIGHTGBM_AP_ADVANTAGE"\n    else:\n        verdict = "the AP interval includes no difference; superiority is not established and this is not proof of equivalence"\n        status = "NO_DEMONSTRATED_AP_ADVANTAGE"\n    base.update(status=status, conclusion=f"{prefix}, {verdict}. Delta AP (Transformer minus LightGBM) = {ap[\'delta\']:.6f}, patient-bootstrap interval [{ap[\'lower\']:.6f}, {ap[\'upper\']:.6f}]. This interval conditions on the fitted models and excludes training/search uncertainty.")\n    if base["restrictions"]:\n        base["conclusion"] += " Unresolved provenance or prior holdout inspection prevents a prospective generalization claim."\n    return base\n\n\ndef compare_predictions(y, transformer_scores, lightgbm_scores, patient_ids, *,\n                        threshold_policies=None, provenance=None, tie_breaker=None,\n                        top_fractions=(0.1,), n_bootstrap=1000, seed=20261006):\n    """Caller aligns both score vectors to identical frozen keys and persists lock first."""\n    policies = threshold_policies or {}\n    t = binary_metrics(y, transformer_scores, policies.get("transformer"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    l = binary_metrics(y, lightgbm_scores, policies.get("lightgbm"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    interval = paired_patient_bootstrap(y, transformer_scores, lightgbm_scores, patient_ids,\n                                        n_bootstrap=n_bootstrap, seed=seed)\n    return dict(transformer=t, lightgbm=l, paired_uncertainty=interval,\n                conclusion=comparison_conclusion(interval, provenance), provenance=dict(provenance or {}))\n', 'warehouse': '"""Embedded by the delivery notebooks; no repository dependency at runtime."""\nimport base64\nimport hashlib\nimport io\nimport json\nimport math\nimport re\nimport tempfile\nimport uuid\nfrom contextlib import contextmanager\nfrom itertools import islice\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef canonical_json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef digest_json(value):\n    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()\n\n\ndef read_sf(suffix):\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("dbtable", f"{PREFIX}_{suffix}").load())\n\n\ndef table_exists(table):\n    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):\n        raise ValueError("Use uppercase letters, numbers and underscores in table names.")\n    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "\n             f"WHERE TABLE_SCHEMA = \'DS_ML\' AND TABLE_NAME = \'{table}\'")\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("query", query).load().limit(1).count() > 0)\n\n\ndef checked_metadata(frame, include_split=False):\n    columns = ["PATIENT_ID", "END_DT", "RESP"]\n    if include_split:\n        columns += ["SPLIT", "SPLIT_CONFIG"]\n    out = frame.loc[:, columns].copy()\n    if out.empty or out.isna().any().any():\n        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")\n    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():\n        raise ValueError("PATIENT_ID must retain its original nonempty string value.")\n    dates = pd.to_datetime(out.END_DT, errors="raise")\n    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():\n        raise ValueError("END_DT must be a date without an intraday time/timezone.")\n    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")\n    if not out.RESP.isin([0, 1]).all():\n        raise ValueError("RESP must be exactly 0 or 1 before conversion.")\n    out["RESP"] = out.RESP.astype("int64")\n    if out.duplicated(["PATIENT_ID", "END_DT"]).any():\n        raise ValueError("Duplicate patient/date keys in snapshot metadata.")\n    if include_split:\n        if set(out.SPLIT) != {"train", "validation", "test"}:\n            raise ValueError("Expected the saved train, validation and test assignments.")\n        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():\n            raise ValueError("Patient overlap between splits.")\n        if len(out.SPLIT_CONFIG.unique()) != 1:\n            raise ValueError("The split contains inconsistent creation settings.")\n        for _, part in out.groupby("SPLIT", observed=True):\n            if set(part.RESP) != {0, 1}:\n                raise ValueError("Each split must contain both response classes.")\n    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n\n\ndef validate_metadata_pair(snapshot_frame, manifest_frame, features):\n    source = checked_metadata(snapshot_frame)\n    manifest = checked_metadata(manifest_frame, include_split=True)\n    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):\n        raise ValueError("Frozen split and source snapshots differ in keys or labels.")\n    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])\n    \n    feature_order_hash = hashlib.sha256(\n        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()\n    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:\n        raise ValueError("Feature order or timesteps differ from the frozen split.")\n    return manifest, creation\n\n\ndef new_tensor_buffer(metadata, feature_count, seq_len=12):\n    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")\n    path = Path(temporary.name) / "counts.float32"\n    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))\n    return temporary, X\n\n\ndef fill_tensor(X, metadata, sequence_rows):\n    """Place rows by canonical keys, independent of Spark partition order."""\n    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}\n    seen = np.zeros(len(metadata), dtype=bool)\n    seq_len, feature_count = X.shape[1:]\n    for row in sequence_rows:\n        raw_date = row["END_DT"]\n        if raw_date is None or row["PATIENT_ID"] is None:\n            raise ValueError("Null monthly snapshot key.")\n        date = pd.Timestamp(raw_date)\n        if date.tzinfo is not None or date != date.normalize():\n            raise ValueError("Monthly END_DT is not an exact date.")\n        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))\n        if key not in positions:\n            raise ValueError("Unexpected monthly snapshot key.")\n        i = positions[key]\n        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):\n            raise ValueError("Duplicate monthly snapshot or changed label.")\n        sequence = row["SEQUENCE"]\n        steps = [month["T"] for month in sequence]\n        if len(sequence) != seq_len or any(t is None for t in steps):\n            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")\n        \n        if sorted(steps) != list(range(seq_len)):\n            raise ValueError("Timesteps must be unique integers 0 through 11.")\n        sequence = sorted(sequence, key=lambda month: month["T"])\n        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)\n        if values.shape != (seq_len, feature_count):\n            raise ValueError("Monthly feature shape does not match the vocabulary.")\n        if not np.isfinite(values).all() or (values < 0).any():\n            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")\n        X[i] = values\n        seen[i] = True\n    if not seen.all():\n        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")\n    X.flush()\n\n\ndef input_fingerprints(X, metadata, features):\n    tensor_hash = hashlib.sha256()\n    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))\n    for start in range(0, len(X), 128):\n        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))\n    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]\n               for r in metadata.itertuples()]\n    return {"model_input_sha256": tensor_hash.hexdigest(),\n            "snapshot_manifest_sha256": digest_json(records),\n            "feature_names_sha256": digest_json(features),\n            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}\n\n\ndef load_inputs():\n    from pyspark.sql import functions as F\n    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()\n    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):\n        raise ValueError("Invalid feature indices.")\n    features = [r["FEATURE_NAME"] for r in mapping]\n    aliases = [r["FEATURE_COLUMN"] for r in mapping]\n    if (aliases != [f"F{i:04d}" for i in range(len(features))]\n            or not all(isinstance(f, str) and f for f in features)\n            or len(set(features)) != len(features)):\n        raise ValueError("Invalid feature names, aliases or order.")\n    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()\n    frozen = read_sf("PATIENT_SPLIT").select(\n        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()\n    metadata, creation = validate_metadata_pair(source, frozen, features)\n    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))\n    if observed != (23151, 12447, 1345, 1028):\n        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")\n    monthly = read_sf("TENSOR_MONTHLY")\n    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):\n        raise ValueError("Monthly table columns differ from the frozen feature map.")\n    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)\n    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")\n        .agg(F.collect_list(F.struct(\n            F.col("TIME_STEP").alias("T"),\n            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))\n                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")\n        )).alias("SEQUENCE"))\n        .repartition(128))\n    temporary, X = new_tensor_buffer(metadata, len(features))\n    try:\n        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))\n        hashes = input_fingerprints(X, metadata, features)\n        X.flags.writeable = False\n    except BaseException:\n        X._mmap.close()\n        temporary.cleanup()\n        raise\n    y = metadata.RESP.to_numpy(dtype=np.float32)\n    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)\n               for name in ("train", "validation", "test")}\n    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)\n    return {"X": X, "y": y, "metadata": metadata, "features": features,\n            "indices": indices, "hashes": hashes, "split_creation": creation,\n            "temporary_directory": temporary}\n\n\n@contextmanager\ndef _blob_reader(blob):\n    """Bytes and caller-owned files share the same raw-byte hash contract."""\n    if isinstance(blob, Path):\n        with blob.open("rb") as stream:\n            yield stream\n    elif isinstance(blob, (bytes, bytearray, memoryview)):\n        \n        class Reader:\n            offset = 0\n            def read(self, size):\n                block = memoryview(blob)[self.offset:self.offset + size]\n                self.offset += len(block)\n                return block\n        yield Reader()\n    else:\n        raise TypeError("Artifact payload must be bytes or pathlib.Path")\n\n\ndef artifact_signature(blob):\n    hasher, size = hashlib.sha256(), 0\n    with _blob_reader(blob) as stream:\n        while True:\n            block = stream.read(1024 * 1024)\n            if not block:\n                break\n            hasher.update(block)\n            size += len(block)\n    return size, hasher.hexdigest()\n\n\ndef iter_artifact_rows(artifacts, chunk_size=50000, signatures=None):\n    """Same rows as the former packer, without whole-blob/base64/list copies."""\n    if not isinstance(chunk_size, int) or chunk_size < 1:\n        raise ValueError("chunk_size must be a positive integer")\n    for name, blob in artifacts.items():\n        size, digest = signatures[name] if signatures is not None else artifact_signature(blob)\n        count = max(1, math.ceil((4 * ((size + 2) // 3)) / chunk_size))\n        \n        raw_size = max(3, (chunk_size // 4) * 3)\n        pending, index = "", 0\n        with _blob_reader(blob) as stream:\n            while True:\n                block = stream.read(raw_size)\n                if not block:\n                    break\n                pending += base64.b64encode(block).decode("ascii")\n                while len(pending) >= chunk_size:\n                    yield (name, index, count, size, digest, pending[:chunk_size])\n                    pending = pending[chunk_size:]\n                    index += 1\n        if pending or size == 0:\n            yield (name, index, count, size, digest, pending)\n\n\ndef pack_artifacts(artifacts, chunk_size=50000):\n    """Compatibility helper for small callers; production writes use the iterator."""\n    return list(iter_artifact_rows(artifacts, chunk_size))\n\n\ndef unpack_artifacts(rows, expected_names, *, disk_names=(), output_dir=None, ordered=False):\n    """Decode ordered chunks incrementally; optional large payloads remain on disk.\n\n    The default still accepts unordered small row collections. read_artifacts\n    supplies bounded, ordered chunks, so it never takes that sorting path.\n    """\n    names, disk_names = set(expected_names), set(disk_names)\n    if not disk_names <= names or (disk_names and output_dir is None):\n        raise ValueError("disk_names must be expected names and require output_dir")\n    if not ordered:\n        rows = sorted(rows, key=lambda row: (row[0], row[1]))\n    result, created, stream = {}, [], None\n    current, count, size, digest, next_index, actual_size, pending = None, 0, 0, None, 0, 0, ""\n\n    def finish():\n        nonlocal stream\n        if next_index != count or pending or actual_size != size or hasher.hexdigest() != digest:\n            raise ValueError("Missing, duplicate or inconsistent artifact chunks, or length/hash mismatch.")\n        if current in disk_names:\n            stream.close()\n            result[current] = path\n        else:\n            result[current] = stream.getvalue()\n            stream.close()\n        stream = None\n\n    try:\n        for row in rows:\n            name, i, row_count, row_size, row_digest, payload = tuple(row)\n            if any(value != int(value) for value in (i, row_count, row_size)):\n                raise ValueError("Nonintegral artifact chunk metadata.")\n            i, row_count, row_size = int(i), int(row_count), int(row_size)\n            if name != current:\n                if current is not None:\n                    finish()\n                if name not in names or name in result or row_count < 1 or row_size < 0:\n                    raise ValueError("Missing, duplicate or unexpected saved artifacts.")\n                current, count, size, digest = name, row_count, row_size, row_digest\n                next_index, actual_size, pending, hasher = 0, 0, "", hashlib.sha256()\n                if name in disk_names:\n                    directory = Path(output_dir)\n                    directory.mkdir(parents=True, exist_ok=True)\n                    \n                    path = directory / ("artifact_" + uuid.uuid4().hex)\n                    stream = path.open("xb")\n                    created.append(path)\n                else:\n                    stream = io.BytesIO()\n            if i != next_index or (row_count, row_size, row_digest) != (count, size, digest):\n                raise ValueError("Missing, duplicate or inconsistent artifact chunks.")\n            pending += payload\n            length = (len(pending) // 4) * 4\n            if length:\n                encoded, pending = pending[:length], pending[length:]\n                block = base64.b64decode(encoded, validate=True)\n                \n                if "=" in encoded and next_index != count - 1:\n                    raise ValueError("Unexpected base64 padding before final artifact chunk")\n                stream.write(block)\n                hasher.update(block)\n                actual_size += len(block)\n            next_index += 1\n        if current is not None:\n            finish()\n        if set(result) != names:\n            raise ValueError("Missing or unexpected saved artifacts.")\n        return result\n    except BaseException:\n        if stream is not None:\n            stream.close()\n        for path in created:\n            path.unlink(missing_ok=True)\n        raise\n\n\nARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]\nARTIFACT_BATCH_ROWS = 1024\n\n\ndef _checked_table(table):\n    if not isinstance(table, str) or not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):\n        raise ValueError("Use uppercase letters, numbers and underscores in table names.")\n    return table\n\n\ndef _query_artifacts(query):\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("query", query).load())\n\n\ndef _read_artifact_rows(table, expected_names, batch_rows=ARTIFACT_BATCH_ROWS):\n    """Bound each connector result, not just the Python iterator\'s buffer."""\n    table = _checked_table(table)\n    if not isinstance(batch_rows, int) or batch_rows < 1:\n        raise ValueError("batch_rows must be positive")\n    metadata = _query_artifacts(\n        f"SELECT ARTIFACT_NAME,CHUNK_COUNT,BYTE_LENGTH,SHA256,COUNT(*) AS ROW_COUNT FROM {table} "\n        "GROUP BY ARTIFACT_NAME,CHUNK_COUNT,BYTE_LENGTH,SHA256").limit(len(set(expected_names)) + 1).collect()\n    descriptions = {}\n    for row in metadata:\n        name, count, size, digest, actual_count = tuple(row)\n        if (name in descriptions or name not in expected_names\n                or any(value != int(value) for value in (count, size, actual_count))\n                or count < 1 or size < 0 or count != actual_count):\n            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")\n        descriptions[name] = (int(count), int(size), digest)\n    if set(descriptions) != set(expected_names):\n        raise ValueError("Missing or unexpected saved artifacts.")\n    for name in sorted(descriptions):\n        count = descriptions[name][0]\n        quoted_name = "\'" + str(name).replace("\'", "\'\'") + "\'"\n        for start in range(0, count, batch_rows):\n            query = ("SELECT " + ",".join(ARTIFACT_COLUMNS) + f" FROM {table} "\n                f"WHERE ARTIFACT_NAME={quoted_name} AND CHUNK_INDEX >= {start} "\n                f"AND CHUNK_INDEX < {min(start + batch_rows, count)} ORDER BY CHUNK_INDEX")\n            \n            batch = _query_artifacts(query).limit(batch_rows + 1).collect()\n            yield from sorted(batch, key=lambda row: row[1])\n            del batch\n\n\ndef read_artifacts(table, expected_names, *, disk_names=(), output_dir=None, batch_rows=ARTIFACT_BATCH_ROWS):\n    return unpack_artifacts(_read_artifact_rows(table, expected_names, batch_rows), expected_names,\n        disk_names=disk_names, output_dir=output_dir, ordered=True)\n\n\ndef _verify_saved_artifacts(table, signatures, batch_rows=ARTIFACT_BATCH_ROWS):\n    with tempfile.TemporaryDirectory(prefix="dl_poc_artifact_verify_") as directory:\n        saved = read_artifacts(table, signatures, disk_names=signatures, output_dir=directory,\n                               batch_rows=batch_rows)\n        if any(artifact_signature(saved[name]) != signature for name, signature in signatures.items()):\n            raise ValueError("Saved artifact read-back differs from the completed run.")\n\n\ndef _artifact_ddl(sql):\n    \n    \n    spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(dict(sf_options_dl_poc), sql)\n\n\ndef _has_artifact_jvm():\n    """Detect JVM availability before transferring any artifact payload."""\n    try:\n        return getattr(spark, "_jvm", None) is not None\n    except (AttributeError, NotImplementedError):\n        \n        return False\n\n\ndef save_artifacts(table, artifacts, *, chunk_size=50000, batch_rows=ARTIFACT_BATCH_ROWS):\n    """Bounded writes with immutable, verified completed artifacts.\n\n    Classic Spark verifies a pending table before atomic rename. Without a JVM,\n    write directly using errorifexists then append. A failed direct transfer can\n    leave an incomplete final-name table: readers reject its counts/hash and a\n    retry fails closed, requiring a new RUN_ID. Neither mode overwrites a table.\n    """\n    table = _checked_table(table)\n    if not artifacts or not isinstance(batch_rows, int) or batch_rows < 1:\n        raise ValueError("Require artifacts and positive batch_rows")\n    signatures = {name: artifact_signature(blob) for name, blob in artifacts.items()}\n    if table_exists(table):\n        try:\n            _verify_saved_artifacts(table, signatures, batch_rows)\n        except ValueError as exc:\n            raise FileExistsError("Destination contains incomplete or different artifacts; choose a new RUN_ID.") from exc\n        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n        return\n    atomic_rename = _has_artifact_jvm()\n    staging = table + "__PENDING_" + uuid.uuid4().hex.upper() if atomic_rename else table\n    _checked_table(staging)\n    if not atomic_rename:\n        print("Artifact publication: JVM unavailable; direct immutable batches. "\n              "Interrupted uploads remain invalid and require a new RUN_ID.", flush=True)\n    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "\n              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")\n    rows, first = iter_artifact_rows(artifacts, chunk_size, signatures), True\n    while True:\n        batch = list(islice(rows, batch_rows))\n        if not batch:\n            break\n        frame = spark.createDataFrame(batch, schema=schema)\n        (frame.write.format("snowflake").options(**sf_options_dl_poc)\n         .option("dbtable", staging).option("truncate_columns", "off")\n         .mode("errorifexists" if first else "append").save())\n        first = False\n        del frame, batch\n    _verify_saved_artifacts(staging, signatures, batch_rows)\n    if atomic_rename:\n        \n        \n        _artifact_ddl(f"ALTER TABLE {staging} RENAME TO {table}")\n    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n'}
EMBEDDED_SOURCE_SHA256 = '86f3e68006f8bcd6474b383724520be556840ca8fdcc16d265763cb94476069c'
IMPLEMENTATION_SHA256 = '69adef9395638375cd5497f598acce84c20c6dff886279601c4953df0e95303b'
if hashlib.sha256(json.dumps(EMBEDDED_SOURCES, sort_keys=True).encode()).hexdigest() != EMBEDDED_SOURCE_SHA256:
    raise ValueError("Embedded implementation changed. Rebuild all five notebooks together.")
modules = {}
for name, source in EMBEDDED_SOURCES.items():
    module = types.ModuleType("_tak861_automl_" + name)
    sys.modules[module.__name__] = module
    exec(compile(source, module.__name__, "exec"), module.__dict__)
    modules[name] = module
inp, cl, mdl, wf, cmp, wh = [modules[n] for n in
    ("inputs", "count_loader", "modeling", "workflow", "comparison", "warehouse")]
rel = modules["reliance"]
recovery = modules["source_recovery"]
raw = modules["raw_automl"]
population = modules["client_universe"]
population.inputs, population.fresh_counts, population.count_loader = inp, modules["fresh_counts"], cl
raw.inputs = inp
wf.inputs, wf.modeling, wf.comparison, cl.inputs = inp, mdl, cmp, inp
wh.spark, wh.sf_options_dl_poc, wh.PREFIX = spark, sf_options_dl_poc, SOURCE_PREFIX
wf.save_artifacts, wf.read_artifacts, wf.table_exists = wh.save_artifacts, wh.read_artifacts, wh.table_exists
RUN_PREFIX = wf.prefix(RUN_ID)
def read_query(query):
    return spark.read.format("snowflake").options(**sf_options_dl_poc).option("query", query).load()
def load_stage():
    data = wf.load_inputs(RUN_PREFIX, IMPLEMENTATION_SHA256)
    if data["manifest"]["configuration_hash"] != wf.digest(CONFIG):
        raise ValueError("Notebook configuration differs from Notebook 01; synchronize configuration or use a new RUN_ID.")
    return data
def aggregate(frame, n=40):
    
    display(frame.head(n))
def memory_receipt(stage):
    
    import shutil, tempfile
    receipt = {"stage": stage, "scratch_directory": tempfile.gettempdir(),
               "scratch_free_bytes": shutil.disk_usage(tempfile.gettempdir()).free}
    try:
        import psutil
        receipt["python_rss_bytes"] = psutil.Process().memory_info().rss
    except ImportError:
        receipt["python_rss_bytes"] = None
    try:
        import resource
        receipt["python_lifetime_peak_rss_bytes"] = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * (1 if sys.platform == "darwin" else 1024))
    except ImportError:
        receipt["python_lifetime_peak_rss_bytes"] = None
    print("Memory receipt:", json.dumps(receipt, sort_keys=True))
print("Run:", RUN_ID, "Implementation:", IMPLEMENTATION_SHA256)
memory_receipt("notebook_start")


In [ ]:
RUN_FINAL_TEST = False
BOOTSTRAP_RESAMPLES = 1000
data = load_stage()
saved = wh.read_artifacts(RUN_PREFIX+"_PREPARATION", wf.PREP_NAMES)
prep, protocol = json.loads(saved["preparation.json"]), json.loads(saved["protocol.json"])
results = wf.read_runs(protocol, RUN_PREFIX)
lock = json.loads(wh.read_artifacts(RUN_PREFIX+"_LOCK", {"lock.json"})["lock.json"])
report = wf.compare_run(data, prep, protocol, results, lock, RUN_PREFIX,
                        evaluate_test=RUN_FINAL_TEST, n_bootstrap=BOOTSTRAP_RESAMPLES)
aggregate(wf.report_table(report))
uncertainty = []
for name, pair in report["comparisons"].items():
    for metric in ("primary_lift", "average_precision", "roc_auc"):
        uncertainty.append(dict(comparison=name, metric=metric, **pair[metric]))
aggregate(pd.DataFrame(uncertainty), 100)
aggregate(pd.DataFrame([dict(comparison=name, **interval)
    for name, pair in report["comparisons"].items()
    for fraction, interval in pair["lift_by_fraction"].items()]), 100)
capacity = []
for arm, values in report["metrics"].items():
    for metrics in values["topk"]:
        capacity.append(dict(arm=arm, **metrics))
aggregate(pd.DataFrame(capacity))
aggregate(pd.DataFrame([dict(arm=arm, **row) for arm, values in report["metrics"].items()
                        for row in values["calibration"]]), 100)
aggregate(pd.DataFrame([dict(arm=arm, **{key: values.get(key) for key in
    ("threshold", "tn", "fp", "fn", "tp", "specificity", "expected_calibration_error")})
    for arm, values in report["metrics"].items()]))
aggregate(pd.DataFrame([dict(arm=arm,
    train_lift=lock["metrics"][arm]["train"]["primary_lift"],
    validation_lift=lock["metrics"][arm]["validation"]["primary_lift"],
    train_validation_lift_gap=lock["metrics"][arm]["train"]["primary_lift"]-lock["metrics"][arm]["validation"]["primary_lift"],
    train_ap=lock["metrics"][arm]["train"]["average_precision"],
    validation_ap=lock["metrics"][arm]["validation"]["average_precision"],
    train_validation_ap_gap=lock["metrics"][arm]["train"]["average_precision"]-lock["metrics"][arm]["validation"]["average_precision"],
    parameters_per_seed=runs[0]["summary"]["parameter_count"],
    total_training_seconds=sum(r["summary"]["wall_seconds"] for r in runs))
    for arm, runs in results.items()]))
print("Split:", report["split"], "TEST evaluation ordinal:", report["test_evaluation_ordinal"])
print("Count comparison:", report["count_reference_status"], "LightGBM:", report["lightgbm_status"])
for limitation in report["limitations"]:
    print("Limitation:", limitation)


In [ ]:
for contrast, pair in report["comparisons"].items():
    lift = pair["primary_lift"]
    if not pair["lift_bootstrap"]["interval_reliable"]:
        statement = "insufficient valid patient-bootstrap draws; interval unavailable"
    elif lift["lower"] > 0:
        statement = "positive lift difference at frozen capacity within this evaluation; inspect size and limitations"
    elif lift["upper"] < 0:
        statement = "negative lift difference at frozen capacity within this evaluation"
    else:
        statement = "lift difference remains inconclusive; interval includes zero"
    print(contrast, statement)
print("No independent OOT conclusion. No completed LightGBM comparison in this run.")
wf.release_inputs(data)
del data
memory_receipt("notebook_05_released")
